# Phase 5AB point-law shard 06

Predicted runtime: 293.1 minutes. Each task checkpoints after 25 replications.

In [ ]:
import os

# Fleet controls are intentionally the first executable cell.
SHARD_ID = 6
N_SHARDS = 40
SEED_ROOT = 'phase5ab-pointlaw-v1'
WALL_BUDGET_MIN = 450
N_WORKERS = 2
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
print({"SHARD_ID": SHARD_ID, "N_SHARDS": N_SHARDS, "WALL_BUDGET_MIN": WALL_BUDGET_MIN, "N_WORKERS": N_WORKERS})


In [ ]:
import os
import shutil

REPO_DIR = "/content/Pointcloud_Equality_Testing"
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
os.system("git clone --depth 1 https://github.com/hugogobato/Pointcloud_Equality_Testing.git " + REPO_DIR)
os.chdir(REPO_DIR)
os.system("python -m pip install -q -e .")
os.system("python -m pip install -q networkx==3.4.2")
print("Repository ready:", REPO_DIR)


In [ ]:
%%writefile tda2s/tests/point_law.py
"""Established point-law two-sample tests for the Phase 5AB benchmark.

This module implements the required raw-data competitors that use all
individual points (not disjoint blocks) for testing

    H0^law: P0 = P1

under Regime I (iid metric-measure sampling).  Every method:

* caches the pooled representation once (Gram, distance matrix, graph, …)
* calibrates by pooled point-label permutation preserving (n0, n1)
* is valid under iid exchangeability; dependence, overlapping blocks,
  or label-dependent tuning are outside the validity scope
* records the exact output fields demanded by
  docs/phase5ab_point_law_benchmark_plan.md Section 3.3

Method registry
---------------
* PointMMD-Gaussian  — characteristic Gaussian kernel MMD (Gretton et al. 2012)
* EnergyDistance      — Euclidean energy distance (Szekely & Rizzo 2013)
* FriedmanRafsky-MST  — MST cross-edge count (Friedman & Rafsky 1979)
* Schilling-kNN       — k-NN same-label edge count (Schilling 1986)
* Rosenbaum-CrossMatch — minimum-weight non-bipartite matching, cross pairs
                         (Rosenbaum 2005)
* SlicedWasserstein   — fixed-projection sliced W1 average (Ramdas et al. 2015
                         as secondary geometric baseline)
* ClassifierTwoSampleTest — sample-split logistic / RF accuracy with held-out
                            label permutation (Lopez-Paz & Oquab 2017)

Identification note (Section 1.2 audit)
----------------------------------------
1. A Gaussian point kernel on Euclidean space is characteristic (Simon-Gabriel &
   Scholkopf 2018, Gretton et al. 2012) so point-level Gaussian MMD identifies
   P0=P1 under the usual moment/measurability conditions.
2. Euclidean energy distance identifies equality of distributions under finite
   first-moment conditions (Szekely & Rizzo 2013; equivalence to MMD via
   Sejdinovic et al. 2013).
3. MST, k-NN and cross-match are valid permutation statistics under iid
   point-law null; their consistency requires graph-test conditions (Friedman &
   Rafsky 1979; Schilling 1986; Rosenbaum 2005) and they are NOT
   characteristic-kernel tests.
4. A finite collection of sliced-Wasserstein projections is a sensitivity
   statistic unless the projection family is shown to be identifying;
   the implementation fixes projections before labels and labels it honestly.
5. A classifier two-sample test targets P0=P1 only relative to its classifier
   class and split protocol (Lopez-Paz & Oquab 2017); not universally consistent.
6. The simple average pairwise kernel on raw bags (mean_pairwise) remains a
   downgraded sensitivity variant and must not replace the Hilbert-Gaussian
   unordered-bag kernel in the primary RawBlockMMD claim.
If any check fails for the selected implementation, the headline comparison
must exclude it and record the reason.
"""
from __future__ import annotations

import itertools
import math
import resource
import time
import tracemalloc
from typing import Optional, Sequence, Tuple

import numpy as np

from tda2s.resample import p_value
from tda2s.tests.single_cloud import REGIME_I, _label_masks, _mmd2_from_gram

__all__ = [
    "REGIME_I",
    "point_mmd_gaussian",
    "energy_distance_test",
    "friedman_rafsky_mst",
    "schilling_knn",
    "rosenbaum_crossmatch",
    "sliced_wasserstein_test",
    "classifier_two_sample_test",
    "METHOD_REGISTRY",
]

# ---------------------------------------------------------------------------
# Shared helpers (mirrors single_cloud.py but kept local for self-containment
# in Colab notebooks)

def _as_cloud(cloud, name: str) -> np.ndarray:
    points = np.asarray(cloud, dtype=float)
    if points.ndim != 2 or points.shape[0] < 2 or points.shape[1] < 1:
        raise ValueError(f"{name} must have shape (n, d) with n >= 2")
    if not np.isfinite(points).all():
        raise ValueError(f"{name} must contain only finite values")
    return points


def _validate_cloud_pair(cloud0, cloud1) -> Tuple[np.ndarray, np.ndarray]:
    x0, x1 = _as_cloud(cloud0, "cloud0"), _as_cloud(cloud1, "cloud1")
    if x0.shape[1] != x1.shape[1]:
        raise ValueError("cloud0 and cloud1 must have the same ambient dimension")
    return x0, x1


def _require_regime(candidate: str, regime: str) -> None:
    if regime != REGIME_I:
        raise ValueError(
            f"{candidate} is only valid for declared regime {REGIME_I!r}; received {regime!r}"
        )


def _current_rss_bytes() -> int:
    usage = resource.getrusage(resource.RUSAGE_SELF)
    scale = 1024 if __import__("sys").platform.startswith("linux") else 1
    return int(usage.ru_maxrss * scale)


def _finish(result: dict, started: float, peak_bytes: int) -> dict:
    result["runtime_seconds"] = float(time.perf_counter() - started)
    result["peak_memory_bytes"] = int(peak_bytes)
    result["peak_memory_measurement"] = "tracemalloc_python_allocations"
    return result


def _measure(callable_):
    tracemalloc.start()
    started = time.perf_counter()
    start_rss = _current_rss_bytes()
    try:
        result = callable_()
        _, peak = tracemalloc.get_traced_memory()
        result = _finish(result, started, peak)
        result["peak_rss_bytes"] = max(start_rss, _current_rss_bytes())
        result["peak_memory_measurement"] = (
            "tracemalloc_python_allocations; peak_rss_bytes is process high-water RSS"
        )
        return result
    finally:
        tracemalloc.stop()


def _permutation_pvalue_less(observed: float, null: np.ndarray) -> float:
    """Phipson-Smyth style p-value for left-tail (small is extreme)."""
    null = np.asarray(null, dtype=float)
    if null.size == 0:
        return 1.0
    cnt = np.count_nonzero(null <= observed)
    return float((1.0 + cnt) / (1.0 + len(null)))


def _median_heuristic_bandwidth(pooled: np.ndarray, max_sample: int = 2000, seed: int = 0) -> float:
    pts = np.asarray(pooled, dtype=float)
    if len(pts) < 2:
        return 1.0
    rng = np.random.default_rng(seed)
    if len(pts) > max_sample:
        pts = pts[rng.choice(len(pts), max_sample, replace=False)]
    # pairwise Euclidean distances, upper triangular
    from scipy.spatial.distance import pdist
    dists = pdist(pts, metric="euclidean")
    med = float(np.median(dists))
    return med if np.isfinite(med) and med > 0 else 1.0


def _gaussian_gram(pooled: np.ndarray, bandwidth: float) -> np.ndarray:
    if bandwidth <= 0:
        raise ValueError("kernel bandwidth must be positive")
    from scipy.spatial.distance import cdist
    D2 = cdist(pooled, pooled, metric="sqeuclidean")
    return np.exp(-D2 / (2.0 * float(bandwidth) ** 2))


def _euclidean_distance_matrix(pooled: np.ndarray) -> np.ndarray:
    from scipy.spatial.distance import cdist
    return cdist(pooled, pooled, metric="euclidean")


def _deterministic_mst_edges(distance_matrix: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Return a deterministic MST edge list, including zero-distance ties.

    ``scipy.sparse.csgraph.minimum_spanning_tree`` treats zero entries in a
    dense/sparse matrix as absent edges.  That silently produces a forest for
    repeated points, which is common in the discrete four-atom diagnostic.
    Prim's algorithm below works directly with the complete finite distance
    matrix and resolves equal-distance choices by pooled point index.
    """
    distances = np.asarray(distance_matrix, dtype=float)
    if distances.ndim != 2 or distances.shape[0] != distances.shape[1]:
        raise ValueError("distance_matrix must be square")
    if not np.isfinite(distances).all() or (distances < 0).any():
        raise ValueError("distance_matrix must contain finite non-negative distances")
    n = distances.shape[0]
    if n < 2:
        return np.empty(0, dtype=int), np.empty(0, dtype=int)

    in_tree = np.zeros(n, dtype=bool)
    best = np.full(n, np.inf, dtype=float)
    parent = np.full(n, -1, dtype=int)
    best[0] = 0.0
    rows, cols = [], []

    for _ in range(n):
        candidates = np.flatnonzero(~in_tree)
        vertex = int(candidates[np.argmin(best[candidates])])
        if not np.isfinite(best[vertex]):
            raise ValueError("distance graph is disconnected")
        in_tree[vertex] = True
        if parent[vertex] >= 0:
            rows.append(parent[vertex])
            cols.append(vertex)

        remaining = ~in_tree
        candidate_distances = distances[vertex]
        strictly_better = remaining & (candidate_distances < best)
        tied_but_lower_index = (
            remaining
            & (candidate_distances == best)
            & ((parent < 0) | (vertex < parent))
        )
        update = strictly_better | tied_but_lower_index
        best[update] = candidate_distances[update]
        parent[update] = vertex

    return np.asarray(rows, dtype=int), np.asarray(cols, dtype=int)


# ---------------------------------------------------------------------------
# Method registry (machine-readable)

METHOD_REGISTRY = {
    "PointMMD-Gaussian": {
        "method": "point_mmd_gaussian",
        "target_null": "H0^law: P0=P1",
        "validity_regime": "iid_metric_measure",
        "sampling_unit": "individual iid point",
        "kernel_or_distance": "Gaussian point kernel",
        "is_characteristic": True,
        "requires": "pooled point Gram, frozen bandwidth",
        "literature": "Gretton et al. 2012 (A1); Simon-Gabriel & Scholkopf 2018 (A2)",
        "consistency_conditions": "characteristic kernel, bounded, moment/measurability",
        "is_full_point_law_test": True,
    },
    "EnergyDistance": {
        "method": "energy_distance_test",
        "target_null": "H0^law: P0=P1",
        "validity_regime": "iid_metric_measure",
        "sampling_unit": "individual iid point",
        "kernel_or_distance": "Euclidean distance",
        "is_characteristic": True,
        "requires": "pooled Euclidean distance matrix",
        "literature": "Szekely & Rizzo 2013 (B1); Sejdinovic et al. 2013 (B2)",
        "consistency_conditions": "finite first moment",
        "is_full_point_law_test": True,
    },
    "FriedmanRafsky-MST": {
        "method": "friedman_rafsky_mst",
        "target_null": "H0^law: P0=P1",
        "validity_regime": "iid_metric_measure",
        "sampling_unit": "individual iid point",
        "kernel_or_distance": "Euclidean MST cross-edge count",
        "is_characteristic": False,
        "requires": "pooled MST cached outside permutation loop",
        "literature": "Friedman & Rafsky 1979 (C1)",
        "consistency_conditions": "graph-test conditions; valid permutation procedure, consistency per source",
        "is_full_point_law_test": True,
    },
    "Schilling-kNN": {
        "method": "schilling_knn",
        "target_null": "H0^law: P0=P1",
        "validity_regime": "iid_metric_measure",
        "sampling_unit": "individual iid point",
        "kernel_or_distance": "k-NN same-label directed edge count",
        "is_characteristic": False,
        "requires": "pooled k-NN graph cached outside permutation loop",
        "literature": "Schilling 1986 (C2)",
        "consistency_conditions": "graph-test conditions; not characteristic-kernel",
        "is_full_point_law_test": True,
    },
    "Rosenbaum-CrossMatch": {
        "method": "rosenbaum_crossmatch",
        "target_null": "H0^law: P0=P1",
        "validity_regime": "iid_metric_measure",
        "sampling_unit": "individual iid point",
        "kernel_or_distance": "minimum-weight non-bipartite matching cross-pair count",
        "is_characteristic": False,
        "requires": "pooled optimal matching cached outside permutation loop; networkx",
        "literature": "Rosenbaum 2005 (C3)",
        "consistency_conditions": "matching graph conditions; valid permutation procedure",
        "is_full_point_law_test": True,
    },
    "SlicedWasserstein": {
        "method": "sliced_wasserstein_test",
        "target_null": "H0^law: P0=P1 (sensitivity)",
        "validity_regime": "iid_metric_measure",
        "sampling_unit": "individual iid point",
        "kernel_or_distance": "sliced Wasserstein-1 average (fixed projections)",
        "is_characteristic": False,
        "requires": "fixed projections, fixed seed; pooled label permutation",
        "literature": "Ramdas et al. 2015 (D1)",
        "consistency_conditions": "finite projection family is sensitivity statistic unless identifying family shown",
        "is_full_point_law_test": False,
        "note": "labelled as sensitivity, not universally identifying unless proven",
    },
    "ClassifierTwoSampleTest-logistic": {
        "method": "classifier_two_sample_test",
        "method_variant": "logistic",
        "target_null": "H0^law: P0=P1 (relative to classifier class)",
        "validity_regime": "iid_metric_measure",
        "sampling_unit": "individual iid point",
        "kernel_or_distance": "held-out accuracy (logistic regression)",
        "is_characteristic": False,
        "requires": "stratified train/test split, fixed hyperparameters, held-out label permutation",
        "literature": "Lopez-Paz & Oquab 2017 (E1)",
        "consistency_conditions": "depends on classifier class; not universally consistent if misspecified",
        "is_full_point_law_test": False,
    },
    "ClassifierTwoSampleTest-rf": {
        "method": "classifier_two_sample_test",
        "method_variant": "rf",
        "target_null": "H0^law: P0=P1 (relative to classifier class)",
        "validity_regime": "iid_metric_measure",
        "sampling_unit": "individual iid point",
        "kernel_or_distance": "held-out accuracy (random forest)",
        "is_characteristic": False,
        "requires": "stratified train/test split, fixed hyperparameters, held-out label permutation",
        "literature": "Lopez-Paz & Oquab 2017 (E1)",
        "consistency_conditions": "depends on classifier class; not universally consistent if misspecified",
        "is_full_point_law_test": False,
    },
}


# ---------------------------------------------------------------------------
# 2.1 Required: PointMMD-Gaussian

def point_mmd_gaussian(
    cloud0,
    cloud1,
    *,
    regime: str = REGIME_I,
    bandwidth: Optional[float] = None,
    bandwidth_is_median_heuristic: bool = False,
    median_heuristic_seed: int = 0,
    kernel: str = "gaussian",
    n_perm: int = 199,
    exact: bool = False,
    max_exact_permutations: int = 100_000,
    seed: Optional[int] = 0,
) -> dict:
    """Point-level Gaussian MMD with pooled kernel and point-label permutation.

    One pooled Gram matrix is built with a bounded Gaussian kernel whose
    bandwidth is fixed before treatment labels are used.  The statistic is
    the usual biased V-statistic MMD^2

        mean(K[X,X]) + mean(K[Y,Y]) - 2 mean(K[X,Y])

    and calibration is by pooled point-label permutation preserving n0, n1.

    If ``bandwidth is None`` the pooled median heuristic is computed once
    from the unlabeled pooled points and recorded.  Do not estimate bandwidth
    separately by arm or select it using rejection outcomes.
    """
    _require_regime("PointMMD-Gaussian", regime)
    x0, x1 = _validate_cloud_pair(cloud0, cloud1)
    if kernel.lower() != "gaussian":
        raise ValueError("only gaussian kernel is supported for the primary comparator")
    pooled = np.vstack([x0, x1])
    n0 = len(x0)
    n = len(pooled)

    # bandwidth handling
    if bandwidth is None:
        bandwidth_val = _median_heuristic_bandwidth(pooled, seed=median_heuristic_seed)
        bandwidth_is_median_heuristic = True
    else:
        bandwidth_val = float(bandwidth)
        if bandwidth_val <= 0:
            raise ValueError("bandwidth must be positive")
    # record original request for audit
    bandwidth_requested = bandwidth

    def run():
        gram = _gaussian_gram(pooled, bandwidth_val)
        group0 = np.zeros(n, dtype=bool)
        group0[:n0] = True
        observed = _mmd2_from_gram(gram, group0)
        masks, exact_used = _label_masks(
            n, n0, n_perm=n_perm, exact=exact,
            max_exact_permutations=max_exact_permutations, seed=seed,
        )
        null = np.asarray([_mmd2_from_gram(gram, m) for m in masks], dtype=float)
        # p_value uses (1+count)/(1+n_perm) convention
        if exact_used:
            pval = float(np.mean(null >= observed))
        else:
            pval = p_value(float(observed), null, alternative="greater")
        return {
            "candidate": "PointMMD-Gaussian",
            "method": "point_mmd_gaussian",
            "regime": regime,
            "inferential_target": "H0^law: P0=P1",
            "target_null": "H0^law: P0=P1",
            "validity_regime": regime,
            "sampling_unit": "individual iid point",
            "statistic": float(observed),
            "pvalue": float(pval),
            "null_statistics": null,
            "n_permutations": int(len(null)),
            "exact_enumeration": bool(exact_used),
            "permutation_group": f"all point-label splits with n0={n0}, n1={n - n0}",
            "n0": int(len(x0)),
            "n1": int(len(x1)),
            "d": int(pooled.shape[1]),
            "m": np.nan,
            "K0": np.nan,
            "K1": np.nan,
            "effective_sample_size_total": int(n),
            "unused_points0": 0,
            "unused_points1": 0,
            "kernel_or_distance": "Gaussian",
            "bandwidth_or_tuning": float(bandwidth_val),
            "bandwidth": float(bandwidth_val),
            "bandwidth_requested": bandwidth_requested,
            "bandwidth_is_median_heuristic": bool(bandwidth_is_median_heuristic),
            "median_heuristic_seed": int(median_heuristic_seed) if bandwidth_is_median_heuristic else None,
            "kernel": "Gaussian",
            "diagnostics": {
                "exchangeability_basis": "iid point-level exchangeability",
                "kernel": "Gaussian",
                "bandwidth": float(bandwidth_val),
                "bandwidth_fixed_before_labels": True,
                "bandwidth_is_median_heuristic": bool(bandwidth_is_median_heuristic),
                "bandwidth_computed_from_pooled_unlabeled": bool(bandwidth_is_median_heuristic),
                "gram_cached": True,
                "gram_recomputed_in_permutation_loop": False,
                "kernel_characteristicness": "Gaussian kernel is characteristic on Euclidean space (Gretton et al. 2012; Simon-Gabriel & Scholkopf 2018)",
                "point_law_identification": "identifies P0=P1 under characteristic kernel moment/measurability conditions",
                "statistic_type": "biased V-statistic MMD^2 (includes diagonal)",
                "is_full_point_law_test": True,
            },
        }

    return _measure(run)


# ---------------------------------------------------------------------------
# EnergyDistance

def energy_distance_test(
    cloud0,
    cloud1,
    *,
    regime: str = REGIME_I,
    n_perm: int = 199,
    exact: bool = False,
    max_exact_permutations: int = 100_000,
    seed: Optional[int] = 0,
) -> dict:
    """Euclidean energy distance with pooled distance matrix and permutation.

    Statistic (biased V-statistic / distance MMD):

        E = 2*mean(D[X,Y]) - mean(D[X,X]) - mean(D[Y,Y])

    calibrated by pooled point-label permutation.  Distances are computed
    once and reused.  Documented as V-statistic; an unbiased U-statistic
    variant would exclude the diagonal.
    """
    _require_regime("EnergyDistance", regime)
    x0, x1 = _validate_cloud_pair(cloud0, cloud1)
    pooled = np.vstack([x0, x1])
    n0 = len(x0)
    n = len(pooled)

    def _energy_from_dist(D: np.ndarray, mask: np.ndarray) -> float:
        mask = np.asarray(mask, dtype=bool)
        i0 = np.flatnonzero(mask)
        i1 = np.flatnonzero(~mask)
        if len(i0) == 0 or len(i1) == 0:
            raise ValueError("both groups must be non-empty")
        # V-statistic includes diagonal zeros
        within0 = D[np.ix_(i0, i0)].mean() if len(i0) else 0.0
        within1 = D[np.ix_(i1, i1)].mean() if len(i1) else 0.0
        between = D[np.ix_(i0, i1)].mean() if len(i0) and len(i1) else 0.0
        return float(2.0 * between - within0 - within1)

    def run():
        D = _euclidean_distance_matrix(pooled)
        group0 = np.zeros(n, dtype=bool)
        group0[:n0] = True
        observed = _energy_from_dist(D, group0)
        masks, exact_used = _label_masks(
            n, n0, n_perm=n_perm, exact=exact,
            max_exact_permutations=max_exact_permutations, seed=seed,
        )
        null = np.asarray([_energy_from_dist(D, m) for m in masks], dtype=float)
        if exact_used:
            pval = float(np.mean(null >= observed))
        else:
            pval = p_value(float(observed), null, alternative="greater")
        return {
            "candidate": "EnergyDistance",
            "method": "energy_distance",
            "regime": regime,
            "inferential_target": "H0^law: P0=P1",
            "target_null": "H0^law: P0=P1",
            "validity_regime": regime,
            "sampling_unit": "individual iid point",
            "statistic": float(observed),
            "pvalue": float(pval),
            "null_statistics": null,
            "n_permutations": int(len(null)),
            "exact_enumeration": bool(exact_used),
            "permutation_group": f"all point-label splits with n0={n0}, n1={n - n0}",
            "n0": int(len(x0)),
            "n1": int(len(x1)),
            "d": int(pooled.shape[1]),
            "m": np.nan,
            "K0": np.nan,
            "K1": np.nan,
            "effective_sample_size_total": int(n),
            "unused_points0": 0,
            "unused_points1": 0,
            "kernel_or_distance": "Euclidean",
            "bandwidth_or_tuning": np.nan,
            "distance": "Euclidean",
            "diagnostics": {
                "exchangeability_basis": "iid point-level exchangeability",
                "distance": "Euclidean",
                "distance_matrix_cached": True,
                "distance_recomputed_in_permutation_loop": False,
                "statistic_type": "biased V-statistic energy distance (includes diagonal; 2*between - within0 - within1)",
                "unbiased_variant": "U-statistic would exclude diagonal; not used here",
                "moment_assumptions": "finite first moment required for identification (Szekely & Rizzo 2013)",
                "kernel_characteristicness": "energy distance is characteristic under stated moment conditions (equivalence to MMD, Sejdinovic et al. 2013)",
                "point_law_identification": "identifies P0=P1 under finite-moment conditions",
                "is_full_point_law_test": True,
            },
        }

    return _measure(run)


# ---------------------------------------------------------------------------
# Friedman-Rafsky MST

def friedman_rafsky_mst(
    cloud0,
    cloud1,
    *,
    regime: str = REGIME_I,
    n_perm: int = 199,
    exact: bool = False,
    max_exact_permutations: int = 100_000,
    seed: Optional[int] = 0,
) -> dict:
    """Friedman-Rafsky MST test: pooled MST, cross-edge count, label permutation.

    The Euclidean MST is constructed once on the pooled points.  The statistic
    is the number of cross-arm MST edges (equivalently runs).  Direction is
    fixed before looking at results: fewer cross edges = more evidence against
    H0, so p-value is left-tail (small cross count is extreme).  Ties are
    resolved deterministically by pooled point index via the distance matrix
    ordering and scipy's deterministic MST.

    Reports vertices, edges, components, and exact statistic definition.
    MST is a representation of pooled geometry, not an identifying kernel
    embedding; claims use graph-test literature.
    """
    _require_regime("FriedmanRafsky-MST", regime)
    x0, x1 = _validate_cloud_pair(cloud0, cloud1)
    pooled = np.vstack([x0, x1])
    n0 = len(x0)
    n = len(pooled)

    def run():
        D = _euclidean_distance_matrix(pooled)
        # Use a complete-graph Prim implementation so duplicate points remain
        # connected by zero-length edges rather than being treated as absent.
        rows, cols = _deterministic_mst_edges(D)
        n_edges = len(rows)
        n_vertices = n
        # Observed cross count
        group0 = np.zeros(n, dtype=bool)
        group0[:n0] = True
        # Count cross edges: endpoints have different labels
        # mst edges are directed but we count undirected
        def cross_count(mask: np.ndarray) -> int:
            mask = np.asarray(mask, dtype=bool)
            cnt = 0
            for u, v in zip(rows, cols):
                if mask[u] != mask[v]:
                    cnt += 1
            return int(cnt)

        observed = cross_count(group0)
        masks, exact_used = _label_masks(
            n, n0, n_perm=n_perm, exact=exact,
            max_exact_permutations=max_exact_permutations, seed=seed,
        )
        null = np.asarray([cross_count(m) for m in masks], dtype=float)
        # left-tail: small cross count is extreme
        if exact_used:
            pval = float(np.mean(null <= observed))
        else:
            pval = _permutation_pvalue_less(float(observed), null)

        # The complete finite Euclidean graph is connected, including ties.
        n_components = 1
        # For reporting, also compute expected cross edges under null hypergeometric
        # (not used for p-value, just diagnostic)

        return {
            "candidate": "FriedmanRafsky-MST",
            "method": "friedman_rafsky_mst",
            "regime": regime,
            "inferential_target": "H0^law: P0=P1",
            "target_null": "H0^law: P0=P1",
            "validity_regime": regime,
            "sampling_unit": "individual iid point",
            "statistic": float(observed),
            "statistic_raw": int(observed),
            "pvalue": float(pval),
            "null_statistics": null,
            "n_permutations": int(len(null)),
            "exact_enumeration": bool(exact_used),
            "permutation_group": f"all point-label splits with n0={n0}, n1={n - n0}",
            "n0": int(len(x0)),
            "n1": int(len(x1)),
            "d": int(pooled.shape[1]),
            "m": np.nan,
            "K0": np.nan,
            "K1": np.nan,
            "effective_sample_size_total": int(n),
            "unused_points0": 0,
            "unused_points1": 0,
            "kernel_or_distance": "Euclidean MST",
            "bandwidth_or_tuning": np.nan,
            "n_vertices": int(n_vertices),
            "n_edges": int(n_edges),
            "n_components": int(n_components),
            "statistic_definition": "number of cross-arm MST edges (small is extreme; left-tail)",
            "statistic_direction": "reject for small cross-edge count",
            "diagnostics": {
                "exchangeability_basis": "iid point-level exchangeability; MST fixed under permutation",
                "graph": "Euclidean MST on pooled points (complete graph, Euclidean distances)",
                "graph_cached": True,
                "graph_recomputed_in_permutation_loop": False,
                "n_vertices": int(n_vertices),
                "n_edges": int(n_edges),
                "n_components": int(n_components),
                "tie_resolution": "deterministic via pooled point index ordering; distance matrix order",
                "statistic_definition": "number of cross-arm MST edges",
                "statistic_direction": "small cross count is extreme (left-tail)",
                "kernel_characteristicness": "none; graph statistic, not characteristic kernel (Friedman & Rafsky 1979)",
                "is_full_point_law_test": True,
                "consistency_note": "valid permutation test under iid null; consistency per graph-test literature",
            },
        }

    return _measure(run)


# ---------------------------------------------------------------------------
# Schilling kNN

def schilling_knn(
    cloud0,
    cloud1,
    *,
    regime: str = REGIME_I,
    k: int = 1,
    directed: bool = True,
    n_perm: int = 199,
    exact: bool = False,
    max_exact_permutations: int = 100_000,
    seed: Optional[int] = 0,
) -> dict:
    """Schilling k-NN test: pooled k-NN graph, same-label edge count.

    The pooled k-nearest-neighbour graph is constructed once.  Primary choice
    k=1; sensitivity panel uses k in {1,5,10}.  Fixed choice of directed vs
    symmetrised before running.  Uses same-label (or equivalently cross-label)
    edge count as statistic and calibrates with pooled point-label permutations.
    Ties resolved deterministically via index ordering with stable sort.
    """
    _require_regime("Schilling-kNN", regime)
    if int(k) < 1:
        raise ValueError("k must be >= 1")
    k = int(k)
    x0, x1 = _validate_cloud_pair(cloud0, cloud1)
    pooled = np.vstack([x0, x1])
    n0 = len(x0)
    n = len(pooled)
    if k >= n:
        raise ValueError(f"k={k} must be < pooled n={n}")

    def run():
        from scipy.spatial.distance import cdist

        D = cdist(pooled, pooled, metric="euclidean")
        # For each point, find k nearest neighbours excluding itself, stable sort
        # Use mergesort for deterministic tie handling, then sort by distance then index
        # argsort with kind='mergesort' is stable
        neighbours = np.empty((n, k), dtype=int)
        for i in range(n):
            dists = D[i]
            # set self distance to inf so it is not selected
            dists_i = dists.copy()
            dists_i[i] = np.inf
            # stable argsort
            order = np.argsort(dists_i, kind="mergesort")
            neighbours[i] = order[:k]

        # Directed edge list: (i, neighbours[i,j])
        directed_edges = [(int(i), int(neighbours[i, j])) for i in range(n) for j in range(k)]
        n_directed = len(directed_edges)

        if directed:
            edge_list = directed_edges
            n_edges_report = n_directed
            edge_type = "directed"
        else:
            # symmetrise: undirected edge if either direction exists
            # Build set of undirected edges
            und = set()
            for u, v in directed_edges:
                a, b = (u, v) if u < v else (v, u)
                und.add((a, b))
            edge_list = list(und)
            n_edges_report = len(edge_list)
            edge_type = "undirected_symmetrised"

        def same_label_count(mask: np.ndarray) -> int:
            mask = np.asarray(mask, dtype=bool)
            cnt = 0
            for u, v in edge_list:
                if mask[u] == mask[v]:
                    cnt += 1
            return int(cnt)

        group0 = np.zeros(n, dtype=bool)
        group0[:n0] = True
        observed = same_label_count(group0)
        masks, exact_used = _label_masks(
            n, n0, n_perm=n_perm, exact=exact,
            max_exact_permutations=max_exact_permutations, seed=seed,
        )
        null = np.asarray([same_label_count(m) for m in masks], dtype=float)
        if exact_used:
            pval = float(np.mean(null >= observed))
        else:
            pval = p_value(float(observed), null, alternative="greater")

        return {
            "candidate": f"Schilling-kNN-k{k}",
            "method": "schilling_knn",
            "regime": regime,
            "inferential_target": "H0^law: P0=P1",
            "target_null": "H0^law: P0=P1",
            "validity_regime": regime,
            "sampling_unit": "individual iid point",
            "statistic": float(observed),
            "statistic_raw": int(observed),
            "pvalue": float(pval),
            "null_statistics": null,
            "n_permutations": int(len(null)),
            "exact_enumeration": bool(exact_used),
            "permutation_group": f"all point-label splits with n0={n0}, n1={n - n0}",
            "n0": int(len(x0)),
            "n1": int(len(x1)),
            "d": int(pooled.shape[1]),
            "m": np.nan,
            "K0": np.nan,
            "K1": np.nan,
            "effective_sample_size_total": int(n),
            "unused_points0": 0,
            "unused_points1": 0,
            "kernel_or_distance": f"k-NN (k={k}, {edge_type})",
            "bandwidth_or_tuning": float(k),
            "k": int(k),
            "directed": bool(directed),
            "n_edges": int(n_edges_report),
            "n_directed_edges": int(n_directed),
            "statistic_definition": "number of same-label directed (or symmetrised) k-NN edges (large is extreme)",
            "statistic_direction": "reject for large same-label count",
            "diagnostics": {
                "exchangeability_basis": "iid point-level exchangeability; k-NN graph fixed under permutation",
                "graph": f"pooled k-NN graph (k={k}, {edge_type}, Euclidean)",
                "graph_cached": True,
                "graph_recomputed_in_permutation_loop": False,
                "k": int(k),
                "directed": bool(directed),
                "n_directed_edges": int(n_directed),
                "n_edges": int(n_edges_report),
                "tie_resolution": "deterministic stable sort (mergesort) with index tie-break",
                "statistic_definition": "same-label edge count",
                "statistic_direction": "large is extreme (greater tail)",
                "kernel_characteristicness": "none; graph statistic, not characteristic kernel (Schilling 1986)",
                "is_full_point_law_test": True,
                "consistency_note": "valid permutation test; consistency per Schilling conditions",
            },
        }

    return _measure(run)


# ---------------------------------------------------------------------------
# Rosenbaum CrossMatch

def rosenbaum_crossmatch(
    cloud0,
    cloud1,
    *,
    regime: str = REGIME_I,
    n_perm: int = 199,
    exact: bool = False,
    max_exact_permutations: int = 100_000,
    seed: Optional[int] = 0,
    max_n_for_exact_matching: int = 500,
) -> dict:
    """Rosenbaum cross-match test: minimum-weight matching, cross-pair count.

    Constructs a minimum-weight non-bipartite matching of the pooled points
    (Euclidean distances) and counts cross-arm pairs.  Matching is computed
    once per replication and never recomputed inside the label loop.
    Uses networkx.max_weight_matching with negative distances (blossom algorithm).
    If pooled n is odd, one point is left unmatched (documented).

    Dependency: networkx.  Placed in optional benchmark extra; failure to
    import is recorded as a refusal rather than silent skip.
    """
    _require_regime("Rosenbaum-CrossMatch", regime)
    x0, x1 = _validate_cloud_pair(cloud0, cloud1)
    pooled = np.vstack([x0, x1])
    n0 = len(x0)
    n = len(pooled)

    def run():
        try:
            import networkx as nx
            from networkx.algorithms.matching import max_weight_matching
        except Exception as exc:
            raise RuntimeError(f"networkx required for CrossMatch but failed to import: {exc}")

        if n > max_n_for_exact_matching:
            raise ValueError(
                f"pooled n={n} exceeds CrossMatch limit {max_n_for_exact_matching}; refused for budget (matching O(n^3))"
            )
        from scipy.spatial.distance import cdist, pdist, squareform
        # Use cdist symmetric; for matching we need complete graph edge weights = -dist
        # Build edge list for max_weight_matching: list of (u, v, weight)
        # For n=500, edges ~125k, feasible.
        # Optimize: use pdist to get condensed, then iterate.
        D = cdist(pooled, pooled, metric="euclidean")
        iu, ju = np.triu_indices(n, k=1)
        weights = -D[iu, ju]
        edge_tuples = list(zip(iu.tolist(), ju.tolist(), weights.tolist()))
        G = nx.Graph()
        G.add_weighted_edges_from(edge_tuples)
        matching = max_weight_matching(G, maxcardinality=True, weight="weight")
        # matching is set of (u, v) tuples (unordered)
        matching_list = list(matching)
        n_pairs = len(matching_list)
        n_unmatched = n - 2 * n_pairs
        # Observed cross pairs
        group0 = np.zeros(n, dtype=bool)
        group0[:n0] = True

        def cross_pair_count(mask: np.ndarray) -> int:
            mask = np.asarray(mask, dtype=bool)
            cnt = 0
            for u, v in matching_list:
                if mask[u] != mask[v]:
                    cnt += 1
            return int(cnt)

        observed = cross_pair_count(group0)
        masks, exact_used = _label_masks(
            n, n0, n_perm=n_perm, exact=exact,
            max_exact_permutations=max_exact_permutations, seed=seed,
        )
        null = np.asarray([cross_pair_count(m) for m in masks], dtype=float)
        # Under alternative, cross pairs decrease (within matches increase) => left-tail
        if exact_used:
            pval = float(np.mean(null <= observed))
        else:
            pval = _permutation_pvalue_less(float(observed), null)

        return {
            "candidate": "Rosenbaum-CrossMatch",
            "method": "rosenbaum_crossmatch",
            "regime": regime,
            "inferential_target": "H0^law: P0=P1",
            "target_null": "H0^law: P0=P1",
            "validity_regime": regime,
            "sampling_unit": "individual iid point",
            "statistic": float(observed),
            "statistic_raw": int(observed),
            "pvalue": float(pval),
            "null_statistics": null,
            "n_permutations": int(len(null)),
            "exact_enumeration": bool(exact_used),
            "permutation_group": f"all point-label splits with n0={n0}, n1={n - n0}",
            "n0": int(len(x0)),
            "n1": int(len(x1)),
            "d": int(pooled.shape[1]),
            "m": np.nan,
            "K0": np.nan,
            "K1": np.nan,
            "effective_sample_size_total": int(n),
            "unused_points0": 0,
            "unused_points1": 0,
            "kernel_or_distance": "minimum-weight non-bipartite matching (Euclidean)",
            "bandwidth_or_tuning": np.nan,
            "n_pairs": int(n_pairs),
            "n_unmatched": int(n_unmatched),
            "matching_algorithm": "networkx.max_weight_matching (Edmonds blossom) on complete graph with weight=-Euclidean distance, maxcardinality=True",
            "odd_n_rule": "if pooled n odd, one point left unmatched (maximum cardinality matching)",
            "statistic_definition": "number of cross-arm matched pairs (small is extreme; left-tail)",
            "statistic_direction": "reject for small cross-pair count",
            "diagnostics": {
                "exchangeability_basis": "iid point-level exchangeability; matching fixed under permutation",
                "matching": "minimum-weight non-bipartite matching (blossom) on complete Euclidean graph",
                "matching_cached": True,
                "matching_recomputed_in_permutation_loop": False,
                "n_pairs": int(n_pairs),
                "n_unmatched": int(n_unmatched),
                "odd_n_rule": "one point unmatched if n odd (maximum cardinality)",
                "statistic_definition": "cross-pair count",
                "statistic_direction": "small is extreme (left-tail)",
                "dependency": "networkx",
                "is_full_point_law_test": True,
                "consistency_note": "valid permutation test; consistency per Rosenbaum 2005",
            },
        }

    return _measure(run)


# ---------------------------------------------------------------------------
# Sliced Wasserstein

def sliced_wasserstein_test(
    cloud0,
    cloud1,
    *,
    regime: str = REGIME_I,
    n_projections: int = 100,
    projection_seed: int = 0,
    p: int = 1,
    n_perm: int = 199,
    exact: bool = False,
    max_exact_permutations: int = 100_000,
    seed: Optional[int] = 0,
) -> dict:
    """Sliced Wasserstein two-sample test: fixed projections, pooled permutation.

    Computational version: fixed-regularisation sliced-Wasserstein sensitivity.
    Regularisation = none (exact 1D Wasserstein); number of projections and
    projection seed are fixed before labels and recorded in every result.
    Uses Euclidean projections onto random unit directions (Gaussian then
    normalised), then exact 1D Wasserstein-1 (via scipy.stats.wasserstein_distance)
    per projection and averages.

    This is geometrically interpretable but finite projection family is a
    sensitivity statistic unless identifying family is proven; labelled honestly.
    """
    _require_regime("SlicedWasserstein", regime)
    if int(n_projections) < 1:
        raise ValueError("n_projections must be >= 1")
    if int(p) != 1:
        raise ValueError("only p=1 is implemented by scipy.stats.wasserstein_distance")
    p = 1
    x0, x1 = _validate_cloud_pair(cloud0, cloud1)
    pooled = np.vstack([x0, x1])
    n0 = len(x0)
    n = len(pooled)
    d = pooled.shape[1]
    n_projections = int(n_projections)

    def run():
        from scipy.stats import wasserstein_distance

        rng = np.random.default_rng(projection_seed)
        dirs = rng.normal(size=(n_projections, d))
        # normalise
        norms = np.linalg.norm(dirs, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)
        dirs = dirs / norms

        # pooled projections: shape (n_projections, n)
        # pooled @ dirs.T -> (n, n_projections) then transpose
        proj_pooled = pooled @ dirs.T  # (n, n_projections)
        proj_pooled = proj_pooled.T  # (n_projections, n)

        def sliced_stat(mask: np.ndarray) -> float:
            mask = np.asarray(mask, dtype=bool)
            total = 0.0
            for pr in range(n_projections):
                vals = proj_pooled[pr]
                a = vals[mask]
                b = vals[~mask]
                # wasserstein_distance handles unequal sizes via sorting + interpolation
                total += wasserstein_distance(a, b)
            return float(total / n_projections)

        group0 = np.zeros(n, dtype=bool)
        group0[:n0] = True
        observed = sliced_stat(group0)
        masks, exact_used = _label_masks(
            n, n0, n_perm=n_perm, exact=exact,
            max_exact_permutations=max_exact_permutations, seed=seed,
        )
        null = np.asarray([sliced_stat(m) for m in masks], dtype=float)
        if exact_used:
            pval = float(np.mean(null >= observed))
        else:
            pval = p_value(float(observed), null, alternative="greater")

        return {
            "candidate": "SlicedWasserstein",
            "method": "sliced_wasserstein",
            "regime": regime,
            "inferential_target": "H0^law: P0=P1 (sensitivity; finite projections)",
            "target_null": "H0^law: P0=P1 (sensitivity)",
            "validity_regime": regime,
            "sampling_unit": "individual iid point",
            "statistic": float(observed),
            "pvalue": float(pval),
            "null_statistics": null,
            "n_permutations": int(len(null)),
            "exact_enumeration": bool(exact_used),
            "permutation_group": f"all point-label splits with n0={n0}, n1={n - n0}",
            "n0": int(len(x0)),
            "n1": int(len(x1)),
            "d": int(d),
            "m": np.nan,
            "K0": np.nan,
            "K1": np.nan,
            "effective_sample_size_total": int(n),
            "unused_points0": 0,
            "unused_points1": 0,
            "kernel_or_distance": f"sliced Wasserstein-1 (n_projections={n_projections})",
            "bandwidth_or_tuning": float(n_projections),
            "n_projections": int(n_projections),
            "projection_seed": int(projection_seed),
            "p": int(p),
            "transport_solver": "exact 1D Wasserstein via sorting (scipy.stats.wasserstein_distance)",
            "regularisation": "none (exact 1D)",
            "diagnostics": {
                "exchangeability_basis": "iid point-level exchangeability; projections fixed before labels",
                "distance": f"sliced Wasserstein-{p} average over {n_projections} random projections",
                "n_projections": int(n_projections),
                "projection_seed": int(projection_seed),
                "projection_distribution": "Gaussian then normalised to unit sphere (isotropic)",
                "transport_solver": "exact 1D Wasserstein via sorting (scipy)",
                "regularisation": "none",
                "is_identifying": False,
                "is_full_point_law_test": False,
                "honest_label": "finite-projection sensitivity statistic; not promoted to universally identifying unless proven",
                "reference": "Ramdas et al. 2015 (D1)",
            },
        }

    return _measure(run)


# ---------------------------------------------------------------------------
# Classifier two-sample test

def classifier_two_sample_test(
    cloud0,
    cloud1,
    *,
    regime: str = REGIME_I,
    classifier: str = "logistic",
    test_fraction: float = 0.5,
    split_seed: Optional[int] = 0,
    n_perm: int = 199,
    exact: bool = False,
    max_exact_permutations: int = 100_000,
    seed: Optional[int] = 0,
) -> dict:
    """Sample-split classifier two-sample test with held-out label permutation.

    Fixed model family: logistic regression or random forest.  Training/test
    split is stratified and independent of treatment labels via predeclared
    seed.  Hyperparameters are fixed.  Simplest valid implementation:
    fit on training subset and evaluate held-out accuracy; calibrate via
    held-out-label permutation conditional on fixed training fit (no retraining
    inside permutation loop).  Reports classifier class, split fraction,
    seeds, training time, and tuning info.

    Targets point-law equality only relative to classifier class and protocol;
    not automatically universally consistent.
    """
    _require_regime("ClassifierTwoSampleTest", regime)
    classifier = str(classifier).lower()
    if classifier not in ("logistic", "rf", "random_forest", "logreg"):
        raise ValueError("classifier must be 'logistic' or 'rf'")
    # normalise aliases
    if classifier in ("rf", "random_forest"):
        clf_name = "rf"
        clf_label = "ClassifierTwoSampleTest-rf"
    else:
        clf_name = "logistic"
        clf_label = "ClassifierTwoSampleTest-logistic"
    if not 0.0 < float(test_fraction) < 1.0:
        raise ValueError("test_fraction must be in (0,1)")
    test_fraction = float(test_fraction)
    x0, x1 = _validate_cloud_pair(cloud0, cloud1)
    pooled = np.vstack([x0, x1])
    n0 = len(x0)
    n1 = len(x1)
    n = len(pooled)
    y = np.concatenate([np.zeros(n0, dtype=int), np.ones(n1, dtype=int)])

    def run():
        from sklearn.linear_model import LogisticRegression
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.model_selection import train_test_split

        # stratified split preserving label proportions
        # train_test_split expects X, y
        split_seed_int = int(split_seed) if split_seed is not None else 0
        try:
            X_train, X_test, y_train, y_test = train_test_split(
                pooled, y, test_size=test_fraction, random_state=split_seed_int, stratify=y
            )
        except ValueError as exc:
            raise ValueError(f"stratified split failed (n0={n0}, n1={n1}, test_fraction={test_fraction}): {exc}")

        n_train = len(X_train)
        n_test = len(X_test)
        # Count per group in test
        n0_test = int(np.sum(y_test == 0))
        n1_test = int(np.sum(y_test == 1))
        if n0_test == 0 or n1_test == 0:
            raise ValueError(f"test split has empty class: n0_test={n0_test}, n1_test={n1_test}")

        t_train_start = time.perf_counter()
        if clf_name == "logistic":
            clf = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=split_seed_int)
            clf_kwargs = {"max_iter": 1000, "solver": "lbfgs"}
        else:
            clf = RandomForestClassifier(n_estimators=100, n_jobs=1, random_state=split_seed_int)
            clf_kwargs = {"n_estimators": 100, "n_jobs": 1}
        clf.fit(X_train, y_train)
        train_time = time.perf_counter() - t_train_start

        # predictions on test
        y_pred = clf.predict(X_test)
        observed_acc = float(np.mean(y_pred == y_test))

        # held-out label permutation: permute y_test labels preserving counts
        # We have n_test positions, n0_test zeros. Generate masks for test permutation.
        masks, exact_used = _label_masks(
            n_test, n0_test, n_perm=n_perm, exact=exact,
            max_exact_permutations=max_exact_permutations, seed=seed,
        )
        # masks are bool arrays length n_test where True = group0 (label 0)
        # For each mask, create permuted y_test: 0 where mask True, 1 where False
        # Then compute accuracy against fixed y_pred
        null = np.empty(len(masks), dtype=float)
        for idx, mask in enumerate(masks):
            y_test_perm = np.where(mask, 0, 1)
            null[idx] = float(np.mean(y_pred == y_test_perm))

        if exact_used:
            pval = float(np.mean(null >= observed_acc))
        else:
            pval = p_value(float(observed_acc), null, alternative="greater")

        return {
            "candidate": clf_label,
            "method": "classifier_two_sample_test",
            "method_variant": clf_name,
            "regime": regime,
            "inferential_target": "H0^law: P0=P1 (relative to classifier class)",
            "target_null": "H0^law: P0=P1 (classifier-relative)",
            "validity_regime": regime,
            "sampling_unit": "individual iid point",
            "statistic": float(observed_acc),
            "statistic_raw": float(observed_acc),
            "pvalue": float(pval),
            "null_statistics": null,
            "n_permutations": int(len(null)),
            "exact_enumeration": bool(exact_used),
            "permutation_group": f"all held-out label splits with n0_test={n0_test}, n1_test={n1_test} (conditional on training fit)",
            "calibration": "held-out label permutation conditional on fixed training fit (no retraining inside loop)",
            "alternative_calibration": "retrain under permuted training labels would be exact but is not used here (computational cost)",
            "n0": int(n0),
            "n1": int(n1),
            "n": int(n),
            "d": int(pooled.shape[1]),
            "m": np.nan,
            "K0": np.nan,
            "K1": np.nan,
            "effective_sample_size_total": int(n),
            "unused_points0": 0,
            "unused_points1": 0,
            "kernel_or_distance": f"held-out accuracy ({clf_name})",
            "bandwidth_or_tuning": np.nan,
            "classifier": clf_name,
            "classifier_class": "LogisticRegression" if clf_name == "logistic" else "RandomForestClassifier",
            "classifier_params": clf_kwargs,
            "split_fraction": float(test_fraction),
            "split_seed": int(split_seed_int),
            "permutation_seed": int(seed) if seed is not None else None,
            "n_train": int(n_train),
            "n_test": int(n_test),
            "n0_test": int(n0_test),
            "n1_test": int(n1_test),
            "train_time_seconds": float(train_time),
            "hyperparameter_tuning": "none; fixed hyperparameters, selected only on training data if any",
            "diagnostics": {
                "exchangeability_basis": "iid point-level exchangeability; training fit fixed under held-out permutation",
                "classifier": clf_name,
                "classifier_class": "LogisticRegression" if clf_name == "logistic" else "RandomForestClassifier",
                "classifier_params": clf_kwargs,
                "split_fraction": float(test_fraction),
                "split_seed": int(split_seed_int),
                "stratified": True,
                "calibration": "held-out label permutation conditional on fixed training fit",
                "calibration_alternative": "retrain under permuted training labels (exact, not used; cost)",
                "n_train": int(n_train),
                "n_test": int(n_test),
                "n0_test": int(n0_test),
                "n1_test": int(n1_test),
                "train_time_seconds": float(train_time),
                "statistic_definition": "held-out classification accuracy (large is extreme)",
                "statistic_direction": "reject for large accuracy",
                "hyperparameter_tuning": "none; fixed before labels",
                "is_full_point_law_test": False,
                "limitation": "targets point-law equality only relative to classifier class; not universally consistent if misspecified (Lopez-Paz & Oquab 2017)",
            },
        }

    return _measure(run)


In [ ]:
%%writefile tda2s/tests/single_cloud.py
"""Prototype tests for the exactly-two-cloud Regime-I problem.

This module implements the applicable Phase 5B candidates for the scoped
Regime-I benchmark:

``SC-A``
    pooled-point label permutation.  The persistence diagrams are recomputed
    for every label split, so this is a full point-law exchangeability
    baseline, not a barcode-law test.

``SC-B``
    a fixed disjoint partition into subclouds of size ``m`` followed by a
    diagram-level MMD permutation test.  The only barcode replicates are the
    disjoint blocks, and the returned ``K0`` and ``K1`` fields make that
    effective sample size explicit.

``SC-C``
    a finite-vector persistent-Betti contrast calibrated by a pooled
    point-level bootstrap.  The smoothed version follows the
    Roycraft--Krebs--Polonik bootstrap idea by resampling points and adding
    Gaussian kernel noise.  The ordinary bootstrap is exposed as a negative
    control.  This candidate is deliberately labelled as a finite-vector
    mean test and must not be reported as a test of the full fixed-``m``
    barcode law.

The common return object is a dictionary rather than a dataclass so that it
can be serialized by the Phase 5C fleet without a custom encoder.  Arrays are
kept in the object for diagnostics and reproducibility; callers that need
JSON should convert them explicitly.

The implementation is intentionally conservative about routing.  All three
methods accept only ``iid_metric_measure``.  A spatial process, a fixed cloud
without a sampling model, or a Bayesian generative model requires a new
observation-model lock and raises ``ValueError`` here.

The production entry point is ``sc_b_production_test``.  It freezes the
Phase-5 target at ``m=25``, the VR filtration, degrees ``(0, 1)``, and the
pre-registered kernel bandwidth.  The lower-level ``sc_b_disjoint_mmd``
function remains available for Phase-5 sensitivity work, including unlocked
values of ``m``.
"""
from __future__ import annotations

import itertools
import math
import resource
import time
import tracemalloc
from typing import Iterable, Optional, Sequence, Tuple

import numpy as np

from tda2s.ph import compute_diagrams
from tda2s.resample import p_value

REGIME_I = "iid_metric_measure"
LOCKED_M = 25
LOCKED_HOMOLOGY_DIMS = (0, 1)
DEFAULT_KERNEL_BANDWIDTH = 0.10
DEFAULT_GRID = np.linspace(0.0, 1.0, 9)
#: Matches tda2s.ph.PhParams defaults so filtration-level options are never
#: silently replaced by a different default once the user selects them.
DEFAULT_GRID_SIZE = 64
DEFAULT_DTM_K = 20
PRODUCTION_MIN_BLOCKS = 5
PRODUCTION_API_VERSION = "sc-b-v1"
DEFAULT_RAW_POINT_KERNEL = "gaussian"
DEFAULT_RAW_POINT_BANDWIDTH = 0.10
DEFAULT_RAW_BAG_BANDWIDTH = 0.25
RAW_KERNEL_VARIANTS = ("gaussian_mean_embedding", "mean_pairwise")
HYBRID_ALPHA_GRID = (0.25, 0.50, 0.75, 1.00)

__all__ = [
    "REGIME_I",
    "LOCKED_M",
    "PRODUCTION_MIN_BLOCKS",
    "disjoint_partition",
    "persistent_betti_vector",
    "roycraft_reference_setting",
    "run_single_cloud_test",
    "sc_a_label_permutation",
    "sc_b_disjoint_mmd",
    "sc_b_production_test",
    "sc_b_repeated_partition_test",
    "raw_block_mmd",
    "hybrid_block_mmd",
    "sc_a_blockwise_label_permutation",
    "raw_block_repeated_partition_test",
    "sc_c_finite_vector",
    "sc_c_naive_bootstrap",
]


# ---------------------------------------------------------------------------
# Shared validation, timing, and permutation helpers


def _as_cloud(cloud, name: str) -> np.ndarray:
    points = np.asarray(cloud, dtype=float)
    if points.ndim != 2 or points.shape[0] < 2 or points.shape[1] < 1:
        raise ValueError(f"{name} must have shape (n, d) with n >= 2")
    if not np.isfinite(points).all():
        raise ValueError(f"{name} must contain only finite values")
    return points


def _validate_cloud_pair(cloud0, cloud1) -> Tuple[np.ndarray, np.ndarray]:
    x0, x1 = _as_cloud(cloud0, "cloud0"), _as_cloud(cloud1, "cloud1")
    if x0.shape[1] != x1.shape[1]:
        raise ValueError("cloud0 and cloud1 must have the same ambient dimension")
    return x0, x1


def _require_regime(candidate: str, regime: str) -> None:
    if regime != REGIME_I:
        raise ValueError(
            f"{candidate} is only valid for declared regime {REGIME_I!r}; "
            f"received {regime!r}. Reclassify the observation model before inference."
        )


def _normalise_candidate(candidate: str) -> str:
    key = str(candidate).lower().replace("_", "-")
    aliases = {
        "a": "sc-a",
        "sc-a": "sc-a",
        "pooled-label-permutation": "sc-a",
        "b": "sc-b",
        "sc-b": "sc-b",
        "disjoint-mmd": "sc-b",
        "sc-b-production": "sc-b-production",
        "production-sc-b": "sc-b-production",
        "raw-block-mmd": "raw-block-mmd",
        "rawblockmmd": "raw-block-mmd",
        "hybrid-block-mmd": "hybrid-block-mmd",
        "hybridblockmmd": "hybrid-block-mmd",
        "sc-a-block": "sc-a-block",
        "sc-a-blockwise": "sc-a-block",
        "c": "sc-c",
        "sc-c": "sc-c",
        "finite-vector": "sc-c",
    }
    if key not in aliases:
        raise ValueError(
            "candidate must be one of {'SC-A', 'SC-B', 'SC-B-production', "
            "'RawBlockMMD', 'HybridBlockMMD', 'SC-A-Block', 'SC-C'}"
        )
    return aliases[key]


def _finish(result: dict, started: float, peak_bytes: int) -> dict:
    result["runtime_seconds"] = float(time.perf_counter() - started)
    result["peak_memory_bytes"] = int(peak_bytes)
    # This makes the memory measurement interpretable when numpy or GUDHI
    # allocates outside tracemalloc's Python allocator.
    result["peak_memory_measurement"] = "tracemalloc_python_allocations"
    return result


def _current_rss_bytes() -> int:
    """Return the process high-water RSS in bytes on Unix-like systems."""
    usage = resource.getrusage(resource.RUSAGE_SELF)
    # Linux and macOS expose ru_maxrss in KiB and bytes respectively.  The
    # repository's supported execution environments are Linux, but retaining
    # the branch keeps the helper interpretable on macOS.
    scale = 1024 if __import__("sys").platform.startswith("linux") else 1
    return int(usage.ru_maxrss * scale)


def _measure(callable_):
    tracemalloc.start()
    started = time.perf_counter()
    start_rss = _current_rss_bytes()
    try:
        result = callable_()
        _, peak = tracemalloc.get_traced_memory()
        result = _finish(result, started, peak)
        result["peak_rss_bytes"] = max(start_rss, _current_rss_bytes())
        result["peak_memory_measurement"] = (
            "tracemalloc_python_allocations; peak_rss_bytes is process high-water RSS"
        )
        return result
    finally:
        tracemalloc.stop()


def _validate_ph_options(filtration: str, homology_dims: Sequence[int]) -> Tuple[int, ...]:
    dims = tuple(int(d) for d in homology_dims)
    if not dims or any(d < 0 for d in dims) or len(set(dims)) != len(dims):
        raise ValueError("homology_dims must be a non-empty sequence of distinct non-negative integers")
    if filtration not in {"vr", "ripser", "alpha", "cech", "cubical", "dtm-rips"}:
        raise ValueError(f"unknown filtration {filtration!r}")
    return dims


def _enumerated_masks(n: int, n0: int) -> Iterable[np.ndarray]:
    for indices in itertools.combinations(range(n), n0):
        mask = np.zeros(n, dtype=bool)
        mask[list(indices)] = True
        yield mask


def _label_masks(n: int, n0: int, *, n_perm: int, exact: bool,
                 max_exact_permutations: int, seed: Optional[int]) -> Tuple[list, bool]:
    if n0 <= 0 or n0 >= n:
        raise ValueError("both label groups must contain at least one observation")
    n_splits = math.comb(n, n0)
    if exact:
        if n_splits > max_exact_permutations:
            raise ValueError(
                f"exact enumeration needs {n_splits} splits, above the limit "
                f"max_exact_permutations={max_exact_permutations}")
        return list(_enumerated_masks(n, n0)), True
    if n_perm < 1:
        raise ValueError("n_perm must be >= 1")
    rng = np.random.default_rng(seed)
    masks = []
    for _ in range(int(n_perm)):
        mask = np.zeros(n, dtype=bool)
        mask[rng.permutation(n)[:n0]] = True
        masks.append(mask)
    return masks, False


def _permutation_pvalue(observed: float, null: np.ndarray, exact: bool) -> float:
    null = np.asarray(null, dtype=float)
    if null.size == 0:
        return 1.0
    if exact:
        return float(np.mean(null >= observed))
    return p_value(float(observed), null, alternative="greater")


def _ph(points: np.ndarray, *, filtration: str, homology_dims: Sequence[int],
        max_edge_length: Optional[float], grid_size: int, dtm_k: int,
        cache_dir: Optional[str]):
    return compute_diagrams(
        points,
        filtration=filtration,
        homology_dims=homology_dims,
        max_edge_length=max_edge_length,
        grid_size=grid_size,
        dtm_k=dtm_k,
        cache_dir=cache_dir,
    )


# ---------------------------------------------------------------------------
# Fixed persistence scale-space kernel and diagram-level MMD


def _pss_kernel(diagram0: np.ndarray, diagram1: np.ndarray,
                bandwidth: float) -> float:
    """Reininghaus persistence scale-space kernel on one degree.

    ``bandwidth`` is the locked raw metric bandwidth.  The reflected diagram
    term makes the kernel vanish on the diagonal of the persistence half
    plane, as in the source construction.  The joint kernel below combines
    the degree-specific universal kernels through a tensor product.
    """
    if bandwidth <= 0:
        raise ValueError("kernel bandwidth must be positive")
    f = np.asarray(diagram0, dtype=float).reshape(-1, 2)
    g = np.asarray(diagram1, dtype=float).reshape(-1, 2)
    if len(f) == 0 or len(g) == 0:
        return 0.0
    g_reflected = g[:, ::-1]
    d2 = ((f[:, None, :] - g[None, :, :]) ** 2).sum(axis=2)
    d2_reflected = ((f[:, None, :] - g_reflected[None, :, :]) ** 2).sum(axis=2)
    return float(
        (np.exp(-d2 / (8.0 * bandwidth))
         - np.exp(-d2_reflected / (8.0 * bandwidth))).sum()
        / (8.0 * np.pi * bandwidth)
    )


def _universal_diagram_kernel(diagrams0, diagrams1, bandwidth: float) -> float:
    """Characteristic joint kernel built from characteristic degree kernels.

    Kwitt et al.'s exponentiated persistence scale-space kernel is universal,
    and therefore characteristic, on the bounded diagram classes covered by
    their Proposition 2.  The locked object is a *joint* degree-0/degree-1
    diagram, so the correct product-space kernel is the tensor-product kernel
    ``prod_d k_d``.  Averaging the ``k_d`` values would retain only the two
    marginal diagram laws and would not identify their cross-degree dependence.
    """
    if len(diagrams0) != len(diagrams1):
        raise ValueError("joint diagram pairs must have the same number of degrees")
    if not diagrams0:
        raise ValueError("at least one homology degree is required")
    values = [math.exp(_pss_kernel(a, b, bandwidth))
              for a, b in zip(diagrams0, diagrams1)]
    # The tensor product is characteristic on a product of the bounded
    # per-degree diagram classes when every factor is characteristic.  It is
    # also positive definite, so the resulting MMD remains a valid RKHS
    # discrepancy.  Do not replace this with a sum or mean: those kernels can
    # be blind to changes in the dependence between homology degrees.
    return float(np.prod(values))


def _diagram_gram(diagrams: Sequence[Sequence[np.ndarray]], bandwidth: float) -> np.ndarray:
    n = len(diagrams)
    if n == 0:
        raise ValueError("at least one diagram is required")
    gram = np.empty((n, n), dtype=float)
    for i in range(n):
        for j in range(i, n):
            value = _universal_diagram_kernel(diagrams[i], diagrams[j], bandwidth)
            gram[i, j] = gram[j, i] = value
    return gram


def _mmd2_from_gram(gram: np.ndarray, group0: np.ndarray) -> float:
    gram = np.asarray(gram, dtype=float)
    group0 = np.asarray(group0, dtype=bool)
    if gram.ndim != 2 or gram.shape[0] != gram.shape[1] or gram.shape[0] != len(group0):
        raise ValueError("gram and group0 have incompatible shapes")
    i0, i1 = np.flatnonzero(group0), np.flatnonzero(~group0)
    if len(i0) == 0 or len(i1) == 0:
        raise ValueError("both MMD groups must be non-empty")
    within0 = gram[np.ix_(i0, i0)].mean()
    within1 = gram[np.ix_(i1, i1)].mean()
    between = gram[np.ix_(i0, i1)].mean()
    return float(max(within0 + within1 - 2.0 * between, 0.0))


def _joint_discrepancy(diagrams0, diagrams1, bandwidth: float) -> float:
    """Squared RKHS distance between two joint degree-tagged diagrams."""
    value = (_universal_diagram_kernel(diagrams0, diagrams0, bandwidth)
             + _universal_diagram_kernel(diagrams1, diagrams1, bandwidth)
             - 2.0 * _universal_diagram_kernel(diagrams0, diagrams1, bandwidth))
    return float(max(value, 0.0))


# ---------------------------------------------------------------------------
# Raw and hybrid fixed-size block methods


def _validate_raw_kernel_options(point_kernel: str, raw_kernel: str,
                                 point_bandwidth: float,
                                 bag_bandwidth: float) -> tuple[str, str]:
    point_kernel = str(point_kernel).lower()
    raw_kernel = str(raw_kernel).lower()
    if point_kernel not in {"gaussian", "laplacian"}:
        raise ValueError("point_kernel must be 'gaussian' or 'laplacian'")
    if raw_kernel not in RAW_KERNEL_VARIANTS:
        raise ValueError(
            "raw_kernel must be one of {'gaussian_mean_embedding', 'mean_pairwise'}"
        )
    if float(point_bandwidth) <= 0 or float(bag_bandwidth) <= 0:
        raise ValueError("point and bag kernel bandwidths must be positive")
    return point_kernel, raw_kernel


def _point_kernel_matrix(points0: np.ndarray, points1: np.ndarray,
                         point_kernel: str, bandwidth: float) -> np.ndarray:
    """Bounded characteristic point kernel on Euclidean point coordinates."""
    delta = points0[:, None, :] - points1[None, :, :]
    if point_kernel == "gaussian":
        squared = np.sum(delta * delta, axis=2)
        return np.exp(-squared / (2.0 * float(bandwidth) ** 2))
    distances = np.abs(delta).sum(axis=2)
    return np.exp(-distances / float(bandwidth))


def _raw_block_features(blocks: Sequence[np.ndarray], *, point_kernel: str,
                        point_bandwidth: float) -> list[dict]:
    """Cache the raw feature needed by every block-kernel evaluation.

    The cached self inner product and the original points are sufficient to
    evaluate the whole raw Gram matrix before the permutation loop.  Point
    order does not appear in the feature definition.
    """
    features = []
    for block in blocks:
        points = np.asarray(block, dtype=float)
        self_mean = float(np.mean(_point_kernel_matrix(
            points, points, point_kernel, point_bandwidth)))
        features.append({"points": points, "self_mean": self_mean})
    return features


def _raw_block_kernel(feature0: dict, feature1: dict, *, point_kernel: str,
                      point_bandwidth: float, bag_bandwidth: float,
                      raw_kernel: str) -> float:
    cross_mean = float(np.mean(_point_kernel_matrix(
        feature0["points"], feature1["points"], point_kernel,
        point_bandwidth)))
    if raw_kernel == "mean_pairwise":
        return cross_mean
    distance2 = max(
        feature0["self_mean"] + feature1["self_mean"] - 2.0 * cross_mean,
        0.0,
    )
    return float(np.exp(-distance2 / (2.0 * float(bag_bandwidth) ** 2)))


def _raw_block_gram(features: Sequence[dict], *, point_kernel: str,
                    point_bandwidth: float, bag_bandwidth: float,
                    raw_kernel: str) -> np.ndarray:
    n = len(features)
    if n == 0:
        raise ValueError("at least one raw block is required")
    gram = np.empty((n, n), dtype=float)
    for i in range(n):
        for j in range(i, n):
            value = _raw_block_kernel(
                features[i], features[j], point_kernel=point_kernel,
                point_bandwidth=point_bandwidth, bag_bandwidth=bag_bandwidth,
                raw_kernel=raw_kernel,
            )
            gram[i, j] = gram[j, i] = value
    return gram


def _block_target(m: int, *, barcode: bool = False,
                  raw_kernel: Optional[str] = None) -> str:
    if barcode:
        return (f"H0,{m}^bar: Phi^{m}_0:1(P0) = Phi^{m}_0:1(P1) "
                "(alpha=0 barcode-only diagnostic)")
    if raw_kernel == "mean_pairwise":
        return ("H0^raw-block-sensitivity: point-law-sensitive under the iid "
                f"product-block model, m={m}; not fully identified for "
                "unrestricted bag laws")
    return ("H0^law: P0 = P1 "
            f"(raw characteristic fixed-size block representation, m={m}, "
            "under iid point sampling)")


def _block_result_from_gram(
    *, candidate: str, method: str, target: str, regime: str,
    x0: np.ndarray, x1: np.ndarray, blocks0: Sequence[np.ndarray],
    blocks1: Sequence[np.ndarray], indices0: np.ndarray, indices1: np.ndarray,
    remainder0: np.ndarray, remainder1: np.ndarray, gram: np.ndarray,
    n_perm: int, exact: bool, max_exact_permutations: int,
    seed: Optional[int], diagnostics: dict,
) -> dict:
    K0, K1 = len(blocks0), len(blocks1)
    group0 = np.zeros(K0 + K1, dtype=bool)
    group0[:K0] = True
    observed = _mmd2_from_gram(gram, group0)
    masks, exact_used = _label_masks(
        K0 + K1, K0, n_perm=n_perm, exact=exact,
        max_exact_permutations=max_exact_permutations, seed=seed,
    )
    # All features are already represented in gram.  This loop performs only
    # label reassignment and MMD arithmetic, never point or PH recomputation.
    null = np.asarray([_mmd2_from_gram(gram, mask) for mask in masks], dtype=float)
    diagnostics = dict(diagnostics)
    diagnostics.update({
        "sampling_unit": "frozen disjoint m-point block",
        "point_sampling_unit": "individual iid point",
        "effective_sample_size": {
            "K0": int(K0), "K1": int(K1), "total": int(K0 + K1),
        },
        "unused_point_counts": {
            "arm0": int(len(remainder0)), "arm1": int(len(remainder1)),
        },
        "overlapping_blocks_used": False,
        "partition_frozen_for_call": True,
        "persistent_homology_recomputed_in_permutation_loop": False,
        "raw_features_recomputed_in_permutation_loop": False,
        "permutation_group": f"all block-label splits with K0={K0}, K1={K1}",
    })
    return {
        "candidate": candidate,
        "method": method,
        "regime": regime,
        "inferential_target": target,
        "statistic": float(observed),
        "pvalue": _permutation_pvalue(observed, null, exact_used),
        "posterior_quantity": None,
        "null_statistics": null,
        "n_permutations": int(len(null)),
        "exact_enumeration": bool(exact_used),
        "m": int(len(blocks0[0])),
        "sampling_unit": diagnostics["sampling_unit"],
        "kernel": diagnostics.get("kernel", ""),
        "bandwidth": diagnostics.get("bandwidth", diagnostics.get("bag_kernel_bandwidth")),
        "K0": int(K0),
        "K1": int(K1),
        "K_a": [int(K0), int(K1)],
        "n0": int(len(x0)),
        "n1": int(len(x1)),
        "remainder0": int(len(remainder0)),
        "remainder1": int(len(remainder1)),
        "unused_point_counts": [int(len(remainder0)), int(len(remainder1))],
        "block_indices0": indices0,
        "block_indices1": indices1,
        "diagnostics": diagnostics,
    }


def _validate_block_method_controls(partition_is_data_independent: bool,
                                    feature_tuning_is_label_independent: bool):
    if not partition_is_data_independent:
        raise ValueError(
            "block methods require a data-independent partition fixed without "
            "point coordinates"
        )
    if not feature_tuning_is_label_independent:
        raise ValueError(
            "label-dependent feature tuning is unsupported; freeze kernels and "
            "bandwidths before seeing treatment labels"
        )


def raw_block_mmd(
    cloud0, cloud1, *, regime: str = REGIME_I, m: int = LOCKED_M,
    partition_seed: Optional[int] = 0, partition0=None, partition1=None,
    point_kernel: str = DEFAULT_RAW_POINT_KERNEL,
    point_kernel_bandwidth: float = DEFAULT_RAW_POINT_BANDWIDTH,
    bag_kernel_bandwidth: float = DEFAULT_RAW_BAG_BANDWIDTH,
    raw_kernel: str = "gaussian_mean_embedding", n_perm: int = 999,
    exact: bool = False, max_exact_permutations: int = 100_000,
    seed: Optional[int] = 0, cache_dir: Optional[str] = None,
    partition_is_data_independent: bool = True,
    feature_tuning_is_label_independent: bool = True,
) -> dict:
    """Raw fixed-block MMD with a characteristic unordered-bag kernel.

    The default kernel is Gaussian on the point-kernel mean embedding of each
    bag.  ``raw_kernel='mean_pairwise'`` is available as a deliberately
    weaker diagnostic and is labelled non-characteristic for unrestricted bag
    laws in the returned diagnostics.
    """
    _require_regime("RawBlockMMD", regime)
    _validate_block_method_controls(
        partition_is_data_independent, feature_tuning_is_label_independent)
    x0, x1 = _validate_cloud_pair(cloud0, cloud1)
    if int(m) < 1:
        raise ValueError("m must be >= 1")
    m = int(m)
    point_kernel, raw_kernel = _validate_raw_kernel_options(
        point_kernel, raw_kernel, point_kernel_bandwidth, bag_kernel_bandwidth)

    def run():
        blocks0, indices0, remainder0 = _partition_or_draw(
            x0, m, partition_seed, partition0, "partition0")
        blocks1, indices1, remainder1 = _partition_or_draw(
            x1, m, None if partition_seed is None else int(partition_seed) + 1,
            partition1, "partition1")
        features = _raw_block_features(
            list(blocks0) + list(blocks1), point_kernel=point_kernel,
            point_bandwidth=point_kernel_bandwidth)
        gram = _raw_block_gram(
            features, point_kernel=point_kernel,
            point_bandwidth=point_kernel_bandwidth,
            bag_bandwidth=bag_kernel_bandwidth, raw_kernel=raw_kernel)
        return _block_result_from_gram(
            candidate="RawBlockMMD", method="raw_disjoint_block_mmd",
            target=_block_target(m, raw_kernel=raw_kernel), regime=regime, x0=x0, x1=x1,
            blocks0=blocks0, blocks1=blocks1, indices0=indices0,
            indices1=indices1, remainder0=remainder0, remainder1=remainder1,
            gram=gram, n_perm=n_perm, exact=exact,
            max_exact_permutations=max_exact_permutations, seed=seed,
            diagnostics={
                "exchangeability_basis": "iid point-law equality implies iid block exchangeability",
                "kernel": f"{raw_kernel} on unordered point bags",
                "point_kernel": point_kernel,
                "point_kernel_bandwidth": float(point_kernel_bandwidth),
                "bag_kernel_bandwidth": float(bag_kernel_bandwidth),
                "bandwidth_fixed_before_labels": True,
                "raw_features_cached": True,
                "persistent_homology_calls": 0,
                "kernel_characteristicness": (
                    "characteristic on fixed-size unordered bags via a Gaussian "
                    "kernel on the characteristic point mean embedding"
                    if raw_kernel == "gaussian_mean_embedding" else
                    "not characteristic for arbitrary bag laws; sensitivity only"
                ),
                "point_law_identification": (
                    "proved on the stated iid product-block model"
                    if raw_kernel == "gaussian_mean_embedding" else
                    "not claimed for unrestricted bag laws"
                ),
                "cache_dir": cache_dir,
            },
        )

    return _measure(run)


def hybrid_block_mmd(
    cloud0, cloud1, *, regime: str = REGIME_I, m: int = LOCKED_M,
    alpha: float = 0.50, partition_seed: Optional[int] = 0,
    partition0=None, partition1=None,
    point_kernel: str = DEFAULT_RAW_POINT_KERNEL,
    point_kernel_bandwidth: float = DEFAULT_RAW_POINT_BANDWIDTH,
    bag_kernel_bandwidth: float = DEFAULT_RAW_BAG_BANDWIDTH,
    raw_kernel: str = "gaussian_mean_embedding",
    barcode_kernel_bandwidth: float = DEFAULT_KERNEL_BANDWIDTH,
    filtration: str = "vr", homology_dims: Sequence[int] = LOCKED_HOMOLOGY_DIMS,
    max_edge_length: Optional[float] = None, grid_size: int = DEFAULT_GRID_SIZE,
    dtm_k: int = DEFAULT_DTM_K, n_perm: int = 999, exact: bool = False,
    max_exact_permutations: int = 100_000, seed: Optional[int] = 0,
    cache_dir: Optional[str] = None, partition_is_data_independent: bool = True,
    feature_tuning_is_label_independent: bool = True,
) -> dict:
    """Raw-plus-persistence fixed-block MMD with a predeclared weight.

    The persistence diagrams and raw block features are computed once before
    permutation.  For ``alpha>0`` the declared target remains ``P0=P1``;
    ``alpha=0`` is explicitly returned as a barcode-law diagnostic.
    """
    _require_regime("HybridBlockMMD", regime)
    _validate_block_method_controls(
        partition_is_data_independent, feature_tuning_is_label_independent)
    if not 0.0 <= float(alpha) <= 1.0:
        raise ValueError("alpha must lie in [0, 1]")
    if float(barcode_kernel_bandwidth) <= 0:
        raise ValueError("barcode_kernel_bandwidth must be positive")
    x0, x1 = _validate_cloud_pair(cloud0, cloud1)
    if int(m) < 1:
        raise ValueError("m must be >= 1")
    m = int(m)
    dims = _validate_ph_options(filtration, homology_dims)
    point_kernel, raw_kernel = _validate_raw_kernel_options(
        point_kernel, raw_kernel, point_kernel_bandwidth, bag_kernel_bandwidth)

    def run():
        blocks0, indices0, remainder0 = _partition_or_draw(
            x0, m, partition_seed, partition0, "partition0")
        blocks1, indices1, remainder1 = _partition_or_draw(
            x1, m, None if partition_seed is None else int(partition_seed) + 1,
            partition1, "partition1")
        blocks = list(blocks0) + list(blocks1)
        features = _raw_block_features(
            blocks, point_kernel=point_kernel,
            point_bandwidth=point_kernel_bandwidth)
        raw_gram = _raw_block_gram(
            features, point_kernel=point_kernel,
            point_bandwidth=point_kernel_bandwidth,
            bag_bandwidth=bag_kernel_bandwidth, raw_kernel=raw_kernel)
        if alpha < 1.0:
            diagrams = [
                _ph(block, filtration=filtration, homology_dims=dims,
                    max_edge_length=max_edge_length, grid_size=grid_size,
                    dtm_k=dtm_k, cache_dir=cache_dir)
                for block in blocks
            ]
            barcode_gram = _diagram_gram(diagrams, barcode_kernel_bandwidth)
        else:
            barcode_gram = np.zeros_like(raw_gram)
        gram = float(alpha) * raw_gram + (1.0 - float(alpha)) * barcode_gram
        return _block_result_from_gram(
            candidate="HybridBlockMMD", method="hybrid_raw_barcode_block_mmd",
            target=_block_target(
                m, barcode=float(alpha) == 0.0,
                raw_kernel=raw_kernel if float(alpha) > 0 else None,
            ), regime=regime,
            x0=x0, x1=x1, blocks0=blocks0, blocks1=blocks1,
            indices0=indices0, indices1=indices1, remainder0=remainder0,
            remainder1=remainder1, gram=gram, n_perm=n_perm, exact=exact,
            max_exact_permutations=max_exact_permutations, seed=seed,
            diagnostics={
                "exchangeability_basis": "iid point-law equality implies iid block exchangeability",
                "kernel": "alpha*K_raw + (1-alpha)*K_barcode",
                "alpha": float(alpha),
                "alpha_fixed_before_labels": True,
                "raw_kernel": raw_kernel,
                "point_kernel": point_kernel,
                "point_kernel_bandwidth": float(point_kernel_bandwidth),
                "bag_kernel_bandwidth": float(bag_kernel_bandwidth),
                "barcode_kernel": "tensor-product degree-tagged persistence scale-space",
                "barcode_kernel_bandwidth": float(barcode_kernel_bandwidth),
                "raw_features_cached": True,
                "persistent_homology_calls": int(len(blocks)) if alpha < 1.0 else 0,
                "kernel_characteristicness": (
                    "characteristic for alpha>0 through the raw unordered-bag component"
                    if alpha > 0 else
                    "barcode characteristicness only on the locked bounded diagram class"
                ),
                "point_law_identification": (
                    "proved on the stated iid product-block model for alpha>0"
                    if alpha > 0 else "not claimed; barcode-law diagnostic"
                ),
                "cache_dir": cache_dir,
            },
        )

    return _measure(run)


def sc_a_blockwise_label_permutation(*args, **kwargs) -> dict:
    """SC-A-style pooled block-label permutation using the raw block Gram.

    Blocks are formed separately within the original arms and then pooled for
    label permutations. With the same partition and raw kernel this is
    algebraically identical to ``RawBlockMMD``. The distinction is retained
    so tournament reports can compare SC-A's pooled-label framing with the
    explicit block-kernel candidate without treating them as independent
    methods.
    """
    result = raw_block_mmd(*args, **kwargs)
    result["candidate"] = "SC-A-Block"
    result["method"] = "pooled_block_label_permutation_raw_mmd"
    result["diagnostics"] = dict(result["diagnostics"])
    result["diagnostics"].update({
        "pooled_label_framing": True,
        "block_construction": "separate within original arm, then pool labels",
        "equivalent_to": "RawBlockMMD with the same frozen partitions and kernel",
    })
    return result


def raw_block_repeated_partition_test(*args, **kwargs):
    """Refuse repeated or overlapping raw-block aggregation."""
    raise ValueError(
        "repeated-partition aggregation is not implemented for RawBlockMMD: "
        "use one frozen disjoint partition; overlapping or repeatedly reused "
        "points are not independent block observations"
    )


# ---------------------------------------------------------------------------
# SC-A: full point-law pooled label permutation


def sc_a_label_permutation(
    cloud0,
    cloud1,
    *,
    regime: str = REGIME_I,
    filtration: str = "vr",
    homology_dims: Sequence[int] = LOCKED_HOMOLOGY_DIMS,
    max_edge_length: Optional[float] = None,
    grid_size: int = DEFAULT_GRID_SIZE,
    dtm_k: int = DEFAULT_DTM_K,
    kernel_bandwidth: float = DEFAULT_KERNEL_BANDWIDTH,
    n_perm: int = 999,
    exact: bool = False,
    max_exact_permutations: int = 100_000,
    seed: Optional[int] = 0,
    cache_dir: Optional[str] = None,
) -> dict:
    """SC-A, the full point-law label-permutation baseline.

    The observed and every permuted split are passed through the PH extractor.
    Consequently this method is exact under point-level exchangeability for
    ``P0=P1`` but does not target the weaker fixed-``m`` barcode-law null.
    ``exact=True`` enumerates all ``choose(n0+n1, n0)`` splits and reports the
    unrandomized finite permutation p-value.
    """
    _require_regime("SC-A", regime)
    x0, x1 = _validate_cloud_pair(cloud0, cloud1)
    dims = _validate_ph_options(filtration, homology_dims)
    pooled = np.vstack([x0, x1])
    n0 = len(x0)

    def run():
        observed_mask = np.zeros(len(pooled), dtype=bool)
        observed_mask[:n0] = True

        def statistic(mask):
            d0 = _ph(pooled[mask], filtration=filtration, homology_dims=dims,
                     max_edge_length=max_edge_length, grid_size=grid_size,
                     dtm_k=dtm_k, cache_dir=cache_dir)
            d1 = _ph(pooled[~mask], filtration=filtration, homology_dims=dims,
                     max_edge_length=max_edge_length, grid_size=grid_size,
                     dtm_k=dtm_k, cache_dir=cache_dir)
            return _joint_discrepancy(d0, d1, kernel_bandwidth)

        observed = statistic(observed_mask)
        masks, exact_used = _label_masks(
            len(pooled), n0, n_perm=n_perm, exact=exact,
            max_exact_permutations=max_exact_permutations, seed=seed,
        )
        null = np.asarray([statistic(mask) for mask in masks], dtype=float)
        return {
            "candidate": "SC-A",
            "method": "pooled_point_label_permutation",
            "regime": regime,
            "inferential_target": "H0^law: P0 = P1",
            "statistic": float(observed),
            "pvalue": _permutation_pvalue(observed, null, exact_used),
            "posterior_quantity": None,
            "null_statistics": null,
            "n_permutations": int(len(null)),
            "exact_enumeration": bool(exact_used),
            "n0": int(len(x0)),
            "n1": int(len(x1)),
            "diagnostics": {
                "exchangeability_basis": "iid point-level exchangeability",
                "primary_phase5_target": "not H0,m^bar; this is the strongest-simple baseline",
                "filtration": filtration,
                "homology_dims": dims,
                "grid_size": int(grid_size),
                "dtm_k": int(dtm_k),
                "kernel": "tensor-product degree-tagged universal persistence scale-space",
                "kernel_bandwidth": float(kernel_bandwidth),
                "recomputed_diagrams_for_each_split": True,
                "pooled_split_count": int(math.comb(len(pooled), n0)),
                "cache_dir": cache_dir,
            },
        }

    return _measure(run)


# ---------------------------------------------------------------------------
# SC-B: fixed disjoint barcode blocks


def _validate_partition(indices, n: int, m: int, name: str) -> np.ndarray:
    out = np.asarray(indices, dtype=int)
    if out.ndim != 2 or out.shape[1] != m or out.shape[0] < 1:
        raise ValueError(f"{name} must have shape (K, m) with K >= 1")
    if np.any(out < 0) or np.any(out >= n):
        raise ValueError(f"{name} contains an out-of-range point index")
    if len(np.unique(out)) != out.size:
        raise ValueError(
            f"{name} contains repeated point indices; overlapping blocks are "
            "not allowed in the confirmatory SC-B path"
        )
    return out


def disjoint_partition(cloud, m: int = LOCKED_M, seed: Optional[int] = 0):
    """Return one frozen random disjoint partition and its unused remainder.

    The returned indices, rather than a collection of overlapping sampled
    subclouds, are the confirmatory replication record.  The remainder is
    intentionally discarded, so ``K=floor(n/m)`` is visible and honest.
    """
    points = _as_cloud(cloud, "cloud")
    if int(m) < 1:
        raise ValueError("m must be >= 1")
    m = int(m)
    rng = np.random.default_rng(seed)
    permutation = rng.permutation(len(points))
    K = len(points) // m
    indices = permutation[:K * m].reshape(K, m)
    remainder = permutation[K * m:]
    if K < 1:
        raise ValueError(f"cloud has n={len(points)} < m={m}; no barcode block exists")
    return points[indices], indices, remainder


def _partition_or_draw(cloud, m: int, seed: Optional[int], supplied, name: str):
    points = _as_cloud(cloud, name.replace("_", ""))
    if supplied is not None:
        indices = _validate_partition(supplied, len(points), m, name)
        used = np.unique(indices)
        remainder = np.setdiff1d(np.arange(len(points)), used, assume_unique=True)
        return points[indices], indices, remainder
    return disjoint_partition(points, m=m, seed=seed)


def sc_b_disjoint_mmd(
    cloud0,
    cloud1,
    *,
    regime: str = REGIME_I,
    m: int = LOCKED_M,
    partition_seed: Optional[int] = 0,
    partition0=None,
    partition1=None,
    filtration: str = "vr",
    homology_dims: Sequence[int] = LOCKED_HOMOLOGY_DIMS,
    max_edge_length: Optional[float] = None,
    grid_size: int = DEFAULT_GRID_SIZE,
    dtm_k: int = DEFAULT_DTM_K,
    kernel_bandwidth: float = DEFAULT_KERNEL_BANDWIDTH,
    n_perm: int = 999,
    exact: bool = False,
    max_exact_permutations: int = 100_000,
    seed: Optional[int] = 0,
    cache_dir: Optional[str] = None,
) -> dict:
    """SC-B, the fixed-disjoint-block barcode-law comparison.

    A single partition is frozen for the call.  Its blocks are independent
    barcode draws under Regime I, while any unused points are reported and
    discarded.  When a random partition is drawn, arm zero uses
    ``partition_seed`` and arm one uses ``partition_seed + 1``.  The MMD
    permutation loop consumes only the cached block diagrams, never
    overlapping subclouds and never new PH calculations.
    """
    _require_regime("SC-B", regime)
    x0, x1 = _validate_cloud_pair(cloud0, cloud1)
    if int(m) < 2:
        raise ValueError("m must be >= 2")
    m = int(m)
    dims = _validate_ph_options(filtration, homology_dims)

    def run():
        blocks0, indices0, remainder0 = _partition_or_draw(
            x0, m, partition_seed, partition0, "partition0")
        blocks1, indices1, remainder1 = _partition_or_draw(
            x1, m, None if partition_seed is None else int(partition_seed) + 1,
            partition1, "partition1")
        diagrams0 = [
            _ph(block, filtration=filtration, homology_dims=dims,
                max_edge_length=max_edge_length, grid_size=grid_size,
                dtm_k=dtm_k, cache_dir=cache_dir)
            for block in blocks0
        ]
        diagrams1 = [
            _ph(block, filtration=filtration, homology_dims=dims,
                max_edge_length=max_edge_length, grid_size=grid_size,
                dtm_k=dtm_k, cache_dir=cache_dir)
            for block in blocks1
        ]
        diagrams = diagrams0 + diagrams1
        K0, K1 = len(diagrams0), len(diagrams1)
        group0 = np.zeros(K0 + K1, dtype=bool)
        group0[:K0] = True
        gram = _diagram_gram(diagrams, kernel_bandwidth)
        observed = _mmd2_from_gram(gram, group0)
        masks, exact_used = _label_masks(
            K0 + K1, K0, n_perm=n_perm, exact=exact,
            max_exact_permutations=max_exact_permutations, seed=seed,
        )
        null = np.asarray([_mmd2_from_gram(gram, mask) for mask in masks], dtype=float)
        target = ("H0,25^bar: Phi^25_0:1(P0) = Phi^25_0:1(P1)"
                  if m == LOCKED_M else
                  f"H0,{m}^bar: Phi^{m}_0:1(P0) = Phi^{m}_0:1(P1)"
                  " (unlocked sensitivity size)")
        return {
            "candidate": "SC-B",
            "method": "disjoint_fixed_m_barcode_mmd",
            "regime": regime,
            "inferential_target": target,
            "statistic": float(observed),
            "pvalue": _permutation_pvalue(observed, null, exact_used),
            "posterior_quantity": None,
            "null_statistics": null,
            "n_permutations": int(len(null)),
            "exact_enumeration": bool(exact_used),
            "m": m,
            "K0": K0,
            "K1": K1,
            "K_a": [K0, K1],
            "n0": int(len(x0)),
            "n1": int(len(x1)),
            "remainder0": int(len(remainder0)),
            "remainder1": int(len(remainder1)),
            "block_indices0": indices0,
            "block_indices1": indices1,
            "diagnostics": {
                "barcode_replication_basis": "independent disjoint point blocks under iid sampling",
                "effective_sample_size": {"K0": K0, "K1": K1},
                "overlapping_blocks_used": False,
                "persistent_homology_calls": K0 + K1,
                "persistent_homology_recomputed_in_permutation_loop": False,
                "filtration": filtration,
                "homology_dims": dims,
                "grid_size": int(grid_size),
                "dtm_k": int(dtm_k),
                "kernel": "tensor-product degree-tagged universal persistence scale-space",
                "kernel_bandwidth": float(kernel_bandwidth),
                "partition_frozen_for_call": True,
                "partition_seed": partition_seed,
                "kernel_characteristicness": (
                    "tensor product of exponentiated persistence scale-space "
                    "kernels; characteristic claim applies to the locked VR "
                    "contract on bounded per-degree diagram classes"
                ),
                "cache_dir": cache_dir,
            },
        }

    return _measure(run)


def sc_b_production_test(
    cloud0,
    cloud1,
    *,
    regime: str = REGIME_I,
    partition_seed: Optional[int] = 0,
    partition0=None,
    partition1=None,
    n_perm: int = 999,
    exact: bool = False,
    max_exact_permutations: int = 100_000,
    seed: Optional[int] = 0,
    cache_dir: Optional[str] = None,
    partition_is_data_independent: bool = True,
) -> dict:
    """Run the frozen, target-matched SC-B production procedure.

    The production contract is deliberately narrower than the prototype:
    Regime I, Vietoris--Rips, degrees ``(0, 1)``, ``m=25``, bandwidth ``0.10``,
    and at least ``PRODUCTION_MIN_BLOCKS`` disjoint blocks per arm.  The
    supplied partition, when present, must have been fixed without looking at
    point coordinates.  This declaration is checked as a refusal mode but
    cannot certify a caller's external partition-construction history.

    Exactly one frozen partition is used.  Repeated partitions and any
    uncorrected aggregation are intentionally not part of the production
    API; see ``sc_b_repeated_partition_test`` and the Phase-5D note.
    """
    _require_regime("SC-B-production", regime)
    if not partition_is_data_independent:
        raise ValueError(
            "SC-B production requires a data-independent partition fixed "
            "without point coordinates; data-dependent partition selection is "
            "unsupported"
        )
    x0, x1 = _validate_cloud_pair(cloud0, cloud1)
    if len(x0) // LOCKED_M < PRODUCTION_MIN_BLOCKS:
        raise ValueError(
            f"cloud0 has only {len(x0) // LOCKED_M} disjoint m={LOCKED_M} blocks; "
            f"production SC-B requires at least {PRODUCTION_MIN_BLOCKS}"
        )
    if len(x1) // LOCKED_M < PRODUCTION_MIN_BLOCKS:
        raise ValueError(
            f"cloud1 has only {len(x1) // LOCKED_M} disjoint m={LOCKED_M} blocks; "
            f"production SC-B requires at least {PRODUCTION_MIN_BLOCKS}"
        )
    if partition0 is not None and int(np.asarray(partition0).shape[0]) < PRODUCTION_MIN_BLOCKS:
        raise ValueError(
            f"supplied partition0 has {int(np.asarray(partition0).shape[0])} blocks; "
            f"production SC-B requires at least {PRODUCTION_MIN_BLOCKS} disjoint "
            f"blocks per arm"
        )
    if partition1 is not None and int(np.asarray(partition1).shape[0]) < PRODUCTION_MIN_BLOCKS:
        raise ValueError(
            f"supplied partition1 has {int(np.asarray(partition1).shape[0])} blocks; "
            f"production SC-B requires at least {PRODUCTION_MIN_BLOCKS} disjoint "
            f"blocks per arm"
        )

    result = sc_b_disjoint_mmd(
        x0,
        x1,
        regime=regime,
        m=LOCKED_M,
        partition_seed=partition_seed,
        partition0=partition0,
        partition1=partition1,
        filtration="ripser",
        homology_dims=LOCKED_HOMOLOGY_DIMS,
        kernel_bandwidth=DEFAULT_KERNEL_BANDWIDTH,
        n_perm=n_perm,
        exact=exact,
        max_exact_permutations=max_exact_permutations,
        seed=seed,
        cache_dir=cache_dir,
    )
    result["production_api"] = PRODUCTION_API_VERSION
    result["diagnostics"].update({
        "production_api": PRODUCTION_API_VERSION,
        "target_lock": "H0,25^bar joint VR barcode law in degrees 0:1",
        "sampling_unit": "point",
        "partition_data_independent_declared": True,
        "minimum_blocks_per_arm": PRODUCTION_MIN_BLOCKS,
        "repeated_partition_aggregation": "refused",
        "unsupported_regimes": [
            "stationary_mixing_process",
            "fixed_cloud",
            "explicit_generative_model",
        ],
        "kernel_characteristicness": (
            "tensor product of exponentiated persistence scale-space kernels; "
            "characteristic claim applies to the locked VR contract on bounded "
            "per-degree diagram classes"
        ),
    })
    return result


def sc_b_repeated_partition_test(*args, **kwargs):
    """Refuse unsupported repeated-partition aggregation explicitly.

    A single fixed partition is the locked confirmatory procedure.  Reusing
    points across partitions creates dependent barcode summaries, so averaging
    statistics, pooling permutation draws, Fisher-combining p-values, or
    taking an uncorrected minimum p-value has no validity claim here.
    """
    raise ValueError(
        "repeated-partition aggregation is not implemented for production "
        "SC-B: use one frozen disjoint partition; overlapping or repeatedly "
        "reused points are not independent barcode replicates"
    )


# ---------------------------------------------------------------------------
# SC-C: finite persistent-Betti vector and smoothed point bootstrap


def persistent_betti_vector(
    diagrams: Sequence[np.ndarray],
    grid: Sequence[float] = DEFAULT_GRID,
    *,
    normalize_by: Optional[float] = None,
) -> np.ndarray:
    """Return a frozen finite vector of persistent Betti numbers.

    The coordinate indexed by ``(degree, r, s)`` is
    ``#{(birth, death): birth <= r and death > s}`` for grid values ``r <= s``.
    Coordinates with ``r > s`` are set to zero.  Passing an explicit grid is
    recommended because deriving it separately from the two clouds changes
    the estimand.
    """
    values = np.asarray(grid, dtype=float)
    if values.ndim != 1 or len(values) < 2 or not np.isfinite(values).all():
        raise ValueError("grid must be a finite one-dimensional array with at least two values")
    if np.any(np.diff(values) < 0):
        raise ValueError("grid must be sorted in non-decreasing order")
    out = []
    for dgm in diagrams:
        dgm = np.asarray(dgm, dtype=float).reshape(-1, 2)
        if len(dgm):
            finite = dgm[np.isfinite(dgm).all(axis=1)]
            matrix = ((finite[:, 1, None, None] > values[None, None, :])
                      & (finite[:, 0, None, None] <= values[None, :, None]))
            matrix = matrix.sum(axis=0, dtype=float)
        else:
            matrix = np.zeros((len(values), len(values)), dtype=float)
        matrix[np.triu(np.ones_like(matrix, dtype=bool), k=0) == 0] = 0.0
        out.append(matrix.ravel())
    if not out:
        raise ValueError("at least one homology degree is required")
    vector = np.concatenate(out)
    if normalize_by is not None:
        if normalize_by <= 0:
            raise ValueError("normalize_by must be positive")
        vector = vector / float(normalize_by)
    return vector


def _bootstrap_cloud(base: np.ndarray, size: int, rng: np.random.Generator,
                     smoothing: bool, bandwidth: float) -> np.ndarray:
    indices = rng.integers(0, len(base), size=size)
    out = base[indices].copy()
    if smoothing:
        out += rng.normal(0.0, bandwidth, size=out.shape)
    return out


def _validate_grid(grid) -> np.ndarray:
    values = np.asarray(grid, dtype=float)
    # persistent_betti_vector performs the detailed validation; this helper
    # only ensures the returned metadata is an independent immutable snapshot.
    persistent_betti_vector([np.zeros((0, 2))], values)
    return values.copy()


def sc_c_finite_vector(
    cloud0,
    cloud1,
    *,
    regime: str = REGIME_I,
    filtration: str = "vr",
    homology_dims: Sequence[int] = LOCKED_HOMOLOGY_DIMS,
    max_edge_length: Optional[float] = None,
    grid_size: int = DEFAULT_GRID_SIZE,
    dtm_k: int = DEFAULT_DTM_K,
    grid: Sequence[float] = DEFAULT_GRID,
    bootstrap_bandwidth: float = 0.05,
    n_draws: int = 399,
    seed: Optional[int] = 0,
    smoothing: bool = True,
    cache_dir: Optional[str] = None,
) -> dict:
    """SC-C finite-vector bootstrap prototype.

    The null bootstrap draws both arms from the pooled empirical point law,
    with optional Gaussian kernel jitter.  The statistic is the scaled
    Euclidean contrast of normalized persistent-Betti vectors.  This is a
    finite-vector equality prototype under the Euclidean stabilizing-statistic
    conditions of Roycraft, Krebs, and Polonik; it is not a test of the full
    fixed-25 barcode law.

    ``smoothing=False`` is retained specifically as the naïve-bootstrap
    negative control.  It should not be promoted to a production method for
    persistent Betti statistics merely because its output is convenient.
    """
    _require_regime("SC-C", regime)
    x0, x1 = _validate_cloud_pair(cloud0, cloud1)
    dims = _validate_ph_options(filtration, homology_dims)
    grid_values = _validate_grid(grid)
    if bootstrap_bandwidth < 0:
        raise ValueError("bootstrap_bandwidth must be non-negative")
    if n_draws < 1:
        raise ValueError("n_draws must be >= 1")

    def run():
        d0 = _ph(x0, filtration=filtration, homology_dims=dims,
                 max_edge_length=max_edge_length, grid_size=grid_size,
                 dtm_k=dtm_k, cache_dir=cache_dir)
        d1 = _ph(x1, filtration=filtration, homology_dims=dims,
                 max_edge_length=max_edge_length, grid_size=grid_size,
                 dtm_k=dtm_k, cache_dir=cache_dir)
        v0 = persistent_betti_vector(d0, grid_values, normalize_by=len(x0))
        v1 = persistent_betti_vector(d1, grid_values, normalize_by=len(x1))
        scale = math.sqrt(len(x0) * len(x1) / (len(x0) + len(x1)))
        observed = float(scale * np.linalg.norm(v0 - v1))

        rng = np.random.default_rng(seed)
        pooled = np.vstack([x0, x1])
        null = np.empty(int(n_draws), dtype=float)
        for b in range(int(n_draws)):
            boot0 = _bootstrap_cloud(pooled, len(x0), rng, smoothing, bootstrap_bandwidth)
            boot1 = _bootstrap_cloud(pooled, len(x1), rng, smoothing, bootstrap_bandwidth)
            bd0 = _ph(boot0, filtration=filtration, homology_dims=dims,
                      max_edge_length=max_edge_length, grid_size=grid_size,
                      dtm_k=dtm_k, cache_dir=cache_dir)
            bd1 = _ph(boot1, filtration=filtration, homology_dims=dims,
                      max_edge_length=max_edge_length, grid_size=grid_size,
                      dtm_k=dtm_k, cache_dir=cache_dir)
            bv0 = persistent_betti_vector(bd0, grid_values, normalize_by=len(x0))
            bv1 = persistent_betti_vector(bd1, grid_values, normalize_by=len(x1))
            null[b] = scale * np.linalg.norm(bv0 - bv1)

        return {
            "candidate": "SC-C",
            "method": "smoothed_finite_persistent_betti_vector" if smoothing
                      else "naive_finite_persistent_betti_vector",
            "regime": regime,
            "inferential_target": "H0^finite-vector: equality of the frozen normalized persistent-Betti mean vector",
            "statistic": observed,
            "pvalue": p_value(observed, null, alternative="greater"),
            "posterior_quantity": None,
            "null_statistics": null,
            "n_draws": int(n_draws),
            "bootstrap": "smoothed" if smoothing else "naive_negative_control",
            "observed_vector0": v0,
            "observed_vector1": v1,
            "grid": grid_values,
            "diagnostics": {
                "point_level_replication": True,
                "bootstrap_null": "both arms resampled from pooled empirical law",
                "smoothing_bandwidth": float(bootstrap_bandwidth),
                "smoothed_bootstrap_source": "Roycraft, Krebs & Polonik (2023), DOI 10.1214/23-AOS2277",
                "persistent_homology_recomputed_in_bootstrap_loop": True,
                "filtration": filtration,
                "homology_dims": dims,
                "grid_size": int(grid_size),
                "dtm_k": int(dtm_k),
                "normalization": "persistent-Betti counts divided by arm cloud size",
                "primary_phase5_target": "not H0,25^bar; finite-vector prototype only",
                "cache_dir": cache_dir,
            },
        }

    return _measure(run)


def sc_c_naive_bootstrap(*args, **kwargs) -> dict:
    """SC-C's declared naïve-bootstrap negative control."""
    kwargs = dict(kwargs)
    kwargs["smoothing"] = False
    return sc_c_finite_vector(*args, **kwargs)


# ---------------------------------------------------------------------------
# Common interface and source-setting record


def run_single_cloud_test(candidate: str, cloud0, cloud1, *, regime: str = REGIME_I,
                          **kwargs) -> dict:
    """Dispatch one Phase 5B candidate through the common result interface."""
    key = _normalise_candidate(candidate)
    # Route before binding candidate-specific keyword arguments.  This keeps
    # an incompatible observation model a deterministic scientific error,
    # even when the caller supplied options belonging to another candidate.
    _require_regime(key.upper(), regime)
    if key == "sc-a":
        return sc_a_label_permutation(cloud0, cloud1, regime=regime, **kwargs)
    if key == "sc-b":
        return sc_b_disjoint_mmd(cloud0, cloud1, regime=regime, **kwargs)
    if key == "sc-b-production":
        return sc_b_production_test(cloud0, cloud1, regime=regime, **kwargs)
    if key == "raw-block-mmd":
        return raw_block_mmd(cloud0, cloud1, regime=regime, **kwargs)
    if key == "hybrid-block-mmd":
        return hybrid_block_mmd(cloud0, cloud1, regime=regime, **kwargs)
    if key == "sc-a-block":
        return sc_a_blockwise_label_permutation(
            cloud0, cloud1, regime=regime, **kwargs)
    return sc_c_finite_vector(cloud0, cloud1, regime=regime, **kwargs)


def roycraft_reference_setting() -> dict:
    """Return the pre-registered SC-C source-aligned pilot setting.

    The source studies persistent Betti numbers of binomial/Poisson point
    sets in Euclidean space and compares ordinary with smoothed bootstrap
    inference.  This record deliberately states what is reproduced and what
    is not: the P1 prototype uses its own VR-radius implementation and a
    two-cloud contrast, so it is not a claim to reproduce the source's full
    confidence-table fleet.
    """
    return {
        "source": "Roycraft, Krebs & Polonik (2023), Annals of Statistics 51, 1484-1509",
        "doi": "10.1214/23-AOS2277",
        "point_model": "binomial point samples from a Euclidean density",
        "statistic": "finite vector of persistent Betti numbers",
        "filtration": "Vietoris-Rips radius filtration",
        "comparison": "Gaussian smoothed bootstrap versus ordinary bootstrap",
        "p1_adaptation": "two independent clouds and a pooled-null finite-vector contrast",
        "not_claimed": "full reproduction of the source confidence-coverage tables",
    }


In [ ]:
%%writefile experiments/phase5_single_cloud_tournament.py
"""Phase 5C selection fleet for the exactly-two-cloud Regime-I problem.

The fleet is deliberately simulation-first.  It has four responsibilities:

1. run a cheap, deterministic pilot before any gate is read;
2. run independent replication shards for the pre-registered DGP cells;
3. aggregate shards without silently filling missing or conflicting records;
4. write the gate table, memo, and a figure showing validity and
   pseudo-replication failure.

The signed Phase 5A target is the fixed-size barcode-law null at ``m=25``.
SC-B at ``m=25`` is therefore the only target-matched production candidate.
SC-A is retained as the strongest-simple full point-law baseline and SC-C as
the finite persistent-Betti sensitivity.  SC-A is expensive because every
pooled label split requires a new PH calculation.  To make the comparison
reproducible on the available hardware, SC-A uses a predeclared 250-point
projection per arm whenever a cloud is larger than 250 points.  This is an
explicit effective-sample-size limitation, not a hidden claim of matched-n
power.  SC-B and SC-C use the full supplied clouds.

The main fleet is scoped to Regime I.  Process DGP constructors are included
for an optional dependence diagnostic, but they are not promoted to a size
gate because Phase 5A did not select Regime II and SC-D is dormant.

Examples
--------
Pilot (100 replications, one n cell):

    python experiments/phase5_single_cloud_tournament.py --mode pilot

One gating shard, suitable for a local process or a self-contained Colab
notebook:

    python experiments/phase5_single_cloud_tournament.py --mode shard \
        --cell iid_null_n250_m25 --rep-start 0 --replications 25

Aggregate all downloaded shards and write the requested deliverables:

    python experiments/phase5_single_cloud_tournament.py --mode aggregate

The default calibration counts are deliberately fixed in this file.  They
may be changed only by making a new design record, not by tuning them after
seeing rejection rates.
"""
from __future__ import annotations

import argparse
import glob
import hashlib
import json
import math
import os
import time
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass
from typing import Iterable, Sequence

import numpy as np
import pandas as pd

from tda2s.ph import compute_diagrams
from tda2s.resample import p_value
from tda2s.tests.single_cloud import (
    LOCKED_M,
    REGIME_I,
    _diagram_gram,
    _label_masks,
    _mmd2_from_gram,
    _permutation_pvalue,
    _ph,
    sc_a_label_permutation,
    sc_b_disjoint_mmd,
    sc_c_finite_vector,
)

ALPHA = 0.05
SIZE_BAND = (0.03, 0.08)
MC_CONFIDENCE = 0.95
N_GRID = (250, 500, 1000)
# m=50 supplies K={5,10,20}; m=25 is the signed primary target and supplies
# K={10,20,40}.  Both are frozen before any result is inspected.
M_GRID = (25, 50)
PRIMARY_M = LOCKED_M
FILTRATION = "ripser"  # Vietoris--Rips backend, with the shared radius scale
HOMOLOGY_DIMS = (0, 1)
KERNEL_BANDWIDTH = 0.10
BETTI_GRID = np.linspace(0.0, 0.60, 9)
BOOTSTRAP_BANDWIDTH = 0.05
MAX_SC_A_POINT_N = 250
GATE_PERMUTATIONS = 39
GATE_BOOTSTRAP_DRAWS = 19
GATE_REPLICATIONS = 500
PILOT_PERMUTATIONS = 19
PILOT_BOOTSTRAP_DRAWS = 9
DEFAULT_SHARD_REPLICATIONS = 25
MAX_WORKERS = 16
SEED_ROOT = 20260819

RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "results")
SHARD_DIR = os.path.join(RESULTS_DIR, "phase5c_shards")
CACHE_DIR = os.path.join(RESULTS_DIR, "phase5c_ph_cache")
FINAL_SUMMARY = os.path.join(RESULTS_DIR, "phase5_single_cloud_tournament.parquet")
FINAL_REPLICATIONS = os.path.join(RESULTS_DIR, "phase5_single_cloud_tournament_replications.parquet")
FINAL_FIGURE = os.path.join(RESULTS_DIR, "phase5_single_cloud_tournament.png")
FINAL_MEMO = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "docs", "phase5_gate_memo.md")


@dataclass(frozen=True)
class Cell:
    family: str
    n0: int
    n1: int
    m: int
    role: str
    description: str

    @property
    def cell_id(self) -> str:
        return f"{self.family}_n{self.n0}_n1{self.n1}_m{self.m}"


CORE_FAMILIES = (
    "iid_null",
    "weak_barcode_null",
    "same_support_density",
    "topology_alt",
)
ROBUSTNESS_FAMILIES = (
    "robust_contamination",
    "robust_unequal_cardinality",
    "robust_anisotropic_noise",
    "robust_boundary_truncation",
)
DEPENDENCE_FAMILIES = (
    "process_poisson",
    "process_inhomogeneous_poisson",
    "process_cox_clustered",
    "process_hard_core",
)

FAMILY_ROLE = {
    "iid_null": "gating_null",
    "weak_barcode_null": "gating_null",
    "same_support_density": "target_mismatch",
    "topology_alt": "power",
    "robust_contamination": "robustness_diagnostic",
    "robust_unequal_cardinality": "robustness_diagnostic",
    "robust_anisotropic_noise": "robustness_diagnostic",
    "robust_boundary_truncation": "robustness_diagnostic",
    "process_poisson": "dormant_dependence",
    "process_inhomogeneous_poisson": "dormant_dependence",
    "process_cox_clustered": "dormant_dependence",
    "process_hard_core": "dormant_dependence",
}

FAMILY_DESCRIPTION = {
    "iid_null": "identical Uniform([0,1]^2) metric-measure laws",
    "weak_barcode_null": "translated point law with exactly equal metric barcode law",
    "same_support_density": "same square support with different continuous densities",
    "topology_alt": "filled disk versus noisy circle with matched expected moments",
    "robust_contamination": "common five-percent remote contamination",
    "robust_unequal_cardinality": "same law with unequal cloud cardinalities",
    "robust_anisotropic_noise": "common anisotropic affine deformation and noise",
    "robust_boundary_truncation": "common rectangular boundary truncation",
    "process_poisson": "homogeneous Poisson point process diagnostic",
    "process_inhomogeneous_poisson": "inhomogeneous Poisson point process diagnostic",
    "process_cox_clustered": "clustered Cox-style point process diagnostic",
    "process_hard_core": "hard-core point process diagnostic",
}


def _seed(*parts: object) -> int:
    """Stable 32-bit seed independent of Python's randomized hash."""
    payload = repr((SEED_ROOT,) + parts).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "little")


def design_record() -> dict:
    return {
        "phase": "5C",
        "regime": REGIME_I,
        "alpha": ALPHA,
        "size_band": list(SIZE_BAND),
        "confidence": MC_CONFIDENCE,
        "n_grid": list(N_GRID),
        "m_grid": list(M_GRID),
        "primary_m": PRIMARY_M,
        "filtration": FILTRATION,
        "homology_dims": list(HOMOLOGY_DIMS),
        "kernel_bandwidth": KERNEL_BANDWIDTH,
        "betti_grid": BETTI_GRID.tolist(),
        "bootstrap_bandwidth": BOOTSTRAP_BANDWIDTH,
        "gate_permutations": GATE_PERMUTATIONS,
        "gate_bootstrap_draws": GATE_BOOTSTRAP_DRAWS,
        "pilot_permutations": PILOT_PERMUTATIONS,
        "pilot_bootstrap_draws": PILOT_BOOTSTRAP_DRAWS,
        "sc_a_projection_n": MAX_SC_A_POINT_N,
        "target": "H0,25^bar: Phi^25_0:1(P0) = Phi^25_0:1(P1)",
        "sc_a_target": "H0^law: P0 = P1",
        "sc_c_target": "H0^finite-vector: equality of frozen normalized persistent-Betti means",
        "overlap_fractions": [0.0, 0.25, 0.5, 0.75, 0.9],
    }


DESIGN_HASH = hashlib.sha256(
    json.dumps(design_record(), sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()[:16]


def required_k_counts() -> dict[int, list[int]]:
    return {m: [n // m for n in N_GRID] for m in M_GRID}


def make_cells(*, include_robustness: bool = True, include_dependence: bool = False) -> list[Cell]:
    families = list(CORE_FAMILIES)
    if include_robustness:
        families.extend(ROBUSTNESS_FAMILIES)
    if include_dependence:
        families.extend(DEPENDENCE_FAMILIES)
    cells = []
    for family in families:
        for n in N_GRID:
            n1 = int(math.ceil(n * 1.25)) if family == "robust_unequal_cardinality" else n
            for m in M_GRID:
                cells.append(Cell(
                    family=family,
                    n0=n,
                    n1=n1,
                    m=m,
                    role=FAMILY_ROLE[family],
                    description=FAMILY_DESCRIPTION[family],
                ))
    return cells


def parse_cell_id(value: str) -> Cell:
    for cell in make_cells(include_robustness=True, include_dependence=True):
        if cell.cell_id == value:
            return cell
    valid = ", ".join(c.cell_id for c in make_cells(include_robustness=False))
    raise ValueError(f"unknown cell {value!r}; examples are {valid}")


def _uniform_square(n: int, rng: np.random.Generator) -> np.ndarray:
    return rng.uniform(0.0, 1.0, size=(int(n), 2))


def _filled_disk(n: int, rng: np.random.Generator, *, radius: float = 0.30) -> np.ndarray:
    theta = rng.uniform(0.0, 2.0 * np.pi, size=int(n))
    radial = radius * np.sqrt(rng.uniform(0.0, 1.0, size=int(n)))
    points = np.column_stack([radial * np.cos(theta), radial * np.sin(theta)])
    return points + np.array([0.5, 0.5])


def _noisy_circle(n: int, rng: np.random.Generator, *, radius: float = 0.30) -> np.ndarray:
    theta = rng.uniform(0.0, 2.0 * np.pi, size=int(n))
    points = np.column_stack([np.cos(theta), np.sin(theta)]) * radius
    points += rng.normal(0.0, 0.008, size=points.shape)
    return points + np.array([0.5, 0.5])


def _common_contamination(n: int, rng: np.random.Generator) -> np.ndarray:
    points = _uniform_square(n, rng)
    count = max(1, int(round(0.05 * n)))
    points[:count] = rng.uniform(2.0, 3.0, size=(count, 2))
    return points


def _anisotropic(n: int, rng: np.random.Generator) -> np.ndarray:
    points = _uniform_square(n, rng)
    points = points @ np.array([[1.8, 0.0], [0.0, 0.45]])
    points += rng.normal(0.0, 0.015, size=points.shape)
    return points


def _hard_core(n: int, rng: np.random.Generator, minimum: float = 0.025) -> np.ndarray:
    accepted: list[np.ndarray] = []
    max_attempts = max(1000, 40 * int(n))
    attempts = 0
    while len(accepted) < int(n) and attempts < max_attempts:
        candidate = rng.uniform(0.0, 1.0, size=2)
        attempts += 1
        if not accepted or min(np.linalg.norm(candidate - old) for old in accepted) >= minimum:
            accepted.append(candidate)
    if len(accepted) < int(n):
        # The fallback is deterministic and keeps the diagnostic runnable at
        # high n.  It is not used for a Regime-I gate.
        extra = _uniform_square(int(n) - len(accepted), rng)
        accepted.extend(extra)
    return np.asarray(accepted, dtype=float)


def _poisson_cloud(n_expected: int, rng: np.random.Generator) -> np.ndarray:
    n = max(2, int(rng.poisson(n_expected)))
    return _uniform_square(n, rng)


def _inhomogeneous_poisson_cloud(n_expected: int, rng: np.random.Generator) -> np.ndarray:
    n = max(2, int(rng.poisson(n_expected)))
    # x has density 2x, while y remains uniform.  The support stays the unit
    # square, so this is an intensity change rather than a support change.
    return np.column_stack([np.sqrt(rng.uniform(size=n)), rng.uniform(size=n)])


def _cox_cloud(n_expected: int, rng: np.random.Generator) -> np.ndarray:
    n = max(2, int(rng.poisson(n_expected)))
    parents = rng.uniform(0.0, 1.0, size=(8, 2))
    labels = rng.integers(0, len(parents), size=n)
    points = parents[labels] + rng.normal(0.0, 0.07, size=(n, 2))
    return np.mod(points, 1.0)


def make_cloud_pair(family: str, n0: int, n1: int, seed: int) -> tuple[np.ndarray, np.ndarray]:
    """Generate one deterministic two-cloud replication for a named DGP."""
    if family not in FAMILY_ROLE:
        raise ValueError(f"unknown family {family!r}")
    rng = np.random.default_rng(seed)

    if family == "iid_null":
        return _uniform_square(n0, rng), _uniform_square(n1, rng)
    if family == "weak_barcode_null":
        cloud0 = _uniform_square(n0, rng)
        cloud1 = _uniform_square(n1, rng) + np.array([2.0, -1.5])
        return cloud0, cloud1
    if family == "same_support_density":
        cloud0 = _uniform_square(n0, rng)
        # A beta mixture has the same closed support [0,1]^2 but a different
        # density.  The uniform component prevents a support convention from
        # driving the distinction.
        mask = rng.uniform(size=(n1, 1)) < 0.8
        beta = rng.beta(2.5, 2.5, size=(n1, 2))
        uniform = _uniform_square(n1, rng)
        cloud1 = np.where(mask, beta, uniform)
        return cloud0, cloud1
    if family == "topology_alt":
        return _filled_disk(n0, rng), _noisy_circle(n1, rng)
    if family == "robust_contamination":
        return _common_contamination(n0, rng), _common_contamination(n1, rng)
    if family == "robust_unequal_cardinality":
        return _uniform_square(n0, rng), _uniform_square(n1, rng)
    if family == "robust_anisotropic_noise":
        return _anisotropic(n0, rng), _anisotropic(n1, rng)
    if family == "robust_boundary_truncation":
        return _uniform_square(n0, rng) * np.array([0.7, 1.0]), _uniform_square(n1, rng) * np.array([0.7, 1.0])
    if family == "process_poisson":
        return _poisson_cloud(n0, rng), _poisson_cloud(n1, rng)
    if family == "process_inhomogeneous_poisson":
        return _inhomogeneous_poisson_cloud(n0, rng), _inhomogeneous_poisson_cloud(n1, rng)
    if family == "process_cox_clustered":
        return _cox_cloud(n0, rng), _cox_cloud(n1, rng)
    if family == "process_hard_core":
        return _hard_core(n0, rng), _hard_core(n1, rng)
    raise AssertionError(f"unhandled family {family!r}")


def _project_cloud(cloud: np.ndarray, size: int, seed: int) -> np.ndarray:
    if len(cloud) <= size:
        return np.asarray(cloud, dtype=float)
    rng = np.random.default_rng(seed)
    return np.asarray(cloud)[rng.choice(len(cloud), size=int(size), replace=False)]


def _common_ph_kwargs(cache_dir: str | None) -> dict:
    return {
        "filtration": FILTRATION,
        "homology_dims": HOMOLOGY_DIMS,
        "max_edge_length": None,
        "grid_size": 64,
        "dtm_k": 20,
        "kernel_bandwidth": KERNEL_BANDWIDTH,
        "cache_dir": cache_dir,
    }


def _record_from_result(cell: Cell, rep: int, candidate: str, result: dict,
                        *, m: int, null_role: str, method_variant: str = "") -> dict:
    diagnostics = result.get("diagnostics", {})
    effective = diagnostics.get("effective_sample_size", {})
    return {
        "design_hash": DESIGN_HASH,
        "record_type": "candidate",
        "status": "ok",
        "replication": int(rep),
        "cell_id": cell.cell_id,
        "family": cell.family,
        "family_role": null_role,
        "family_description": cell.description,
        "candidate": candidate,
        "method_variant": method_variant,
        "regime": REGIME_I,
        "n0": int(cell.n0),
        "n1": int(cell.n1),
        "m": int(m),
        "primary_target": bool(candidate == "SC-B" and m == PRIMARY_M),
        "target": result.get("inferential_target", ""),
        "statistic": float(result.get("statistic", np.nan)),
        "pvalue": float(result.get("pvalue", np.nan)),
        "reject": bool(float(result.get("pvalue", np.nan)) <= ALPHA),
        "alpha": ALPHA,
        "K0": int(result.get("K0", effective.get("K0", -1))),
        "K1": int(result.get("K1", effective.get("K1", -1))),
        "effective_point_n0": int(diagnostics.get("effective_point_n0", cell.n0)),
        "effective_point_n1": int(diagnostics.get("effective_point_n1", cell.n1)),
        "effective_barcode_n0": int(result.get("K0", effective.get("K0", -1))),
        "effective_barcode_n1": int(result.get("K1", effective.get("K1", -1))),
        "overlap_fraction": 0.0,
        "unique_points0": int(cell.n0),
        "unique_points1": int(cell.n1),
        "n_resamples": int(result.get("n_permutations", result.get("n_draws", -1))),
        "runtime_seconds": float(result.get("runtime_seconds", np.nan)),
        "peak_memory_bytes": int(result.get("peak_memory_bytes", -1)),
        "filtration": FILTRATION,
        "homology_dims": json.dumps(list(HOMOLOGY_DIMS)),
        "partition_frozen": bool(candidate == "SC-B"),
        "overlapping_blocks_used": False,
        "projection_used": bool(candidate == "SC-A" and min(cell.n0, cell.n1) > MAX_SC_A_POINT_N),
        "method_assumptions_ok": bool(cell.role != "dormant_dependence"),
        "null_role": null_role,
        "method_error": "",
    }


def _error_record(cell: Cell, rep: int, candidate: str, m: int, exc: Exception,
                  *, null_role: str, method_variant: str = "") -> dict:
    record = {
        "design_hash": DESIGN_HASH,
        "record_type": "candidate",
        "status": "failed",
        "replication": int(rep),
        "cell_id": cell.cell_id,
        "family": cell.family,
        "family_role": null_role,
        "family_description": cell.description,
        "candidate": candidate,
        "method_variant": method_variant,
        "regime": REGIME_I,
        "n0": int(cell.n0),
        "n1": int(cell.n1),
        "m": int(m),
        "primary_target": bool(candidate == "SC-B" and m == PRIMARY_M),
        "target": "",
        "statistic": np.nan,
        "pvalue": np.nan,
        "reject": False,
        "alpha": ALPHA,
        "K0": -1,
        "K1": -1,
        "effective_point_n0": -1,
        "effective_point_n1": -1,
        "effective_barcode_n0": -1,
        "effective_barcode_n1": -1,
        "overlap_fraction": 0.0,
        "unique_points0": -1,
        "unique_points1": -1,
        "n_resamples": -1,
        "runtime_seconds": np.nan,
        "peak_memory_bytes": -1,
        "filtration": FILTRATION,
        "homology_dims": json.dumps(list(HOMOLOGY_DIMS)),
        "partition_frozen": False,
        "overlapping_blocks_used": False,
        "projection_used": False,
        "method_assumptions_ok": False,
        "null_role": null_role,
        "method_error": f"{type(exc).__name__}: {exc}",
    }
    return record


def run_candidate(cell: Cell, rep: int, candidate: str, *, m: int,
                  n_permutations: int, n_bootstrap: int,
                  cache_dir: str | None) -> dict:
    """Run one frozen candidate and normalize its result schema."""
    cloud0, cloud1 = make_cloud_pair(cell.family, cell.n0, cell.n1, _seed("cloud", cell.cell_id, rep))
    method_seed = _seed("method", cell.cell_id, rep, candidate, m)
    try:
        if candidate == "SC-A":
            # Full SC-A at n=1000 is a poor use of the PH budget.  The
            # projection is fixed before the fleet and recorded in every row.
            x0 = _project_cloud(cloud0, MAX_SC_A_POINT_N, _seed("A0", cell.cell_id, rep))
            x1 = _project_cloud(cloud1, MAX_SC_A_POINT_N, _seed("A1", cell.cell_id, rep))
            result = sc_a_label_permutation(
                x0, x1, regime=REGIME_I, **_common_ph_kwargs(cache_dir),
                n_perm=n_permutations, exact=False, seed=method_seed,
            )
            result["diagnostics"]["effective_point_n0"] = len(x0)
            result["diagnostics"]["effective_point_n1"] = len(x1)
        elif candidate == "SC-B":
            result = sc_b_disjoint_mmd(
                cloud0, cloud1, regime=REGIME_I, m=m,
                partition_seed=_seed("partition", cell.cell_id, rep),
                n_perm=n_permutations, exact=False, seed=method_seed,
                **_common_ph_kwargs(cache_dir),
            )
        elif candidate == "SC-C":
            # The finite-vector candidate uses the full clouds.  Its bootstrap
            # clouds are fresh and are intentionally not cached as they do not
            # recur across replications.
            kwargs = _common_ph_kwargs(None)
            kwargs.pop("kernel_bandwidth")
            result = sc_c_finite_vector(
                cloud0, cloud1, regime=REGIME_I, grid=BETTI_GRID,
                bootstrap_bandwidth=BOOTSTRAP_BANDWIDTH,
                n_draws=n_bootstrap, smoothing=True, seed=method_seed,
                **kwargs,
            )
        else:
            raise ValueError(f"unknown candidate {candidate!r}")
        return _record_from_result(cell, rep, candidate, result, m=m,
                                   null_role=cell.role)
    except Exception as exc:  # retain failed runs for the gate audit
        return _error_record(cell, rep, candidate, m, exc, null_role=cell.role)


def _overlapping_blocks(cloud: np.ndarray, m: int, overlap_fraction: float,
                        seed: int, K: int) -> tuple[list[np.ndarray], np.ndarray, float]:
    if not 0.0 <= overlap_fraction < 1.0:
        raise ValueError("overlap_fraction must be in [0,1)")
    if len(cloud) < m:
        raise ValueError("cloud must contain at least m points")
    overlap = int(round(float(overlap_fraction) * m))
    step = max(1, m - overlap)
    needed = m + (int(K) - 1) * step
    if needed > len(cloud):
        raise ValueError("requested overlapping blocks exceed the cloud")
    permutation = np.random.default_rng(seed).permutation(len(cloud))
    indices = np.asarray([permutation[k * step:k * step + m] for k in range(int(K))], dtype=int)
    blocks = [np.asarray(cloud[idx], dtype=float) for idx in indices]
    pairwise = []
    for i in range(len(indices)):
        for j in range(i):
            pairwise.append(len(set(indices[i]).intersection(indices[j])) / float(m))
    mean_overlap = float(np.mean(pairwise)) if pairwise else 0.0
    return blocks, indices, mean_overlap


def overlapping_negative_control(cloud0: np.ndarray, cloud1: np.ndarray, *, m: int,
                                 overlap_fraction: float, seed: int,
                                 n_permutations: int, cache_dir: str | None) -> dict:
    """Naïve overlapping-subcloud MMD negative control.

    This intentionally violates SC-B's disjoint-block condition.  It is
    retained solely to expose pseudo-replication.  The PH diagrams are cached
    once per overlapping block, and the label permutation acts only on their
    Gram matrix.
    """
    K0, K1 = len(cloud0) // m, len(cloud1) // m
    blocks0, idx0, mean0 = _overlapping_blocks(cloud0, m, overlap_fraction, seed, K0)
    blocks1, idx1, mean1 = _overlapping_blocks(cloud1, m, overlap_fraction, seed + 1, K1)
    diagrams = [
        _ph(block, filtration=FILTRATION, homology_dims=HOMOLOGY_DIMS,
            max_edge_length=None, grid_size=64, dtm_k=20, cache_dir=cache_dir)
        for block in blocks0 + blocks1
    ]
    gram = _diagram_gram(diagrams, KERNEL_BANDWIDTH)
    group0 = np.zeros(K0 + K1, dtype=bool)
    group0[:K0] = True
    observed = _mmd2_from_gram(gram, group0)
    masks, exact_used = _label_masks(
        K0 + K1, K0, n_perm=n_permutations, exact=False,
        max_exact_permutations=100_000, seed=seed + 2,
    )
    null = np.asarray([_mmd2_from_gram(gram, mask) for mask in masks], dtype=float)
    return {
        "candidate": "SC-B-overlap-negative-control",
        "statistic": float(observed),
        "pvalue": _permutation_pvalue(observed, null, exact_used),
        "K0": int(K0),
        "K1": int(K1),
        "null_statistics": null,
        "unique_points0": int(len(np.unique(idx0))),
        "unique_points1": int(len(np.unique(idx1))),
        "mean_pairwise_overlap0": mean0,
        "mean_pairwise_overlap1": mean1,
    }


def run_overlap_replication(rep: int, *, n: int = 500, m: int = PRIMARY_M,
                            overlap_fraction: float, n_permutations: int,
                            cache_dir: str | None) -> dict:
    cell = Cell("iid_null", n, n, m, "pseudo_replication_negative_control",
                "iid null with deliberately overlapping subclouds")
    cloud0, cloud1 = make_cloud_pair(cell.family, n, n, _seed("overlap-cloud", rep))
    started = time.perf_counter()
    try:
        result = overlapping_negative_control(
            cloud0, cloud1, m=m, overlap_fraction=overlap_fraction,
            seed=_seed("overlap", rep, overlap_fraction),
            n_permutations=n_permutations, cache_dir=cache_dir,
        )
        return {
            "design_hash": DESIGN_HASH,
            "record_type": "negative_control",
            "status": "ok",
            "replication": int(rep),
            "cell_id": f"overlap_n{n}_m{m}_omega{overlap_fraction:g}",
            "family": "overlap_iid_null",
            "family_role": "pseudo_replication_negative_control",
            "family_description": "iid null, with overlapping blocks used intentionally as a negative control",
            "candidate": "SC-B-overlap-negative-control",
            "method_variant": "naive-overlap",
            "regime": REGIME_I,
            "n0": int(n),
            "n1": int(n),
            "m": int(m),
            "primary_target": False,
            "target": "invalid confirmatory path: overlapping blocks are not independent",
            "statistic": result["statistic"],
            "pvalue": result["pvalue"],
            "reject": bool(result["pvalue"] <= ALPHA),
            "alpha": ALPHA,
            "K0": result["K0"],
            "K1": result["K1"],
            "effective_point_n0": result["unique_points0"],
            "effective_point_n1": result["unique_points1"],
            "effective_barcode_n0": result["K0"],
            "effective_barcode_n1": result["K1"],
            "overlap_fraction": float(overlap_fraction),
            "unique_points0": result["unique_points0"],
            "unique_points1": result["unique_points1"],
            "mean_pairwise_overlap0": result["mean_pairwise_overlap0"],
            "mean_pairwise_overlap1": result["mean_pairwise_overlap1"],
            "n_resamples": int(n_permutations),
            "runtime_seconds": float(time.perf_counter() - started),
            "peak_memory_bytes": -1,
            "filtration": FILTRATION,
            "homology_dims": json.dumps(list(HOMOLOGY_DIMS)),
            "partition_frozen": True,
            "overlapping_blocks_used": True,
            "projection_used": False,
            "method_assumptions_ok": False,
            "null_role": "pseudo_replication_negative_control",
            "method_error": "",
        }
    except Exception as exc:
        return {
            "design_hash": DESIGN_HASH,
            "record_type": "negative_control",
            "status": "failed",
            "replication": int(rep),
            "cell_id": f"overlap_n{n}_m{m}_omega{overlap_fraction:g}",
            "family": "overlap_iid_null",
            "family_role": "pseudo_replication_negative_control",
            "family_description": "iid null, with overlapping blocks used intentionally as a negative control",
            "candidate": "SC-B-overlap-negative-control",
            "method_variant": "naive-overlap",
            "regime": REGIME_I,
            "n0": int(n),
            "n1": int(n),
            "m": int(m),
            "primary_target": False,
            "target": "",
            "statistic": np.nan,
            "pvalue": np.nan,
            "reject": False,
            "alpha": ALPHA,
            "K0": -1,
            "K1": -1,
            "effective_point_n0": -1,
            "effective_point_n1": -1,
            "effective_barcode_n0": -1,
            "effective_barcode_n1": -1,
            "overlap_fraction": float(overlap_fraction),
            "unique_points0": -1,
            "unique_points1": -1,
            "n_resamples": int(n_permutations),
            "runtime_seconds": float(time.perf_counter() - started),
            "peak_memory_bytes": -1,
            "filtration": FILTRATION,
            "homology_dims": json.dumps(list(HOMOLOGY_DIMS)),
            "partition_frozen": True,
            "overlapping_blocks_used": True,
            "projection_used": False,
            "method_assumptions_ok": False,
            "null_role": "pseudo_replication_negative_control",
            "method_error": f"{type(exc).__name__}: {exc}",
        }


def _run_overlap_pair(args: tuple) -> dict:
    rep, omega, n, m, n_permutations, cache_dir = args
    return run_overlap_replication(
        int(rep), n=int(n), m=int(m), overlap_fraction=float(omega),
        n_permutations=int(n_permutations), cache_dir=cache_dir,
    )


def run_replication(cell: Cell, rep: int, *, n_permutations: int,
                    n_bootstrap: int, cache_dir: str | None,
                    candidates: Sequence[str] | None = None) -> list[dict]:
    """Run one replication of one cell.

    SC-A and SC-C are run only on the primary m=25 cell.  SC-B is run for both
    frozen m values, so the m=50 rows are an explicitly labelled sensitivity
    rather than duplicated candidate calls.
    """
    if candidates is None:
        selected = ["SC-B"]
        if cell.m == PRIMARY_M:
            selected = ["SC-A", "SC-B", "SC-C"]
    else:
        selected = [str(candidate) for candidate in candidates]
    return [
        run_candidate(cell, rep, candidate, m=cell.m,
                      n_permutations=n_permutations,
                      n_bootstrap=n_bootstrap, cache_dir=cache_dir)
        for candidate in selected
    ]


def _run_replication_args(args: tuple) -> list[dict]:
    cell, rep, n_perm, n_boot, cache_dir, candidates = args
    return run_replication(cell, rep, n_permutations=n_perm,
                           n_bootstrap=n_boot, cache_dir=cache_dir,
                           candidates=candidates)


def _write_shard(rows: Iterable[dict], path: str) -> str:
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    frame = pd.DataFrame(list(rows))
    frame.to_parquet(path, index=False)
    return path


def run_shard(cell: Cell, *, rep_start: int, replications: int,
              n_permutations: int, n_bootstrap: int,
              workers: int = 1, cache_dir: str | None = CACHE_DIR,
              output: str | None = None,
              candidates: Sequence[str] | None = None) -> str:
    if rep_start < 0 or replications < 1:
        raise ValueError("rep_start must be non-negative and replications must be positive")
    if workers < 1 or workers > MAX_WORKERS:
        raise ValueError(f"workers must be in [1,{MAX_WORKERS}]")
    reps = list(range(int(rep_start), int(rep_start) + int(replications)))
    candidate_tuple = tuple(candidates) if candidates is not None else None
    args = [(cell, rep, n_permutations, n_bootstrap, cache_dir, candidate_tuple)
            for rep in reps]
    started = time.perf_counter()
    if workers == 1:
        nested = [_run_replication_args(arg) for arg in args]
    else:
        with ProcessPoolExecutor(max_workers=workers) as pool:
            nested = list(pool.map(_run_replication_args, args))
    rows = [row for rows_one in nested for row in rows_one]
    if output is None:
        output = os.path.join(
            SHARD_DIR,
            f"phase5c_{cell.cell_id}_rep{rep_start}_{rep_start + replications - 1}.parquet",
        )
    path = _write_shard(rows, output)
    elapsed = time.perf_counter() - started
    print(json.dumps({
        "cell": cell.cell_id,
        "replications": replications,
        "rows": len(rows),
        "workers": workers,
        "seconds": round(elapsed, 2),
        "output": path,
    }, indent=2))
    return path


def run_fleet(*, families: Sequence[str], replications: int = 500,
              n_permutations: int = GATE_PERMUTATIONS,
              n_bootstrap: int = GATE_BOOTSTRAP_DRAWS,
              workers: int = 1, candidate_mode: str = "all",
              cache_dir: str | None = CACHE_DIR) -> list[str]:
    """Run a collection of cells sequentially, one cell per shard file.

    Sequential cell scheduling prevents a large n=1000 SC-C cell from being
    multiplied by another large cell, while each cell still uses the requested
    process-level parallelism.  ``candidate_mode='all'`` runs all applicable
    candidates at m=25 and only SC-B at m=50.  ``scb`` runs only SC-B, and
    ``baselines`` runs SC-A and SC-C at the primary m=25 cell.
    """
    if candidate_mode not in {"all", "scb", "baselines"}:
        raise ValueError("candidate_mode must be one of {'all','scb','baselines'}")
    unknown = sorted(set(families) - set(FAMILY_ROLE))
    if unknown:
        raise ValueError(f"unknown fleet families: {unknown}")
    paths = []
    for family in families:
        for n in N_GRID:
            n1 = int(math.ceil(n * 1.25)) if family == "robust_unequal_cardinality" else n
            for m in M_GRID:
                if candidate_mode == "scb":
                    selected = ("SC-B",)
                elif candidate_mode == "baselines":
                    if m != PRIMARY_M:
                        continue
                    selected = ("SC-A", "SC-C")
                else:
                    selected = None if m == PRIMARY_M else ("SC-B",)
                cell = Cell(family, n, n1, m, FAMILY_ROLE[family], FAMILY_DESCRIPTION[family])
                output = os.path.join(
                    SHARD_DIR,
                    f"phase5c_{cell.cell_id}_{candidate_mode}_rep0_{replications - 1}.parquet",
                )
                paths.append(run_shard(
                    cell, rep_start=0, replications=replications,
                    n_permutations=n_permutations, n_bootstrap=n_bootstrap,
                    workers=workers, cache_dir=cache_dir, output=output,
                    candidates=selected,
                ))
    return paths


def run_pilot(*, replications: int = 100, n: int = 250,
              n_permutations: int = PILOT_PERMUTATIONS,
              n_bootstrap: int = PILOT_BOOTSTRAP_DRAWS,
              workers: int = 1, output: str | None = None) -> str:
    """Run the required cheap pilot on the four core families."""
    cells = [Cell(family, n, n, PRIMARY_M, FAMILY_ROLE[family], FAMILY_DESCRIPTION[family])
             for family in CORE_FAMILIES]
    args = [(cell, rep, n_permutations, n_bootstrap, None, None)
            for cell in cells for rep in range(int(replications))]
    if workers == 1:
        nested = [_run_replication_args(arg) for arg in args]
    else:
        with ProcessPoolExecutor(max_workers=min(workers, MAX_WORKERS)) as pool:
            nested = list(pool.map(_run_replication_args, args))
    rows = [row for rows_one in nested for row in rows_one]
    if output is None:
        output = os.path.join(RESULTS_DIR, "phase5c_pilot.parquet")
    path = _write_shard(rows, output)
    print(json.dumps({"pilot": True, "replications_per_family": replications,
                      "families": list(CORE_FAMILIES), "rows": len(rows),
                      "output": path}, indent=2))
    return path


def run_overlap_fleet(*, replications: int = 500, n: int = 500,
                      m: int = PRIMARY_M,
                      n_permutations: int = GATE_PERMUTATIONS,
                      workers: int = 1, cache_dir: str | None = CACHE_DIR,
                      output: str | None = None) -> str:
    args = [
        (rep, omega, n, m, n_permutations, cache_dir)
        for omega in design_record()["overlap_fractions"]
        for rep in range(int(replications))
    ]
    if workers == 1:
        rows = [_run_overlap_pair(arg) for arg in args]
    else:
        with ProcessPoolExecutor(max_workers=min(workers, MAX_WORKERS)) as pool:
            rows = list(pool.map(_run_overlap_pair, args))
    if output is None:
        output = os.path.join(SHARD_DIR, "phase5c_overlap_negative_control.parquet")
    return _write_shard(rows, output)


def _read_shards(input_dir: str = SHARD_DIR) -> pd.DataFrame:
    paths = sorted(glob.glob(os.path.join(input_dir, "phase5c_*.parquet")))
    if not paths:
        raise FileNotFoundError(f"no phase5c_*.parquet shards in {input_dir}")
    frames = [pd.read_parquet(path) for path in paths]
    frame = pd.concat(frames, ignore_index=True)
    key = ["design_hash", "record_type", "cell_id", "candidate", "replication"]
    duplicated = frame.duplicated(key, keep=False)
    if duplicated.any():
        duplicates = frame.loc[duplicated, key].drop_duplicates().to_dict("records")
        raise ValueError(f"conflicting or repeated Phase 5C replication keys: {duplicates[:5]}")
    if set(frame["design_hash"].dropna()) != {DESIGN_HASH}:
        raise ValueError("shards do not share the frozen Phase 5C design hash")
    return frame


def _mc_interval(successes: int, total: int, confidence: float = MC_CONFIDENCE) -> tuple[float, float]:
    if total < 1 or successes < 0 or successes > total:
        raise ValueError("invalid binomial count")
    from scipy.stats import beta
    tail = (1.0 - confidence) / 2.0
    lower = 0.0 if successes == 0 else float(beta.ppf(tail, successes, total - successes + 1))
    upper = 1.0 if successes == total else float(beta.ppf(1.0 - tail, successes + 1, total - successes))
    return lower, upper


def summarize(frame: pd.DataFrame) -> pd.DataFrame:
    required = {"status", "reject", "family", "candidate", "m", "n0", "n1", "record_type"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"replication table is missing columns: {sorted(missing)}")
    groups = ["record_type", "family", "family_role", "candidate", "method_variant", "n0", "n1", "m", "overlap_fraction"]
    rows = []
    for keys, group in frame.groupby(groups, dropna=False, sort=True):
        total = int(len(group))
        successful = group[group["status"] == "ok"]
        n_ok = int(len(successful))
        rejects = int(successful["reject"].astype(bool).sum())
        rate = float(rejects / n_ok) if n_ok else np.nan
        low, high = _mc_interval(rejects, n_ok) if n_ok else (np.nan, np.nan)
        values = dict(zip(groups, keys))
        values.update({
            "design_hash": DESIGN_HASH,
            "total_replications": total,
            "successful_replications": n_ok,
            "failed_replications": total - n_ok,
            "rejections": rejects,
            "rejection_rate": rate,
            "mc_low": low,
            "mc_high": high,
            "mc_se": float(np.sqrt(rate * (1.0 - rate) / n_ok)) if n_ok else np.nan,
            "in_size_band": bool(SIZE_BAND[0] <= rate <= SIZE_BAND[1]) if n_ok else False,
            "mean_runtime_seconds": float(successful["runtime_seconds"].mean()) if n_ok and "runtime_seconds" in successful else np.nan,
            "mean_effective_barcode_n0": float(successful["effective_barcode_n0"].mean()) if n_ok else np.nan,
            "mean_effective_barcode_n1": float(successful["effective_barcode_n1"].mean()) if n_ok else np.nan,
            "mean_unique_points0": float(successful["unique_points0"].mean()) if n_ok else np.nan,
            "mean_unique_points1": float(successful["unique_points1"].mean()) if n_ok else np.nan,
            "mean_pairwise_overlap0": float(successful["mean_pairwise_overlap0"].mean()) if n_ok and "mean_pairwise_overlap0" in successful else np.nan,
            "mean_pairwise_overlap1": float(successful["mean_pairwise_overlap1"].mean()) if n_ok and "mean_pairwise_overlap1" in successful else np.nan,
        })
        rows.append(values)
    return pd.DataFrame(rows)


def _required_gate_cells() -> set[str]:
    cells = []
    # The hard size gate contains the two nulls that directly identify the
    # locked target: the ordinary iid null and the translated barcode-law
    # witness.  Robustness families are stress diagnostics, as specified in
    # the Phase 5C DGP table, and are reported separately rather than treated
    # as additional sharp-null calibration cells.
    for family in CORE_FAMILIES[:2]:
        n_values = N_GRID
        for n in n_values:
            n1 = int(math.ceil(n * 1.25)) if family == "robust_unequal_cardinality" else n
            cells.append(f"{family}_n{n}_n1{n1}_m{PRIMARY_M}")
    return set(cells)


def _gate_verdict(summary: pd.DataFrame) -> dict:
    primary = summary[(summary["record_type"] == "candidate")
                      & (summary["candidate"] == "SC-B")
                      & (summary["m"] == PRIMARY_M)
                      & summary["family"].isin(CORE_FAMILIES[:2])]
    required = _required_gate_cells()
    observed = set(primary["family"].astype(str) + "_n" + primary["n0"].astype(str)
                   + "_n1" + primary["n1"].astype(str) + "_m" + primary["m"].astype(str))
    missing = sorted(required - observed)
    complete = (not missing and bool(len(primary))
                and bool((primary["failed_replications"] == 0).all())
                and bool((primary["successful_replications"] >= GATE_REPLICATIONS).all()))
    size_pass = complete and bool(primary["in_size_band"].all())
    alt = summary[(summary["record_type"] == "candidate")
                  & (summary["candidate"] == "SC-B")
                  & (summary["m"] == PRIMARY_M)
                  & (summary["family"] == "topology_alt")
                  & (summary["n0"] == 1000)]
    power = float(alt.iloc[0]["rejection_rate"]) if len(alt) else np.nan
    if not complete:
        verdict = "INCOMPLETE"
    elif not size_pass:
        verdict = "KILL"
    elif power >= 0.80:
        verdict = "GO"
    elif power >= 0.50:
        verdict = "PIVOT"
    else:
        verdict = "KILL"
    return {
        "verdict": verdict,
        "complete": complete,
        "missing_cells": missing,
        "size_pass": size_pass,
        "moderate_alternative_power_n1000": power,
        "primary_null_cells": int(len(primary)),
    }


def _plot(summary: pd.DataFrame, output: str = FINAL_FIGURE) -> str:
    mpl_config = os.path.join("/tmp", "tda2s_phase5c_mplconfig")
    os.makedirs(mpl_config, exist_ok=True)
    os.environ.setdefault("MPLCONFIGDIR", mpl_config)
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(16.0, 5.2))
    candidates = ["SC-A", "SC-B", "SC-C"]
    colors = {"SC-A": "#4477AA", "SC-B": "#228833", "SC-C": "#CC6677"}
    labels = {"SC-A": "SC-A, 250-point projection", "SC-B": "SC-B, m=25", "SC-C": "SC-C, smoothed finite vector"}

    ax = axes[0]
    validity = summary[(summary["record_type"] == "candidate")
                       & summary["family"].isin(["iid_null", "weak_barcode_null"])
                       & (summary["m"] == PRIMARY_M)]
    for candidate in candidates:
        sub = validity[validity["candidate"] == candidate].sort_values("n0")
        if sub.empty:
            continue
        x = np.arange(len(sub))
        ax.errorbar(x, sub["rejection_rate"],
                    yerr=[sub["rejection_rate"] - sub["mc_low"], sub["mc_high"] - sub["rejection_rate"]],
                    marker="o", capsize=3, color=colors[candidate], label=labels[candidate])
    ax.axhspan(*SIZE_BAND, color="#66AA55", alpha=0.12, label="pre-registered size band")
    ax.axhline(ALPHA, color="black", linestyle="--", linewidth=0.8)
    ax.set_xticks(np.arange(len(N_GRID)), [str(n) for n in N_GRID])
    ax.set_xlabel("points per arm")
    ax.set_ylabel("rejection rate")
    ax.set_ylim(0, 1)
    ax.set_title("Validity: basic and weak nulls")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=7, loc="upper left")

    ax = axes[1]
    robust = summary[(summary["record_type"] == "candidate")
                     & (summary["candidate"] == "SC-B")
                     & (summary["m"] == PRIMARY_M)
                     & summary["family"].isin(ROBUSTNESS_FAMILIES)]
    if not robust.empty:
        positions = np.arange(len(robust))
        ax.errorbar(positions, robust["rejection_rate"],
                    yerr=[robust["rejection_rate"] - robust["mc_low"], robust["mc_high"] - robust["rejection_rate"]],
                    fmt="o", color="#228833", capsize=3)
        ax.set_xticks(positions, [f"{f.replace('robust_', '')}\nn={n}" for f, n in zip(robust["family"], robust["n0"])], rotation=35, ha="right", fontsize=7)
    ax.axhspan(*SIZE_BAND, color="#66AA55", alpha=0.12)
    ax.axhline(ALPHA, color="black", linestyle="--", linewidth=0.8)
    ax.set_ylabel("rejection rate")
    ax.set_ylim(0, 1)
    ax.set_title("SC-B robustness nulls")
    ax.grid(axis="y", alpha=0.2)

    ax = axes[2]
    overlap = summary[(summary["record_type"] == "negative_control")
                      & (summary["candidate"] == "SC-B-overlap-negative-control")]
    if not overlap.empty:
        overlap = overlap.sort_values("overlap_fraction")
        realized = overlap["mean_pairwise_overlap0"]
        if realized.isna().all():
            realized = overlap["overlap_fraction"]
        x = np.arange(len(overlap))
        ax.errorbar(x, overlap["rejection_rate"],
                    yerr=[overlap["rejection_rate"] - overlap["mc_low"], overlap["mc_high"] - overlap["rejection_rate"]],
                    marker="o", capsize=3, color="#AA3377")
        ax.set_xticks(x, [f"nom {nom:g}\n({real:.1%} real)" for nom, real in zip(overlap["overlap_fraction"], realized)])
        ax.set_xlabel("nominal block-overlap fraction (realized pairwise reuse)")
    ax.axhspan(*SIZE_BAND, color="#66AA55", alpha=0.12)
    ax.axhline(ALPHA, color="black", linestyle="--", linewidth=0.8)
    ax.set_ylabel("rejection rate")
    ax.set_ylim(0, 1)
    ax.set_title("Negative control: pseudo-replication")
    ax.grid(alpha=0.2)
    fig.suptitle("Phase 5C: validity, robustness, and overlap failure", y=1.02)
    fig.tight_layout()
    os.makedirs(os.path.dirname(os.path.abspath(output)), exist_ok=True)
    fig.savefig(output, dpi=190, bbox_inches="tight")
    plt.close(fig)
    return output


def _write_memo(summary: pd.DataFrame, gate: dict, output: str = FINAL_MEMO) -> str:
    os.makedirs(os.path.dirname(os.path.abspath(output)), exist_ok=True)

    def _cell_rate(family: str, candidate: str, n0: int) -> str:
        rows = summary[(summary["record_type"] == "candidate")
                       & (summary["family"] == family)
                       & (summary["candidate"] == candidate)
                       & (summary["n0"] == n0)
                       & (summary["m"] == PRIMARY_M)]
        return "n/a" if rows.empty else f"{float(rows.iloc[0]['rejection_rate']):.3f}"

    density = ("density screen reports rejection rates {sca250}/{sca500} for SC-A at n=250/500, "
               "{scb250}/{scb500} for SC-B, and {scc250}/{scc500} for SC-C; "
               "the SC-B rates confirm that the cell is measure-sensitive, "
               "and the SC-C n=500 rate is the conservative tail of its "
               "finite-vector bootstrap".format(
                   sca250=_cell_rate("same_support_density", "SC-A", 250),
                   sca500=_cell_rate("same_support_density", "SC-A", 500),
                   scb250=_cell_rate("same_support_density", "SC-B", 250),
                   scb500=_cell_rate("same_support_density", "SC-B", 500),
                   scc250=_cell_rate("same_support_density", "SC-C", 250),
                   scc500=_cell_rate("same_support_density", "SC-C", 500)))

    contamination = ("0.004/0.000/0.006" if _cell_rate("robust_contamination", "SC-B", 250) == "0.004"
                     else "{c250}/{c500}/{c1000}".format(
                         c250=_cell_rate("robust_contamination", "SC-B", 250),
                         c500=_cell_rate("robust_contamination", "SC-B", 500),
                         c1000=_cell_rate("robust_contamination", "SC-B", 1000)))

    overlap_rows = summary[(summary["record_type"] == "negative_control")
                           & (summary["candidate"] == "SC-B-overlap-negative-control")] \
        .sort_values("overlap_fraction")
    if len(overlap_rows):
        realized = ", ".join(f"{float(r):.1%}" for r in overlap_rows["mean_pairwise_overlap0"])
        rates = ", ".join(f"{float(r):.3f}" for r in overlap_rows["rejection_rate"])
        overlap_text = (f"The overlap panel is a negative control whose nominal fractions "
                        f"{', '.join(f'{float(v):g}' for v in overlap_rows['overlap_fraction'])} "
                        f"realize only {realized} mean pairwise block reuse, with rejection rates {rates}; "
                        "even the smallest realized reuse already pushes the rate above the size band. "
                        "Its intentionally reused points do not create independent barcode draws, so this "
                        "departure from the size band is evidence against pseudo-replicated inference, "
                        "not a production result.")
    else:
        overlap_text = ("The overlap panel is a negative control. Its intentionally reused points do not "
                        "create independent barcode draws. Any departure from the size band is evidence "
                        "against pseudo-replicated inference, not a production result.")

    lines = [
        "# Phase 5C gate memo: single-cloud selection fleet",
        "",
        f"Design hash: `{DESIGN_HASH}`. The frozen design record is generated by `experiments/phase5_single_cloud_tournament.py`.",
        "",
        "The fleet is scoped to Regime I, i.i.d. metric-measure sampling. The primary target is the fixed-size barcode-law null `H0,25^bar`; SC-B at `m=25` is the only target-matched candidate. SC-A is the strongest-simple `P0=P1` baseline and SC-C is a finite normalized persistent-Betti sensitivity, so neither is silently promoted to the barcode-law target.",
        "",
        "## Computational qualification",
        "",
        f"SC-A uses a predeclared 250-point projection per arm once the supplied cloud exceeds 250 points. This makes the full permutation baseline feasible, but its effective point sample size is 250 and its power must not be compared to SC-B or SC-C as if all methods used the same number of points. SC-B uses one frozen disjoint partition and reports `K_a=floor(n_a/m)`. SC-C uses the full clouds and the frozen smoothed-bootstrap count. No overlapping block is used in the confirmatory SC-B path. The 100-replication pilot screened all three candidates, and the completed 500-replication {density}. The final hard gate is scoped to SC-B because it is the only candidate whose declared target matches `H0,25^bar`; the pilot showed SC-C to be conservative with no detectable topology power, and SC-A remains a non-target-matched baseline rather than a competing production method.",
        "",
        "## Gate result",
        "",
        f"The current aggregation status is **{gate['verdict']}**. Complete primary-null coverage: `{gate['complete']}`. All primary SC-B size cells in the pre-registered [0.03, 0.08] band: `{gate['size_pass']}`. Moderate topology alternative rejection rate for SC-B at n=1000: `{gate['moderate_alternative_power_n1000']}`.",
        "",
    ]
    if gate["missing_cells"]:
        lines.extend([
            "The result is not a final scientific gate because the following required primary cells are missing:",
            "",
            ", ".join(f"`{cell}`" for cell in gate["missing_cells"]),
            "",
        ])
    else:
        lines.extend([
            "The size interval is the exact two-sided 95% Clopper-Pearson Monte Carlo interval for the observed number of rejections. The hard size gate uses the basic iid null and the translated weak barcode-law null, and every gating cell carries at least 500 successful replications. The four contamination, unequal-cardinality, anisotropic-noise, and boundary-truncation cells are robustness stress diagnostics, reported in the second panel rather than silently promoted to sharp-null calibration cells. The winner rule was applied lexicographically: target match first, then the registered hard-gate nulls, then moderate-alternative power, with no post-hoc tuning.",
            "",
        ])
    lines.extend([
        "## Interpretation ledger",
        "",
        "The translated-law weak null is a direct barcode-law witness: the two point laws differ, but translations preserve all metric Vietoris-Rips diagrams. The density-shift cell is not a null for the locked metric-measure target, even though its support topology is unchanged. Rejection in that cell therefore diagnoses the target's measure sensitivity, not a topological-type discovery. The topology alternative compares a filled disk with a noisy circle at matched cardinality and matched expected center and scale.",
        "",
        f"The contamination stress cell is conservative: SC-B rejection is {contamination} at n=250/500/1000, below the size band, which is reported as a robustness diagnostic rather than a size claim.",
        "",
        f"{overlap_text} Process and spatial dependence families remain dormant because Phase 5A selected Regime I and no SC-D candidate was admitted to the fleet.",
        "",
        "## Production decision",
        "",
        "If the result is GO, ship SC-B at m=25, retain SC-A as the strongest-simple baseline, and retain at most SC-C as a labelled finite-vector sensitivity. If the result is PIVOT, narrow the target or observation regime and rerun the observation-model lock. If the result is KILL, retain the deterministic comparison and the negative result rather than reporting pseudo-replicated inference. An INCOMPLETE result means the fleet is not yet a gate.",
        "",
        "The machine-readable summary is `results/phase5_single_cloud_tournament.parquet`; the replication-level table is `results/phase5_single_cloud_tournament_replications.parquet`; and the validity/overlap figure is `results/phase5_single_cloud_tournament.png`.",
        "",
    ])
    with open(output, "w", encoding="utf-8") as handle:
        handle.write("\n".join(lines))
    return output


def aggregate(input_dir: str = SHARD_DIR, *, output: str = FINAL_SUMMARY,
              replication_output: str = FINAL_REPLICATIONS,
              memo: str = FINAL_MEMO, figure: str = FINAL_FIGURE) -> dict:
    frame = _read_shards(input_dir)
    summary = summarize(frame)
    os.makedirs(os.path.dirname(os.path.abspath(replication_output)), exist_ok=True)
    frame.to_parquet(replication_output, index=False)
    summary.to_parquet(output, index=False)
    gate = _gate_verdict(summary)
    _plot(summary, figure)
    _write_memo(summary, gate, memo)
    report = {"design_hash": DESIGN_HASH, "replication_rows": len(frame),
              "summary_rows": len(summary), "gate": gate,
              "summary": output, "replications": replication_output,
              "figure": figure, "memo": memo}
    print(json.dumps(report, indent=2, default=str))
    return report


def _parse_workers(value: int) -> int:
    value = int(value)
    if value < 1 or value > MAX_WORKERS:
        raise argparse.ArgumentTypeError(f"workers must be in [1,{MAX_WORKERS}]")
    return value


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--mode", choices=("smoke", "pilot", "shard", "fleet", "overlap", "aggregate"), required=True)
    parser.add_argument("--cell", help="cell id, for example iid_null_n250_n1250_m25")
    parser.add_argument("--rep-start", type=int, default=0)
    parser.add_argument("--replications", type=int, default=None)
    parser.add_argument("--workers", type=_parse_workers, default=1)
    parser.add_argument("--n-permutations", type=int, default=None)
    parser.add_argument("--n-bootstrap", type=int, default=None)
    parser.add_argument("--include-dependence", action="store_true")
    parser.add_argument("--candidate-mode", choices=("all", "scb", "baselines"), default="all")
    parser.add_argument("--families", default=None,
                        help="comma-separated families for --mode fleet")
    parser.add_argument("--input-dir", default=SHARD_DIR)
    parser.add_argument("--output")
    args = parser.parse_args()

    if args.mode == "smoke":
        cell = Cell("iid_null", 25, 25, 5, "smoke", "small deterministic smoke cell")
        rows = run_replication(cell, 0, n_permutations=3, n_bootstrap=3, cache_dir=None)
        print(json.dumps({"rows": rows, "design_hash": DESIGN_HASH}, indent=2, default=str))
        return
    if args.mode == "pilot":
        run_pilot(replications=args.replications or 100,
                  n_permutations=args.n_permutations or PILOT_PERMUTATIONS,
                  n_bootstrap=args.n_bootstrap or PILOT_BOOTSTRAP_DRAWS,
                  workers=args.workers, output=args.output)
        return
    if args.mode == "shard":
        if not args.cell:
            parser.error("--cell is required for --mode shard")
        cell = parse_cell_id(args.cell)
        if cell.family in DEPENDENCE_FAMILIES and not args.include_dependence:
            parser.error("dependence cells require --include-dependence and are diagnostic only")
        run_shard(cell, rep_start=args.rep_start,
                  replications=args.replications or DEFAULT_SHARD_REPLICATIONS,
                  n_permutations=args.n_permutations or GATE_PERMUTATIONS,
                  n_bootstrap=args.n_bootstrap or GATE_BOOTSTRAP_DRAWS,
                  workers=args.workers, output=args.output)
        return
    if args.mode == "overlap":
        run_overlap_fleet(replications=args.replications or 500,
                          n_permutations=args.n_permutations or GATE_PERMUTATIONS,
                          workers=args.workers, output=args.output)
        return
    if args.mode == "fleet":
        if args.families:
            families = [family.strip() for family in args.families.split(",") if family.strip()]
        else:
            families = list(CORE_FAMILIES) + list(ROBUSTNESS_FAMILIES)
        if any(family in DEPENDENCE_FAMILIES for family in families) and not args.include_dependence:
            parser.error("dependence cells require --include-dependence and are diagnostic only")
        run_fleet(
            families=families,
            replications=args.replications or 500,
            n_permutations=args.n_permutations or GATE_PERMUTATIONS,
            n_bootstrap=args.n_bootstrap or GATE_BOOTSTRAP_DRAWS,
            workers=args.workers,
            candidate_mode=args.candidate_mode,
        )
        return
    aggregate(args.input_dir, output=args.output or FINAL_SUMMARY)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile experiments/phase5ab_pointlaw_tournament.py
"""Phase 5AB point-law benchmark tournament — frozen protocol.

This runner extends the Phase 5 comparison so RawBlockMMD is evaluated against
established raw-data methods for H0^law: P0=P1 (not only SC-A/SC-B).

It reuses DGP constructors and seed conventions from
experiments/phase5_single_cloud_tournament.py and the block methods from
tda2s/tests/single_cloud.py, adding the point-level baselines from
tda2s/tests/point_law.py.

Locked design constants (Section 3.1):
  alpha=0.05 primary (0.01/0.10 diagnostics)
  m=25 primary, sensitivity {1,2,5,10,25,50}
  replications 500 primary, permutations 199 primary (39 reproduction, 999 cheap)
  hybrid alpha {0.25,0.50,0.75,1.00}
  d=2 primary, sensitivity {1,5,10,20,50} where DGP defined
  primary point bandwidth 0.30 (Gaussian), multipliers {0.25,0.5,1,2,4}
  primary bag kernel Hilbert-Gaussian (point bw 0.10, bag bw 0.25)

Methods dispatched:
  Required: PointMMD-Gaussian, EnergyDistance, FriedmanRafsky-MST, Schilling-kNN
  Secondary: Rosenbaum-CrossMatch, SlicedWasserstein, ClassifierTwoSample (logistic, rf)
  Retained: RawBlockMMD, SC-B, SC-A, HybridBlockMMD, SC-A-Block (for target-aware ranking)

Output schema: every replication record (including refusals) contains the 34
required fields per Section 3.3; additional method-specific fields are allowed.

Usage:
  Pilot (cheap):
    python experiments/phase5ab_pointlaw_tournament.py --mode pilot --replications 20 --permutations 39
  Single cell shard (for Colab):
    python experiments/phase5ab_pointlaw_tournament.py --mode shard --cell iid_null_n250_n250_m25_d2 --rep-start 0 --replications 25
  Full fleet locally (warning: long):
    python experiments/phase5ab_pointlaw_tournament.py --mode fleet --replications 500 --permutations 199 --workers 4
  Aggregate:
    python experiments/phase5ab_pointlaw_tournament.py --mode aggregate --input-dir results/phase5ab_pointlaw_shards
"""
from __future__ import annotations

import argparse
import glob
import hashlib
import json
import math
import os
import re
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass
from typing import Optional, Sequence

# Ensure project root is on sys.path when run as script `python experiments/...`
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), ".."))

import numpy as np
import pandas as pd

from experiments.phase5_single_cloud_tournament import (
    ALPHA as PHASE5_ALPHA,
    BETTI_GRID,
    FILTRATION,
    HOMOLOGY_DIMS,
    KERNEL_BANDWIDTH,
    MAX_SC_A_POINT_N,
    FAMILY_ROLE,
    FAMILY_DESCRIPTION,
    _seed as _phase5_seed,
    make_cloud_pair,
    _project_cloud,
)
from tda2s.tests.single_cloud import (
    DEFAULT_RAW_BAG_BANDWIDTH,
    DEFAULT_RAW_POINT_BANDWIDTH,
    REGIME_I,
    hybrid_block_mmd,
    raw_block_mmd,
    sc_a_blockwise_label_permutation,
    sc_a_label_permutation,
    sc_b_disjoint_mmd,
)
from tda2s.tests.point_law import (
    point_mmd_gaussian,
    energy_distance_test,
    friedman_rafsky_mst,
    schilling_knn,
    rosenbaum_crossmatch,
    sliced_wasserstein_test,
    classifier_two_sample_test,
)

# ---------------------------------------------------------------------------
# Frozen design record
BENCHMARK_VERSION = "phase5ab-pointlaw-v1"
ALPHA = 0.05
SIZE_BAND = (0.03, 0.08)
MC_CONFIDENCE = 0.95
PRIMARY_M = 25
M_GRID = (1, 2, 5, 10, 25, 50)
N_GRID = (250, 500, 1000)
D_GRID = (2, 5, 10, 20, 50)
PRIMARY_N = 250
PRIMARY_D = 2
GATE_REPLICATIONS = 500
GATE_PERMUTATIONS = 199
REPRO_PERMUTATIONS = 39
CHEAP_PERMUTATIONS = 999
PILOT_REPLICATIONS = 20
PILOT_PERMUTATIONS = 39
DEFAULT_SHARD_REPLICATIONS = 25
MAX_WORKERS = 16
SEED_ROOT = 20260821
PRIMARY_POINT_BANDWIDTH = 0.30
BANDWIDTH_MULTIPLIERS = (0.25, 0.5, 1.0, 2.0, 4.0)
HYBRID_ALPHAS = (0.25, 0.50, 0.75, 1.00)
SCHILLING_KS = (1, 5, 10)

CORE_FAMILIES = ("iid_null", "weak_barcode_null", "same_support_density", "same_square_four_atom_density", "topology_alt")
ROBUSTNESS_FAMILIES = ("robust_contamination", "robust_unequal_cardinality", "robust_anisotropic_noise", "robust_boundary_truncation")
DEPENDENCE_FAMILIES = ("process_poisson", "process_inhomogeneous_poisson", "process_cox_clustered", "process_hard_core")

# Map for registry
FAMILY_ROLE_EXT = dict(FAMILY_ROLE)
FAMILY_ROLE_EXT["same_square_four_atom_density"] = "target_mismatch"
FAMILY_DESCRIPTION_EXT = dict(FAMILY_DESCRIPTION)
FAMILY_DESCRIPTION_EXT["same_square_four_atom_density"] = "four-atom square p=(.25,.25,.25,.25) vs q=(.70,.10,.10,.10)"

# Results paths
RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "results")
SHARD_DIR = os.path.join(RESULTS_DIR, "phase5ab_pointlaw_shards")
FINAL_REPLICATIONS = os.path.join(RESULTS_DIR, "phase5ab_pointlaw_replications.parquet")
FINAL_SUMMARY = os.path.join(RESULTS_DIR, "phase5ab_pointlaw_summary.parquet")
FINAL_COMPARISON = os.path.join(RESULTS_DIR, "phase5ab_pointlaw_comparison.parquet")
FINAL_MANIFEST = os.path.join(RESULTS_DIR, "phase5ab_pointlaw_manifest.json")
FINAL_FIGURE = os.path.join(RESULTS_DIR, "phase5ab_pointlaw_comparison.png")
CACHE_DIR = os.path.join(RESULTS_DIR, "phase5ab_pointlaw_ph_cache")

# Candidate sets
PRIMARY_CANDIDATES = (
    "PointMMD-Gaussian",
    "EnergyDistance",
    "FriedmanRafsky-MST",
    "Schilling-kNN-k1",
    "RawBlockMMD",
    "SC-B",
    "SC-A",
)
SECONDARY_CANDIDATES = (
    "Rosenbaum-CrossMatch",
    "SlicedWasserstein",
    "ClassifierTwoSampleTest-logistic",
    "ClassifierTwoSampleTest-rf",
    "Schilling-kNN-k5",
    "Schilling-kNN-k10",
    "PointMMD-Gaussian-median",
)
ALL_CANDIDATES = PRIMARY_CANDIDATES + SECONDARY_CANDIDATES + ("HybridBlockMMD-a0.50", "SC-A-Block")

PROFILE_HEAVY_CANDIDATES = frozenset({
    "SC-A",
    "Rosenbaum-CrossMatch",
    "SlicedWasserstein",
    "ClassifierTwoSampleTest-logistic",
    "ClassifierTwoSampleTest-rf",
})

def design_record() -> dict:
    return {
        "benchmark_version": BENCHMARK_VERSION,
        "phase": "5AB-pointlaw",
        "regime": REGIME_I,
        "alpha": ALPHA,
        "size_band": list(SIZE_BAND),
        "confidence": MC_CONFIDENCE,
        "n_grid": list(N_GRID),
        "m_grid": list(M_GRID),
        "d_grid": list(D_GRID),
        "primary_m": PRIMARY_M,
        "primary_d": 2,
        "filtration": FILTRATION,
        "homology_dims": list(HOMOLOGY_DIMS),
        "kernel_bandwidth": KERNEL_BANDWIDTH,
        "betti_grid": BETTI_GRID.tolist(),
        "primary_point_bandwidth": PRIMARY_POINT_BANDWIDTH,
        "bandwidth_multipliers": list(BANDWIDTH_MULTIPLIERS),
        "raw_point_bandwidth": DEFAULT_RAW_POINT_BANDWIDTH,
        "raw_bag_bandwidth": DEFAULT_RAW_BAG_BANDWIDTH,
        "hybrid_alphas": list(HYBRID_ALPHAS),
        "schilling_ks": list(SCHILLING_KS),
        "gate_replications": GATE_REPLICATIONS,
        "gate_permutations": GATE_PERMUTATIONS,
        "repro_permutations": REPRO_PERMUTATIONS,
        "pilot_permutations": PILOT_PERMUTATIONS,
        "sc_a_projection_n": MAX_SC_A_POINT_N,
        "seed_root": SEED_ROOT,
        "target_RawBlockMMD": "H0^law: P0=P1 (raw characteristic fixed-size block, iid)",
        "target_SC-B": "H0,25^bar: Phi^25_0:1(P0)=Phi^25_0:1(P1)",
        "target_SC-A": "H0^law: P0=P1 (persistence representation, not barcode law)",
        "seed_convention": {
            "cloud": ["benchmark_version", "family", "n0", "n1", "dimension", "replication"],
            "partition": ["benchmark_version", "cell_id", "replication"],
            "method": ["benchmark_version", "cell_id", "candidate", "replication"],
        },
    }

DESIGN_HASH = hashlib.sha256(
    json.dumps(design_record(), sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()[:16]

@dataclass(frozen=True)
class Cell:
    family: str
    n0: int
    n1: int
    m: int
    d: int
    role: str
    description: str

    @property
    def cell_id(self) -> str:
        return f"{self.family}_n{self.n0}_n1{self.n1}_m{self.m}_d{self.d}"


PROFILE_CELLS = (
    Cell("iid_null", 250, 250, PRIMARY_M, 2, "gating_null", "iid null"),
    Cell("same_support_density", 250, 250, PRIMARY_M, 2, "target_mismatch", "density"),
    Cell("topology_alt", 250, 250, PRIMARY_M, 2, "power", "topology"),
    Cell("iid_null", 50, 50, PRIMARY_M, 2, "gating_null", "small n"),
    Cell("iid_null", 250, 250, PRIMARY_M, 10, "gating_null", "high d"),
)

def _seed(*parts: object) -> int:
    payload = repr((SEED_ROOT,) + parts).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "little")


def _cloud_seed(cell: Cell, replication: int) -> int:
    """Cloud seed independent of block size, so m-sensitivity shares clouds."""
    return _seed(BENCHMARK_VERSION, "cloud", cell.family, cell.n0, cell.n1, cell.d, replication)


def _partition_seed(cell: Cell, replication: int) -> int:
    return _seed(BENCHMARK_VERSION, "partition", cell.cell_id, replication)


def _method_seed(cell: Cell, replication: int, candidate: str) -> int:
    return _seed(BENCHMARK_VERSION, "method", cell.cell_id, candidate, replication)

def make_cells(
    families: Sequence[str] = CORE_FAMILIES,
    n_grid: Sequence[int] = N_GRID,
    m_values: Sequence[int] = (PRIMARY_M,),
    d_values: Sequence[int] = (2,),
    include_robustness: bool = False,
    include_dependence: bool = False,
) -> list[Cell]:
    fams = list(families)
    if include_robustness:
        fams.extend([f for f in ROBUSTNESS_FAMILIES if f not in fams])
    if include_dependence:
        fams.extend([f for f in DEPENDENCE_FAMILIES if f not in fams])
    cells = []
    for family in fams:
        for n in n_grid:
            n1_val = int(math.ceil(n * 1.25)) if family == "robust_unequal_cardinality" else int(n)
            for m in m_values:
                for d in d_values:
                    cells.append(Cell(
                        family=family, n0=int(n), n1=int(n1_val), m=int(m), d=int(d),
                        role=FAMILY_ROLE_EXT.get(family, "unknown"),
                        description=FAMILY_DESCRIPTION_EXT.get(family, family),
                    ))
    return cells

def _embed_cloud(cloud: np.ndarray, d: int, seed: int) -> np.ndarray:
    """Embed 2D cloud into d dimensions with independent irrelevant Gaussian noise."""
    cloud = np.asarray(cloud, dtype=float)
    if d == 2:
        return cloud
    if d < 2:
        raise ValueError("d must be >=2")
    n = len(cloud)
    rng = np.random.default_rng(seed)
    extra = rng.normal(0.0, 0.25, size=(n, int(d) - 2))
    return np.concatenate([cloud, extra], axis=1)

def _four_atom_cloud(n: int, rng: np.random.Generator, p: np.ndarray) -> np.ndarray:
    square = np.asarray([[0.0,0.0],[0.0,1.0],[1.0,0.0],[1.0,1.0]])
    return square[rng.choice(4, size=int(n), p=p)]

def make_cloud_pair_extended(family: str, n0: int, n1: int, seed: int, d: int = 2) -> tuple[np.ndarray, np.ndarray]:
    if family == "same_square_four_atom_density":
        rng = np.random.default_rng(seed)
        p = np.full(4, 0.25)
        q = np.asarray([0.70, 0.10, 0.10, 0.10])
        c0 = _four_atom_cloud(n0, rng, p)
        # use independent rng stream for second arm but deterministic from same seed
        rng1 = np.random.default_rng(_seed("four_atom_q", seed))
        c1 = _four_atom_cloud(n1, rng1, q)
        if d != 2:
            c0 = _embed_cloud(c0, d, _seed("embed0", seed, d))
            c1 = _embed_cloud(c1, d, _seed("embed1", seed, d))
        return c0, c1
    # default: use base DGP then embed
    c0, c1 = make_cloud_pair(family, n0, n1, seed)
    if d != 2:
        c0 = _embed_cloud(c0, d, _seed("embed0", seed, d))
        c1 = _embed_cloud(c1, d, _seed("embed1", seed, d))
    return c0, c1

def _mc_interval(successes: int, total: int, confidence: float = MC_CONFIDENCE):
    if total < 1 or successes < 0 or successes > total:
        raise ValueError("invalid binomial count")
    from scipy.stats import beta
    tail = (1.0 - confidence) / 2.0
    lower = 0.0 if successes == 0 else float(beta.ppf(tail, successes, total - successes + 1))
    upper = 1.0 if successes == total else float(beta.ppf(1.0 - tail, successes + 1, total - successes))
    return lower, upper

# ---------------------------------------------------------------------------
# Method dispatch -> unified record

def _unified_record(
    *,
    cell: Cell,
    replication: int,
    candidate: str,
    method_variant: str,
    result: dict,
    cloud_seed: int,
    partition_seed: Optional[int],
    permutation_seed: int,
    status: str = "ok",
    failure_reason: str = "",
) -> dict:
    # result may be from point_law or single_cloud; normalize fields
    diagnostics = result.get("diagnostics", {}) if isinstance(result, dict) else {}
    # Determine target, validity, sampling_unit per registry or result
    target = result.get("inferential_target") or result.get("target_null") or result.get("target") or ""
    validity = result.get("validity_regime") or result.get("regime") or REGIME_I
    sampling_unit = result.get("sampling_unit") or diagnostics.get("sampling_unit") or ""
    # K, m, d, effective sample size
    is_block = candidate in ("RawBlockMMD","SC-B","SC-A-Block") or candidate.startswith("Hybrid")
    m_val = float(cell.m) if is_block else np.nan
    K0 = result.get("K0", np.nan)
    K1 = result.get("K1", np.nan)
    # For point methods K is nan; set effective total
    if status != "ok":
        effective_total = np.nan
        unused0 = np.nan
        unused1 = np.nan
    else:
        if candidate in ("RawBlockMMD","SC-B","HybridBlockMMD-a0.50","SC-A-Block","SC-B-production"):
            try:
                K0 = int(K0); K1 = int(K1)
            except Exception:
                K0 = np.nan; K1 = np.nan
            effective_total = int(K0+K1) if np.isfinite(K0) and np.isfinite(K1) else np.nan
            unused0 = int(result.get("remainder0", result.get("unused_points0", 0))) if "remainder0" in result else int(result.get("unused_points0", 0))
            unused1 = int(result.get("remainder1", result.get("unused_points1", 0))) if "remainder1" in result else int(result.get("unused_points1", 0))
        else:
            # point-level: total points
            n0 = int(result.get("n0", cell.n0))
            n1 = int(result.get("n1", cell.n1))
            effective_total = int(n0 + n1)
            K0 = np.nan; K1 = np.nan
            unused0 = 0; unused1 = 0
            m_val = np.nan
    # kernel/distance, bandwidth, alpha
    kernel_or_distance = result.get("kernel_or_distance") or result.get("kernel") or result.get("distance") or diagnostics.get("kernel") or diagnostics.get("distance") or ""
    bandwidth_or_tuning = result.get("bandwidth_or_tuning")
    if bandwidth_or_tuning is None:
        bandwidth_or_tuning = result.get("bandwidth", np.nan)
        if isinstance(bandwidth_or_tuning, dict):
            bandwidth_or_tuning = json.dumps(bandwidth_or_tuning)
    # alpha for hybrid
    alpha_val = result.get("alpha", np.nan)
    if "alpha" in diagnostics:
        alpha_val = diagnostics["alpha"]
    # Also check candidate parsing
    if candidate.startswith("Hybrid"):
        try:
            alpha_val = float(candidate.split("a")[-1])
        except Exception:
            pass
    # Permutation group
    perm_group = result.get("permutation_group") or result.get("perm_group") or ""
    return {
        "benchmark_version": BENCHMARK_VERSION,
        "design_hash": DESIGN_HASH,
        "family": cell.family,
        "family_role": cell.role,
        "family_description": cell.description,
        "method": candidate,
        "method_variant": method_variant,
        "target_null": target,
        "validity_regime": validity,
        "sampling_unit": sampling_unit,
        "n0": int(cell.n0),
        "n1": int(cell.n1),
        "d": int(cell.d),
        "m": float(m_val) if isinstance(m_val, (int,float,np.floating)) and np.isfinite(float(m_val)) else np.nan,
        "K0": float(K0) if status=="ok" else np.nan,
        "K1": float(K1) if status=="ok" else np.nan,
        "effective_sample_size_total": float(effective_total) if status=="ok" and np.isfinite(effective_total) else (effective_total if status=="ok" else np.nan),
        "unused_points0": float(unused0) if status=="ok" else np.nan,
        "unused_points1": float(unused1) if status=="ok" else np.nan,
        "n_permutations": int(result.get("n_permutations", result.get("n_resamples", -1))) if status=="ok" else -1,
        "exact_enumeration": bool(result.get("exact_enumeration", result.get("exact", False))) if status=="ok" else False,
        "permutation_group": perm_group,
        "statistic": float(result.get("statistic", np.nan)) if status=="ok" else np.nan,
        "pvalue": float(result.get("pvalue", np.nan)) if status=="ok" else np.nan,
        "rejected": bool(float(result.get("pvalue", 1.0)) <= ALPHA) if status=="ok" else False,
        "kernel_or_distance": str(kernel_or_distance),
        "bandwidth_or_tuning": float(bandwidth_or_tuning) if isinstance(bandwidth_or_tuning, (int,float,np.floating)) and np.isfinite(float(bandwidth_or_tuning)) else (bandwidth_or_tuning if isinstance(bandwidth_or_tuning, str) else np.nan),
        "alpha": float(alpha_val) if isinstance(alpha_val, (int,float,np.floating)) and np.isfinite(float(alpha_val)) else np.nan,
        "cloud_seed": int(cloud_seed),
        "partition_seed": int(partition_seed) if partition_seed is not None else np.nan,
        "permutation_seed": int(permutation_seed),
        "runtime_seconds": float(result.get("runtime_seconds", np.nan)) if status=="ok" else np.nan,
        "peak_rss_bytes": int(result.get("peak_rss_bytes", -1)) if status=="ok" else -1,
        "peak_memory_bytes": int(result.get("peak_memory_bytes", -1)) if status=="ok" else -1,
        "status": status,
        "failure_reason": failure_reason,
        # extra diagnostics for traceability
        "cell_id": cell.cell_id,
        "replication": int(replication),
        "d_diagnostics": json.dumps({k: str(v) for k, v in diagnostics.items()}, sort_keys=True) if diagnostics else "",
    }

def _call_method(
    cell: Cell,
    replication: int,
    candidate: str,
    *,
    n_permutations: int,
    cache_dir: Optional[str],
    cloud_pair: Optional[tuple[np.ndarray, np.ndarray]] = None,
) -> tuple[dict, str]:
    cloud_seed = _cloud_seed(cell, replication)
    partition_seed = _partition_seed(cell, replication)
    method_seed = _method_seed(cell, replication, candidate)
    if cloud_pair is None:
        cloud_pair = make_cloud_pair_extended(cell.family, cell.n0, cell.n1, cloud_seed, d=cell.d)
    cloud0, cloud1 = cloud_pair

    # Dispatch
    try:
        if candidate == "PointMMD-Gaussian":
            result = point_mmd_gaussian(cloud0, cloud1, regime=REGIME_I, bandwidth=PRIMARY_POINT_BANDWIDTH, n_perm=n_permutations, seed=method_seed)
            variant = f"bw{PRIMARY_POINT_BANDWIDTH}"
        elif candidate == "PointMMD-Gaussian-median":
            result = point_mmd_gaussian(cloud0, cloud1, regime=REGIME_I, bandwidth=None, n_perm=n_permutations, seed=method_seed)
            variant = "median_heuristic"
        elif candidate.startswith("PointMMD-Gaussian-bw"):
            # e.g. PointMMD-Gaussian-bw0.15
            try:
                bw = float(candidate.split("bw")[1])
            except Exception:
                bw = PRIMARY_POINT_BANDWIDTH
            result = point_mmd_gaussian(cloud0, cloud1, regime=REGIME_I, bandwidth=bw, n_perm=n_permutations, seed=method_seed)
            variant = f"bw{bw}"
        elif candidate == "EnergyDistance":
            result = energy_distance_test(cloud0, cloud1, regime=REGIME_I, n_perm=n_permutations, seed=method_seed)
            variant = ""
        elif candidate == "FriedmanRafsky-MST":
            result = friedman_rafsky_mst(cloud0, cloud1, regime=REGIME_I, n_perm=n_permutations, seed=method_seed)
            variant = ""
        elif candidate.startswith("Schilling-kNN"):
            # parse k (candidate is e.g. Schilling-kNN-k1, contains two '-k' substrings)
            if "-k" in candidate:
                try:
                    k = int(candidate.rsplit("-k", 1)[1])
                except Exception:
                    k = 1
            else:
                k = 1
            result = schilling_knn(cloud0, cloud1, regime=REGIME_I, k=k, directed=True, n_perm=n_permutations, seed=method_seed)
            variant = f"k{k}"
        elif candidate == "Rosenbaum-CrossMatch":
            result = rosenbaum_crossmatch(cloud0, cloud1, regime=REGIME_I, n_perm=n_permutations, seed=method_seed)
            variant = ""
        elif candidate == "SlicedWasserstein":
            result = sliced_wasserstein_test(cloud0, cloud1, regime=REGIME_I, n_projections=100, projection_seed=0, n_perm=n_permutations, seed=method_seed)
            variant = "100proj"
        elif candidate == "ClassifierTwoSampleTest-logistic":
            result = classifier_two_sample_test(cloud0, cloud1, regime=REGIME_I, classifier="logistic", test_fraction=0.5, split_seed=method_seed, n_perm=n_permutations, seed=method_seed)
            variant = "logistic"
        elif candidate == "ClassifierTwoSampleTest-rf":
            result = classifier_two_sample_test(cloud0, cloud1, regime=REGIME_I, classifier="rf", test_fraction=0.5, split_seed=method_seed, n_perm=n_permutations, seed=method_seed)
            variant = "rf"
        elif candidate == "RawBlockMMD":
            result = raw_block_mmd(cloud0, cloud1, regime=REGIME_I, m=cell.m, partition_seed=partition_seed, n_perm=n_permutations, seed=method_seed)
            variant = f"m{cell.m}"
        elif candidate == "SC-B":
            result = sc_b_disjoint_mmd(cloud0, cloud1, regime=REGIME_I, m=cell.m, partition_seed=partition_seed, n_perm=n_permutations, seed=method_seed, filtration=FILTRATION, homology_dims=HOMOLOGY_DIMS, kernel_bandwidth=KERNEL_BANDWIDTH)
            variant = f"m{cell.m}"
        elif candidate == "SC-A":
            # SC-A uses pooled point permutation but expensive; apply projection cap like Phase 5
            x0p = _project_cloud(cloud0, MAX_SC_A_POINT_N, _seed(BENCHMARK_VERSION, "SC-A-project0", cell.cell_id, replication))
            x1p = _project_cloud(cloud1, MAX_SC_A_POINT_N, _seed(BENCHMARK_VERSION, "SC-A-project1", cell.cell_id, replication))
            result = sc_a_label_permutation(x0p, x1p, regime=REGIME_I, filtration=FILTRATION, homology_dims=HOMOLOGY_DIMS, kernel_bandwidth=KERNEL_BANDWIDTH, n_perm=n_permutations, seed=method_seed, cache_dir=cache_dir)
            # record projection in diagnostics
            result["diagnostics"]["projection_used"] = len(x0p) < len(cloud0) or len(x1p) < len(cloud1)
            variant = ""
        elif candidate.startswith("HybridBlockMMD"):
            # format HybridBlockMMD-a0.50
            try:
                alpha = float(candidate.split("a")[1])
            except Exception:
                alpha = 0.5
            result = hybrid_block_mmd(cloud0, cloud1, regime=REGIME_I, m=cell.m, alpha=alpha, partition_seed=partition_seed, n_perm=n_permutations, seed=method_seed, filtration=FILTRATION, homology_dims=HOMOLOGY_DIMS, barcode_kernel_bandwidth=KERNEL_BANDWIDTH)
            variant = f"a{alpha:.2f}"
        elif candidate == "SC-A-Block":
            result = sc_a_blockwise_label_permutation(cloud0, cloud1, regime=REGIME_I, m=cell.m, partition_seed=partition_seed, n_perm=n_permutations, seed=method_seed)
            variant = f"m{cell.m}"
        else:
            raise ValueError(f"unknown candidate {candidate!r}")
        return result, variant
    except Exception as exc:
        # bubble up for unified_record to catch as failure/refusal
        raise exc

def _run_one(args) -> list[dict]:
    cell, replication, candidates, n_permutations, cache_dir = args
    rows = []
    cloud_seed = _cloud_seed(cell, replication)
    partition_seed = _partition_seed(cell, replication)
    cloud_pair = make_cloud_pair_extended(cell.family, cell.n0, cell.n1, cloud_seed, d=cell.d)
    for candidate in candidates:
        method_seed = _method_seed(cell, replication, candidate)
        try:
            result, variant = _call_method(cell, replication, candidate, n_permutations=n_permutations, cache_dir=cache_dir, cloud_pair=cloud_pair)
            rec = _unified_record(cell=cell, replication=replication, candidate=candidate, method_variant=variant, result=result, cloud_seed=cloud_seed, partition_seed=partition_seed if candidate in ("RawBlockMMD","SC-B","HybridBlockMMD-a0.50","SC-A-Block") or candidate.startswith("Hybrid") else None, permutation_seed=method_seed, status="ok")
            # ensure rejected flag correct
            rec["rejected"] = bool(rec["pvalue"] <= ALPHA) if np.isfinite(rec["pvalue"]) else False
            rows.append(rec)
        except Exception as exc:
            # Refusal vs failed: check if expected refusal condition
            msg = f"{type(exc).__name__}: {exc}"
            # mark as refused if known condition phrases
            status = "failed"
            if any(k in str(exc).lower() for k in ["exceeds", "no barcode block", "m must be", "k must be", "cloud must contain", "pooled n", "refused", "k="]):
                status = "refused"
            # create dummy result for schema
            rec = _unified_record(cell=cell, replication=replication, candidate=candidate, method_variant="", result={"statistic": np.nan, "pvalue": np.nan, "n_permutations": -1, "exact_enumeration": False, "permutation_group": "", "diagnostics": {}, "kernel": "", "n0": cell.n0, "n1": cell.n1}, cloud_seed=cloud_seed, partition_seed=partition_seed if candidate in ("RawBlockMMD","SC-B","HybridBlockMMD-a0.50","SC-A-Block") or candidate.startswith("Hybrid") else None, permutation_seed=method_seed, status=status, failure_reason=msg)
            rows.append(rec)
    return rows

def run_replicates(
    *,
    families: Sequence[str] = CORE_FAMILIES,
    n_grid: Sequence[int] = (250,),
    m_values: Sequence[int] = (PRIMARY_M,),
    d_values: Sequence[int] = (2,),
    replications: int = PILOT_REPLICATIONS,
    n_permutations: int = PILOT_PERMUTATIONS,
    workers: int = 1,
    candidates: Sequence[str] = PRIMARY_CANDIDATES,
    cache_dir: Optional[str] = None,
    include_robustness: bool = False,
    include_dependence: bool = False,
) -> pd.DataFrame:
    if replications < 1 or n_permutations < 1:
        raise ValueError("replications and n_permutations must be positive")
    if workers < 1 or workers > MAX_WORKERS:
        raise ValueError(f"workers must be in [1,{MAX_WORKERS}]")
    cells = make_cells(families=families, n_grid=n_grid, m_values=m_values, d_values=d_values, include_robustness=include_robustness, include_dependence=include_dependence)
    cand_tuple = tuple(candidates)
    args = [(cell, rep, cand_tuple, int(n_permutations), cache_dir) for cell in cells for rep in range(int(replications))]
    if workers == 1:
        nested = [_run_one(a) for a in args]
    else:
        with ProcessPoolExecutor(max_workers=min(int(workers), MAX_WORKERS)) as pool:
            nested = list(pool.map(_run_one, args))
    return pd.DataFrame([r for rows in nested for r in rows])

def summarize(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    groups = ["method","family","family_role","n0","n1","m","d","alpha","kernel_or_distance","target_null","sampling_unit","validity_regime"]
    # ensure required columns exist
    for col in groups:
        if col not in frame.columns:
            frame[col] = np.nan
    rows = []
    for keys, group in frame.groupby(groups, dropna=False, sort=True):
        ok = group[group["status"]=="ok"]
        total = int(len(ok))
        fails = int(len(group)-total)
        refuses = int((group["status"]=="refused").sum())
        rejects = int(ok["rejected"].sum()) if total else 0
        rate = rejects/total if total else np.nan
        lo, hi = _mc_interval(rejects, total) if total else (np.nan, np.nan)
        values = dict(zip(groups, keys))
        values.update({
            "design_hash": DESIGN_HASH,
            "benchmark_version": BENCHMARK_VERSION,
            "replications": total,
            "failed_replications": fails - refuses,
            "refused_replications": refuses,
            "rejections": rejects,
            "rejection_rate": rate,
            "mc_low": lo,
            "mc_high": hi,
            "in_size_band": bool(SIZE_BAND[0] <= rate <= SIZE_BAND[1]) if total else False,
            "mean_runtime_seconds": float(ok["runtime_seconds"].mean()) if total and "runtime_seconds" in ok else np.nan,
            "mean_peak_rss_bytes": float(ok["peak_rss_bytes"].mean()) if total else np.nan,
            "mean_K0": float(ok["K0"].mean()) if total else np.nan,
            "mean_K1": float(ok["K1"].mean()) if total else np.nan,
            "effective_total_mean": float(ok["effective_sample_size_total"].mean()) if total else np.nan,
        })
        rows.append(values)
    return pd.DataFrame(rows)

def _comparison_table(summary: pd.DataFrame) -> pd.DataFrame:
    if summary.empty:
        return pd.DataFrame()
    primary = summary[
        (summary["m"].isna() | summary["m"].eq(PRIMARY_M) | summary["m"].eq(float(PRIMARY_M)))
        & (summary["n0"] == PRIMARY_N)
        & (summary["n1"] == PRIMARY_N)
        & (summary["d"] == PRIMARY_D)
    ]
    # Keep both point and block methods at primary size
    # Select headline candidates
    keep = [c for c in ALL_CANDIDATES if c in summary["method"].unique()]
    primary = primary[primary["method"].isin(keep)]
    rows = []
    for method in sorted(primary["method"].unique()):
        sub = primary[primary["method"]==method]
        # pick target from first non-null
        try:
            target = sub["target_null"].dropna().iloc[0] if len(sub["target_null"].dropna()) else ""
        except Exception:
            target = ""
        try:
            unit = sub["sampling_unit"].dropna().iloc[0] if len(sub["sampling_unit"].dropna()) else ""
        except Exception:
            unit = ""
        def rate(fam):
            v = sub[sub["family"]==fam]["rejection_rate"]
            return float(v.iloc[0]) if len(v) else np.nan
        def mc(fam):
            v = sub[sub["family"]==fam]
            if len(v):
                return float(v.iloc[0]["mc_low"]), float(v.iloc[0]["mc_high"])
            return np.nan, np.nan
        iid_lo, iid_hi = mc("iid_null")
        rows.append({
            "method": method,
            "n0": PRIMARY_N,
            "n1": PRIMARY_N,
            "d": PRIMARY_D,
            "target_null": target,
            "sampling_unit": unit,
            "m": PRIMARY_M,
            "null_rejection_rate": rate("iid_null"),
            "null_mc_low": iid_lo,
            "null_mc_high": iid_hi,
            "density_power": rate("same_support_density"),
            "topology_power": rate("topology_alt"),
            "translated_pointlaw_power_or_barcode_null": rate("weak_barcode_null"),
            "four_atom_power": rate("same_square_four_atom_density"),
            "mean_runtime": float(sub["mean_runtime_seconds"].mean()) if len(sub) else np.nan,
            "mean_peak_rss": float(sub["mean_peak_rss_bytes"].mean()) if len(sub) else np.nan,
        })
    return pd.DataFrame(rows)

def _plot(summary: pd.DataFrame, output: str) -> None:
    mpl_config = os.path.join("/tmp", "tda2s_phase5ab_pointlaw_mplconfig")
    os.makedirs(mpl_config, exist_ok=True)
    os.environ.setdefault("MPLCONFIGDIR", mpl_config)
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    # The headline figure is explicitly the primary n=250, d=2 comparison.
    # The summary parquet retains all n and d cells separately.
    summary = summary[
        (summary["n0"] == PRIMARY_N)
        & (summary["n1"] == PRIMARY_N)
        & (summary["d"] == PRIMARY_D)
        & (summary["m"].isna() | summary["m"].eq(PRIMARY_M) | summary["m"].eq(float(PRIMARY_M)))
    ].copy()
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    # Panel 1: calibration (iid null + weak barcode null)
    ax = axes[0,0]
    calibr = summary[summary["family"].isin(["iid_null","weak_barcode_null"])]
    methods = sorted(summary["method"].unique())
    colors = {"PointMMD-Gaussian":"#4477AA","EnergyDistance":"#228833","FriedmanRafsky-MST":"#CC6677","Schilling-kNN-k1":"#AA3377","RawBlockMMD":"#66CCEE","SC-B":"#228833","SC-A":"#4477AA","Rosenbaum-CrossMatch":"#332288","SlicedWasserstein":"#888888","ClassifierTwoSampleTest-logistic":"#CC3311","ClassifierTwoSampleTest-rf":"#EE7733"}
    for method in methods:
        sub = calibr[calibr["method"]==method].sort_values("family")
        if sub.empty:
            continue
        x = np.arange(len(sub))
        ax.errorbar(x, sub["rejection_rate"], yerr=[sub["rejection_rate"]-sub["mc_low"], sub["mc_high"]-sub["rejection_rate"]], marker="o", capsize=3, label=method, color=colors.get(method, None), linewidth=1)
    ax.axhspan(*SIZE_BAND, color="#66AA55", alpha=0.12)
    ax.axhline(ALPHA, color="black", linestyle="--", linewidth=0.8)
    ax.set_xticks([0,1], ["iid null","weak barcode"])
    ax.set_ylim(0,1)
    ax.set_ylabel("rejection rate")
    ax.set_title("Target separation and calibration (m=25, d=2)")
    ax.legend(fontsize=6, ncol=2)
    ax.grid(alpha=0.2)

    # Panel 2: power density vs topology by method
    ax = axes[0,1]
    power = summary[summary["family"].isin(["same_support_density","topology_alt"])]
    # pivot
    for method in methods:
        sub = power[power["method"]==method].sort_values("family")
        if sub.empty:
            continue
        if len(sub) == 2:
            ax.plot([0,1], sub["rejection_rate"].values, marker="o", label=method)
    ax.set_xticks([0,1], ["density","topology"])
    ax.set_ylim(0,1)
    ax.set_ylabel("rejection rate")
    ax.set_title("Density vs topology power")
    ax.legend(fontsize=6, ncol=2)
    ax.grid(alpha=0.2)

    # Panel 3: runtime vs peak RSS
    ax = axes[1,0]
    for method in methods:
        sub = summary[summary["method"]==method]
        if sub.empty:
            continue
        ax.scatter(sub["mean_runtime_seconds"], sub["mean_peak_rss_bytes"]/1e6, label=method)
    ax.set_xlabel("mean runtime (s)")
    ax.set_ylabel("mean peak RSS (MB)")
    ax.set_title("Computation")
    ax.legend(fontsize=6)
    ax.grid(alpha=0.2)

    # Panel 4: effective sample size diagnostic (K0+K1 vs n)
    ax = axes[1,1]
    valid = []
    labels = []
    vals = []
    for method in methods:
        sub = summary[summary["method"]==method]
        if sub.empty:
            continue
        eff = sub["effective_total_mean"].mean()
        if np.isfinite(eff):
            labels.append(method)
            vals.append(eff)
    if vals:
        ax.bar(np.arange(len(vals)), vals, tick_label=labels)
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=6)
    ax.set_ylabel("effective sample size total")
    ax.set_title("Effective sample size (point vs block)")
    ax.grid(alpha=0.2)
    fig.tight_layout()
    os.makedirs(os.path.dirname(os.path.abspath(output)), exist_ok=True)
    fig.savefig(output, dpi=180, bbox_inches="tight")
    plt.close(fig)

def _write_report(summary: pd.DataFrame, comparison: pd.DataFrame, output: str, *, replications: int, n_permutations: int) -> None:
    lines = [
        "# Phase 5AB point-law benchmark report",
        "",
        f"Benchmark version `{BENCHMARK_VERSION}`, design hash `{DESIGN_HASH}`. Replications per cell `{replications}`, permutations `{n_permutations}`.",
        "",
        "## Design audit",
        "",
        "Frozen protocol per `docs/phase5ab_point_law_registry.md`. Primary `m=25` locked; point-level baselines use pooled kernels/distances/graphs cached outside permutation loop; block methods use frozen disjoint partition with `K_a=floor(n_a/m)` and remainders discarded.",
        "",
        "## Method registry (headline comparison distinguishes target)",
        "",
        "| method | target_null | sampling_unit | null_rejection | density | topology | translated | 4-atom | runtime | peak_RSS |",
        "|---|---|---|---|---:|---:|---:|---:|---:|---:|",
    ]
    for _, row in comparison.iterrows():
        lines.append(f"| {row['method']} | {row['target_null']} | {row['sampling_unit']} | {row['null_rejection_rate']:.3f} | {row['density_power']:.3f} | {row['topology_power']:.3f} | {row['translated_pointlaw_power_or_barcode_null']:.3f} | {row['four_atom_power']:.3f} | {row['mean_runtime']:.3f} | {row['mean_peak_rss']:.0f} |")
    lines.extend([
        "",
        "## Gates (B1-B8)",
        "",
        "B1 target/literature validity: passes by construction for registered methods. SC-B remains under `H0,25^bar`; RawBlockMMD and required raw baselines under `H0^law` when assumptions hold.",
        "B2 iid size: primary H0^law methods should lie in [0.03,0.08] at 500 replications (with Clopper-Pearson intervals). Cells with K_a<5 are diagnostic only.",
        "B3 point-law power: RawBlockMMD gate passes only if competitive with strongest raw baseline on density/location/sparse/dense/rare alternatives (paired MC interval).",
        "B4 topology retention: useful power on filled-disk vs noisy-circle; hybrid predeclared alpha only.",
        "B5 barcode separation: translated cell is `translated_pointlaw_power` for RawBlockMMD vs `translated_barcode_null_rejection` for SC-B.",
        "B6 small-sample honesty: m=25 becomes unusable where K_a too small; report p-value grid coarseness.",
        "B7 robustness: overlap/dependence marked unsupported.",
        "B8 computation: runtime, peak RSS, failure rate, coverage reported; methods exceeding Colab budget flagged as computationally limited.",
        "",
        "## Effective sample size note",
        "",
        "Point-level methods use all points (`n0+n1` observations). Block methods use `K0+K1` blocks; the `m=1` RawBlock sensitivity is the bridge. Report contains both all-point and effective-sample-size comparisons.",
        "",
        "All headline numbers are regenerable from aggregated parquet without rerunning methods.",
        "",
    ])
    four = summary[summary["family"]=="same_square_four_atom_density"]
    if not four.empty:
        lines.extend(["## Four-atom diagnostic", "", "| method | rejection | 95% MC |", "|---|---:|---:|"])
        for _, r in four.iterrows():
            lines.append(f"| {r['method']} | {r['rejection_rate']:.3f} | [{r['mc_low']:.3f},{r['mc_high']:.3f}] |")
        lines.append("")
    os.makedirs(os.path.dirname(os.path.abspath(output)), exist_ok=True)
    with open(output, "w", encoding="utf-8") as h:
        h.write("\n".join(lines))

# ---------------------------------------------------------------------------
# Shard helpers (Colab-friendly)

def run_shard(
    cell: Cell,
    *,
    rep_start: int,
    replications: int,
    n_permutations: int,
    candidates: Sequence[str],
    workers: int = 1,
    cache_dir: Optional[str] = None,
    output: Optional[str] = None,
) -> str:
    if rep_start < 0 or replications < 1:
        raise ValueError("rep_start must be >=0 and replications positive")
    args = [(cell, rep, tuple(candidates), int(n_permutations), cache_dir) for rep in range(int(rep_start), int(rep_start)+replications)]
    if workers == 1:
        nested = [_run_one(a) for a in args]
    else:
        with ProcessPoolExecutor(max_workers=min(workers, MAX_WORKERS)) as pool:
            nested = list(pool.map(_run_one, args))
    rows = [r for part in nested for r in part]
    frame = pd.DataFrame(rows)
    if output is None:
        output = os.path.join(SHARD_DIR, f"phase5ab_pointlaw_{cell.cell_id}_rep{rep_start}_{rep_start+replications-1}.parquet")
    os.makedirs(os.path.dirname(os.path.abspath(output)), exist_ok=True)
    frame.to_parquet(output, index=False)
    print(json.dumps({"cell": cell.cell_id, "replications": replications, "rows": len(rows), "workers": workers, "output": output}, indent=2))
    return output

def run_pilot(
    *,
    replications: int = PILOT_REPLICATIONS,
    n_permutations: int = PILOT_PERMUTATIONS,
    workers: int = 1,
    candidates: Sequence[str] = PRIMARY_CANDIDATES,
    families: Sequence[str] = CORE_FAMILIES,
    n_grid: Sequence[int] = (250,),
    output: Optional[str] = None,
) -> str:
    frame = run_replicates(families=families, n_grid=n_grid, m_values=(PRIMARY_M,), d_values=(2,), replications=replications, n_permutations=n_permutations, workers=workers, candidates=candidates)
    summary = summarize(frame)
    comparison = _comparison_table(summary)
    if output is None:
        output = os.path.join(RESULTS_DIR, "phase5ab_pointlaw_pilot.parquet")
    os.makedirs(os.path.dirname(os.path.abspath(output)), exist_ok=True)
    frame.to_parquet(output, index=False)
    # also write pilot summary/comparison for inspection
    summary.to_parquet(output.replace(".parquet","_summary.parquet"), index=False)
    comparison.to_parquet(output.replace(".parquet","_comparison.parquet"), index=False)
    print(json.dumps({"pilot": True, "replications": replications, "families": list(families), "rows": len(frame), "output": output}, indent=2))
    return output

def run_fleet(
    *,
    families: Sequence[str] = CORE_FAMILIES,
    replications: int = GATE_REPLICATIONS,
    n_permutations: int = GATE_PERMUTATIONS,
    workers: int = 1,
    candidates: Sequence[str] = PRIMARY_CANDIDATES,
    cache_dir: Optional[str] = None,
) -> list[str]:
    cells = make_cells(families=families, n_grid=N_GRID, m_values=(PRIMARY_M,), d_values=(2,))
    # sequential cell scheduling to avoid memory blow-up
    paths = []
    for cell in cells:
        out = os.path.join(SHARD_DIR, f"phase5ab_pointlaw_{cell.cell_id}_rep0_{replications-1}.parquet")
        paths.append(run_shard(cell, rep_start=0, replications=replications, n_permutations=n_permutations, candidates=candidates, workers=workers, cache_dir=cache_dir, output=out))
    return paths

def aggregate(input_dir: str = SHARD_DIR, output_prefix: str = "phase5ab_pointlaw") -> dict:
    paths = sorted(glob.glob(os.path.join(input_dir, "phase5ab_pointlaw*.parquet")))
    if not paths:
        raise FileNotFoundError(f"no shards in {input_dir}")
    frames = [pd.read_parquet(p) for p in paths]
    frame = pd.concat(frames, ignore_index=True)
    key = ["design_hash","cell_id","method","replication"]
    dup = frame.duplicated(key, keep=False)
    if dup.any():
        dups = frame.loc[dup, key].drop_duplicates().to_dict("records")
        raise ValueError(f"duplicate keys: {dups[:3]}")
    if set(frame["design_hash"].dropna()) != {DESIGN_HASH}:
        raise ValueError("shards have conflicting design hash")
    summary = summarize(frame)
    comparison = _comparison_table(summary)
    # write
    os.makedirs(os.path.dirname(os.path.abspath(FINAL_REPLICATIONS)), exist_ok=True)
    frame.to_parquet(FINAL_REPLICATIONS, index=False)
    summary.to_parquet(FINAL_SUMMARY, index=False)
    comparison.to_parquet(FINAL_COMPARISON, index=False)
    _plot(summary, FINAL_FIGURE)
    _write_report(summary, comparison, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "docs", "phase5ab_pointlaw_report.md"), replications=int(frame["replication"].nunique()), n_permutations=int(frame["n_permutations"].iloc[0]) if len(frame) else GATE_PERMUTATIONS)
    manifest = {
        "benchmark_version": BENCHMARK_VERSION,
        "design_hash": DESIGN_HASH,
        "design_record": design_record(),
        "input_dir": input_dir,
        "shards": paths,
        "n_shards": len(paths),
        "replication_rows": int(len(frame)),
        "cell_replications": int(frame[["cell_id", "replication"]].drop_duplicates().shape[0]),
        "summary_rows": int(len(summary)),
        "comparison_rows": int(len(comparison)),
        "outputs": {"replications": FINAL_REPLICATIONS, "summary": FINAL_SUMMARY, "comparison": FINAL_COMPARISON, "figure": FINAL_FIGURE},
    }
    with open(FINAL_MANIFEST, "w", encoding="utf-8") as h:
        json.dump(manifest, h, indent=2, sort_keys=True)
    print(json.dumps(manifest, indent=2))
    return manifest


def profile_representative_cells(
    *,
    replications: int = 3,
    n_permutations: int = REPRO_PERMUTATIONS,
    candidates: Sequence[str] = ALL_CANDIDATES,
    output: Optional[str] = None,
    cache_dir: Optional[str] = None,
) -> dict:
    """Measure method cost on the five predeclared fleet-profile cells.

    Heavy methods are timed for one replication per cell because matching and
    SC-A can dominate a local profile.  Cheap methods use ``replications``;
    the resulting per-call median is used for fleet estimates.  The output is
    deliberately a machine-readable manifest consumed by the Colab generator.
    """
    if replications < 1 or n_permutations < 1:
        raise ValueError("replications and n_permutations must be positive")
    if output is None:
        output = os.path.join(RESULTS_DIR, "phase5ab_pointlaw_profile.json")
    observations = []
    for cell in PROFILE_CELLS:
        for candidate in tuple(candidates):
            n_reps = 1 if candidate in PROFILE_HEAVY_CANDIDATES else int(replications)
            elapsed = []
            method_runtime = []
            statuses = []
            failure_reasons = []
            rss = []
            for replication in range(n_reps):
                started = time.perf_counter()
                rows = _run_one((cell, replication, (candidate,), int(n_permutations), cache_dir))
                elapsed.append(time.perf_counter() - started)
                row = rows[0]
                statuses.append(str(row["status"]))
                if row.get("failure_reason"):
                    failure_reasons.append(str(row["failure_reason"]))
                if np.isfinite(row.get("runtime_seconds", np.nan)):
                    method_runtime.append(float(row["runtime_seconds"]))
                if int(row.get("peak_rss_bytes", -1)) >= 0:
                    rss.append(int(row["peak_rss_bytes"]))
            ok_times = method_runtime or elapsed
            per_call = float(np.median(ok_times))
            observation = {
                "cell_id": cell.cell_id,
                "family": cell.family,
                "n0": cell.n0,
                "n1": cell.n1,
                "d": cell.d,
                "m": cell.m,
                "method": candidate,
                "profile_replications": n_reps,
                "n_permutations": int(n_permutations),
                "per_call_seconds": per_call,
                "mean_call_seconds": float(np.mean(ok_times)),
                "median_call_seconds": per_call,
                "min_call_seconds": float(np.min(ok_times)),
                "max_call_seconds": float(np.max(ok_times)),
                "mean_method_runtime_seconds": float(np.mean(method_runtime)) if method_runtime else None,
                "mean_peak_rss_bytes": float(np.mean(rss)) if rss else None,
                "statuses": {status: statuses.count(status) for status in sorted(set(statuses))},
                "failure_reasons": sorted(set(failure_reasons)),
                "predicted_500_replications_minutes": per_call * GATE_REPLICATIONS / 60.0,
            }
            observations.append(observation)

    manifest = {
        "benchmark_version": BENCHMARK_VERSION,
        "design_hash": DESIGN_HASH,
        "profile_version": 1,
        "profile_cells": [cell.cell_id for cell in PROFILE_CELLS],
        "cheap_profile_replications": int(replications),
        "heavy_profile_replications": 1,
        "n_permutations": int(n_permutations),
        "observations": observations,
    }
    os.makedirs(os.path.dirname(os.path.abspath(output)), exist_ok=True)
    with open(output, "w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2, sort_keys=True, allow_nan=True)
    print(json.dumps({"profile": output, "observations": len(observations)}, indent=2))
    return manifest

# ---------------------------------------------------------------------------
# CLI

def main() -> None:
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--mode", choices=["pilot","shard","fleet","aggregate","profile"], default="pilot")
    p.add_argument("--replications", type=int, default=None)
    p.add_argument("--permutations", type=int, default=None)
    p.add_argument("--workers", type=int, default=1)
    p.add_argument("--families", default=None, help="comma-separated families")
    p.add_argument("--candidates", default=None, help="comma-separated candidates")
    p.add_argument("--n-grid", default=None, help="comma-separated n values")
    p.add_argument("--m-grid", default=None)
    p.add_argument("--d-grid", default=None)
    p.add_argument("--cell", default=None, help="cell_id for shard mode")
    p.add_argument("--rep-start", type=int, default=0)
    p.add_argument("--input-dir", default=SHARD_DIR)
    p.add_argument("--output", default=None)
    p.add_argument("--cache-dir", default=None)
    args = p.parse_args()

    # defaults per mode
    if args.mode == "pilot":
        reps = args.replications or PILOT_REPLICATIONS
        perms = args.permutations or PILOT_PERMUTATIONS
        fams = tuple(v.strip() for v in args.families.split(",")) if args.families else CORE_FAMILIES
        cands = tuple(v.strip() for v in args.candidates.split(",")) if args.candidates else PRIMARY_CANDIDATES
        n_grid = tuple(int(v) for v in args.n_grid.split(",")) if args.n_grid else (250,)
        frame = run_replicates(families=fams, n_grid=n_grid, m_values=(PRIMARY_M,), d_values=(2,), replications=reps, n_permutations=perms, workers=args.workers, candidates=cands, cache_dir=args.cache_dir)
        # write pilot
        out = args.output or os.path.join(RESULTS_DIR, "phase5ab_pointlaw_pilot.parquet")
        os.makedirs(os.path.dirname(os.path.abspath(out)), exist_ok=True)
        frame.to_parquet(out, index=False)
        summary = summarize(frame)
        comparison = _comparison_table(summary)
        summary.to_parquet(out.replace(".parquet","_summary.parquet"), index=False)
        comparison.to_parquet(out.replace(".parquet","_comparison.parquet"), index=False)
        _plot(summary, out.replace(".parquet",".png"))
        _write_report(summary, comparison, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "docs", "phase5ab_pointlaw_pilot_report.md"), replications=reps, n_permutations=perms)
        print(json.dumps({"mode":"pilot","replications":reps,"permutations":perms,"rows":len(frame),"output":out}, indent=2))
    elif args.mode == "shard":
        if not args.cell:
            raise SystemExit("--cell required for shard mode")
        # parse cell_id: format family_n{n0}_n1{n1}_m{m}_d{d}
        # reuse make_cells to find matching cell
        all_cells = make_cells(families=list(FAMILY_ROLE_EXT.keys()), n_grid=N_GRID + (10,15,20,25,30,50,75,100,125,150), m_values=M_GRID, d_values=D_GRID, include_robustness=True, include_dependence=True)
        # also handle four_atom family not in FAMILY_ROLE_EXT? already added
        cell = next((c for c in all_cells if c.cell_id == args.cell), None)
        if cell is None:
            # try to parse manually
            try:
                parts = args.cell.split("_")
                # naive parse
                family = parts[0]
                # find n0,n1,m,d
                n0 = int([x for x in parts if x.startswith("n")][0][1:])
                n1 = int([x for x in parts if x.startswith("n1")][0][2:])
                m = int([x for x in parts if x.startswith("m")][0][1:])
                d = int([x for x in parts if x.startswith("d")][0][1:])
                cell = Cell(family=family, n0=n0, n1=n1, m=m, d=d, role=FAMILY_ROLE_EXT.get(family,"unknown"), description=FAMILY_DESCRIPTION_EXT.get(family, family))
            except Exception as exc:
                raise SystemExit(f"unknown cell {args.cell!r}: {exc}")
        reps = args.replications or DEFAULT_SHARD_REPLICATIONS
        perms = args.permutations or GATE_PERMUTATIONS
        cands = tuple(v.strip() for v in args.candidates.split(",")) if args.candidates else PRIMARY_CANDIDATES
        run_shard(cell, rep_start=args.rep_start, replications=reps, n_permutations=perms, candidates=cands, workers=args.workers, cache_dir=args.cache_dir, output=args.output)
    elif args.mode == "fleet":
        reps = args.replications or GATE_REPLICATIONS
        perms = args.permutations or GATE_PERMUTATIONS
        fams = tuple(v.strip() for v in args.families.split(",")) if args.families else CORE_FAMILIES
        cands = tuple(v.strip() for v in args.candidates.split(",")) if args.candidates else PRIMARY_CANDIDATES
        run_fleet(families=fams, replications=reps, n_permutations=perms, workers=args.workers, candidates=cands, cache_dir=args.cache_dir)
    elif args.mode == "aggregate":
        aggregate(input_dir=args.input_dir)
    elif args.mode == "profile":
        reps = args.replications or 5
        perms = args.permutations or REPRO_PERMUTATIONS
        cands = tuple(v.strip() for v in args.candidates.split(",")) if args.candidates else ALL_CANDIDATES
        profile_representative_cells(replications=reps, n_permutations=perms, candidates=cands, output=args.output, cache_dir=args.cache_dir)
    else:
        raise SystemExit(f"unknown mode {args.mode}")

if __name__ == "__main__":
    main()


In [ ]:
import json
import os
import time
from pathlib import Path

import pandas as pd

from experiments.phase5ab_pointlaw_tournament import (
    DESIGN_HASH, FAMILY_DESCRIPTION_EXT, FAMILY_ROLE_EXT, GATE_PERMUTATIONS, Cell, run_shard,
)

EXPECTED_DESIGN_HASH = 'df1e561dc12a58e4'
SOURCE_HASH = '9787ce40f60911b7d455e7fccaac3398612518f315fd7cf57f0deb9330f9be47'
TASKS = [{'task_id': 'iid_null_n1000_n11000_m25_d2_rep150_174', 'cell_id': 'iid_null_n1000_n11000_m25_d2', 'family': 'iid_null', 'n0': 1000, 'n1': 1000, 'm': 25, 'd': 2, 'rep_start': 150, 'replications': 25, 'candidates': ['PointMMD-Gaussian', 'EnergyDistance', 'FriedmanRafsky-MST', 'Schilling-kNN-k1', 'RawBlockMMD', 'SC-B', 'SC-A', 'SlicedWasserstein', 'ClassifierTwoSampleTest-logistic', 'ClassifierTwoSampleTest-rf', 'Schilling-kNN-k5', 'Schilling-kNN-k10', 'PointMMD-Gaussian-median'], 'predicted_seconds': 3412.5578012073674}, {'task_id': 'same_square_four_atom_density_n1000_n11000_m25_d2_rep150_174', 'cell_id': 'same_square_four_atom_density_n1000_n11000_m25_d2', 'family': 'same_square_four_atom_density', 'n0': 1000, 'n1': 1000, 'm': 25, 'd': 2, 'rep_start': 150, 'replications': 25, 'candidates': ['PointMMD-Gaussian', 'EnergyDistance', 'FriedmanRafsky-MST', 'Schilling-kNN-k1', 'RawBlockMMD', 'SC-B', 'SC-A', 'SlicedWasserstein', 'ClassifierTwoSampleTest-logistic', 'ClassifierTwoSampleTest-rf', 'Schilling-kNN-k5', 'Schilling-kNN-k10', 'PointMMD-Gaussian-median'], 'predicted_seconds': 3412.5578012073674}, {'task_id': 'same_square_four_atom_density_n250_n1250_m25_d2_rep150_174', 'cell_id': 'same_square_four_atom_density_n250_n1250_m25_d2', 'family': 'same_square_four_atom_density', 'n0': 250, 'n1': 250, 'm': 25, 'd': 2, 'rep_start': 150, 'replications': 25, 'candidates': ['PointMMD-Gaussian', 'EnergyDistance', 'FriedmanRafsky-MST', 'Schilling-kNN-k1', 'RawBlockMMD', 'SC-B', 'SC-A', 'Rosenbaum-CrossMatch', 'SlicedWasserstein', 'ClassifierTwoSampleTest-logistic', 'ClassifierTwoSampleTest-rf', 'Schilling-kNN-k5', 'Schilling-kNN-k10', 'PointMMD-Gaussian-median'], 'predicted_seconds': 3320.018130190802}, {'task_id': 'same_support_density_n250_n1250_m25_d2_rep150_174', 'cell_id': 'same_support_density_n250_n1250_m25_d2', 'family': 'same_support_density', 'n0': 250, 'n1': 250, 'm': 25, 'd': 2, 'rep_start': 150, 'replications': 25, 'candidates': ['PointMMD-Gaussian', 'EnergyDistance', 'FriedmanRafsky-MST', 'Schilling-kNN-k1', 'RawBlockMMD', 'SC-B', 'SC-A', 'Rosenbaum-CrossMatch', 'SlicedWasserstein', 'ClassifierTwoSampleTest-logistic', 'ClassifierTwoSampleTest-rf', 'Schilling-kNN-k5', 'Schilling-kNN-k10', 'PointMMD-Gaussian-median'], 'predicted_seconds': 2482.8869214611773}, {'task_id': 'same_support_density_n1000_n11000_m25_d2_rep150_174', 'cell_id': 'same_support_density_n1000_n11000_m25_d2', 'family': 'same_support_density', 'n0': 1000, 'n1': 1000, 'm': 25, 'd': 2, 'rep_start': 150, 'replications': 25, 'candidates': ['PointMMD-Gaussian', 'EnergyDistance', 'FriedmanRafsky-MST', 'Schilling-kNN-k1', 'RawBlockMMD', 'SC-B', 'SC-A', 'SlicedWasserstein', 'ClassifierTwoSampleTest-logistic', 'ClassifierTwoSampleTest-rf', 'Schilling-kNN-k5', 'Schilling-kNN-k10', 'PointMMD-Gaussian-median'], 'predicted_seconds': 2202.989337200779}, {'task_id': 'weak_barcode_null_n500_n1500_m25_d2_rep150_174', 'cell_id': 'weak_barcode_null_n500_n1500_m25_d2', 'family': 'weak_barcode_null', 'n0': 500, 'n1': 500, 'm': 25, 'd': 2, 'rep_start': 150, 'replications': 25, 'candidates': ['PointMMD-Gaussian', 'EnergyDistance', 'FriedmanRafsky-MST', 'Schilling-kNN-k1', 'RawBlockMMD', 'SC-B', 'SC-A', 'SlicedWasserstein', 'ClassifierTwoSampleTest-logistic', 'ClassifierTwoSampleTest-rf', 'Schilling-kNN-k5', 'Schilling-kNN-k10', 'PointMMD-Gaussian-median'], 'predicted_seconds': 1119.2601953555152}, {'task_id': 'same_support_density_n500_n1500_m25_d2_rep150_174', 'cell_id': 'same_support_density_n500_n1500_m25_d2', 'family': 'same_support_density', 'n0': 500, 'n1': 500, 'm': 25, 'd': 2, 'rep_start': 150, 'replications': 25, 'candidates': ['PointMMD-Gaussian', 'EnergyDistance', 'FriedmanRafsky-MST', 'Schilling-kNN-k1', 'RawBlockMMD', 'SC-B', 'SC-A', 'SlicedWasserstein', 'ClassifierTwoSampleTest-logistic', 'ClassifierTwoSampleTest-rf', 'Schilling-kNN-k5', 'Schilling-kNN-k10', 'PointMMD-Gaussian-median'], 'predicted_seconds': 848.4478347907906}, {'task_id': 'topology_alt_n500_n1500_m25_d2_rep150_174', 'cell_id': 'topology_alt_n500_n1500_m25_d2', 'family': 'topology_alt', 'n0': 500, 'n1': 500, 'm': 25, 'd': 2, 'rep_start': 150, 'replications': 25, 'candidates': ['PointMMD-Gaussian', 'EnergyDistance', 'FriedmanRafsky-MST', 'Schilling-kNN-k1', 'RawBlockMMD', 'SC-B', 'SC-A', 'SlicedWasserstein', 'ClassifierTwoSampleTest-logistic', 'ClassifierTwoSampleTest-rf', 'Schilling-kNN-k5', 'Schilling-kNN-k10', 'PointMMD-Gaussian-median'], 'predicted_seconds': 789.1782011416839}]
SHARD_OUT = os.path.join(REPO_DIR, "phase5ab_pointlaw_shard_{:02d}.parquet".format(SHARD_ID))
MANIFEST_OUT = os.path.join(REPO_DIR, "phase5ab_pointlaw_shard_{:02d}_manifest.json".format(SHARD_ID))
CACHE_DIR = os.path.join(REPO_DIR, "phase5ab_pointlaw_cache_{:02d}".format(SHARD_ID))
os.makedirs(CACHE_DIR, exist_ok=True)

if os.path.exists(SHARD_OUT):
    accumulated = pd.read_parquet(SHARD_OUT)
else:
    accumulated = pd.DataFrame()
completed = set()
if not accumulated.empty:
    completed = set(zip(accumulated["cell_id"], accumulated["method"], accumulated["replication"]))
completed_tasks = []
started = time.time()
for task in TASKS:
    expected_cell = task["cell_id"]
    expected_reps = range(task["rep_start"], task["rep_start"] + task["replications"])
    if all((expected_cell, method, rep) in completed for method in task["candidates"] for rep in expected_reps):
        completed_tasks.append(task["task_id"])
        continue
    parts = task["cell_id"].rsplit("_n", 1)
    family = parts[0]
    suffix = "_n" + parts[1]
    n0_text, suffix = suffix.split("_n1", 1)
    n1_text, suffix = suffix.split("_m", 1)
    m_text, d_text = suffix.split("_d", 1)
    family, n0_text = family.rsplit("_n", 1)
    cell = Cell(family, int(n0_text), int(n1_text), int(m_text), int(d_text), FAMILY_ROLE_EXT.get(family, "unknown"), FAMILY_DESCRIPTION_EXT.get(family, family))
    task_file = os.path.join(REPO_DIR, "phase5ab_task_{}_{}.parquet".format(SHARD_ID, task["task_id"]))
    run_shard(cell, rep_start=task["rep_start"], replications=task["replications"], n_permutations=GATE_PERMUTATIONS, candidates=task["candidates"], workers=N_WORKERS, cache_dir=CACHE_DIR, output=task_file)
    part = pd.read_parquet(task_file)
    accumulated = pd.concat([accumulated, part], ignore_index=True)
    accumulated = accumulated.drop_duplicates(["design_hash", "cell_id", "method", "replication"], keep="last")
    accumulated = accumulated.sort_values(["cell_id", "method", "replication"]).reset_index(drop=True)
    accumulated.to_parquet(SHARD_OUT, index=False)
    completed = set(zip(accumulated["cell_id"], accumulated["method"], accumulated["replication"]))
    completed_tasks.append(task["task_id"])
    with open(MANIFEST_OUT, "w", encoding="utf-8") as handle:
        json.dump({"benchmark_version": 'phase5ab-pointlaw-v1', "design_hash": DESIGN_HASH, "expected_design_hash": EXPECTED_DESIGN_HASH, "source_hash": SOURCE_HASH, "shard_id": SHARD_ID, "n_shards": N_SHARDS, "completed_tasks": completed_tasks, "elapsed_seconds": time.time() - started}, handle, indent=2, sort_keys=True)
    print("Checkpoint:", task["task_id"], "rows:", len(accumulated), "elapsed_min:", round((time.time() - started) / 60, 1))

assert DESIGN_HASH == EXPECTED_DESIGN_HASH
print("Shard complete:", SHARD_OUT, "rows:", len(accumulated), "elapsed_min:", round((time.time() - started) / 60, 1))

output_file = SHARD_OUT
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as exc:
    print("(Not on Colab / download skipped):", exc)
